# Day 20 — PyBaMM full-protocol segmented audit

## Purpose

This notebook starts after Day 19A, which closed the retrospective phase and geometry audit.

Day 19A established that the historical PyBaMM DC–AC implementation line from nb04–nb20 was discharge-first relative to the MJ1 experimental charge-first ARB intent. It also established the decomposition

$$
\Delta t_{\mathrm{model}}(Q)
=
\Delta t_{\mathrm{geom}}(Q)
+
\Delta t_{\mathrm{resid}}(Q)
$$

and showed that, in prescribed-current CC simulations, raw $\Delta t(Q)$ is largely explained by current geometry, while the non-geometric residual remains near zero or weak in clean lineages.

## Day 20 question

Day 20 does **not** repeat the phase audit.

The purpose is to examine whether full-protocol PyBaMM simulations contain additional contributions beyond the prescribed-current CC geometry term, especially from:

1. AC-on prescribed-current CC geometry  
2. Voltage-limit event timing  
3. AC-off / control-state transition  
4. CV feedback trajectory  
5. Any remaining non-geometric residual

## Scope

This notebook is simulation-focused.

Primary data source:

- PyBaMM-generated trajectories and audit outputs in this repository

Not primary data source:

- NGU201 MJ1 raw experimental CSV files

Experimental MJ1 data may be used later as an external reference, but this notebook first audits the PyBaMM full-protocol behavior.

## Methodological boundary

The following claims are **not** allowed without segmented evidence:

- “PyBaMM reproduces MJ1 non-geometric state-layer acceleration.”
- “DC–AC acceleration is absent in all full-protocol cases.”
- “Raw $\Delta t(Q)$ is a mechanism-level observable.”

The correct Day 20 objective is to separate current-geometry, voltage-boundary, control-transition, and CV-feedback contributions.

In [2]:
# Cell 1 — Day20 bootstrap: PyBaMM full-protocol segmented audit
#
# Purpose:
#   Start a new audit layer after Day19A closure.
#   Day20 does not modify Day19A outputs.
#
# Scope:
#   Simulation-focused PyBaMM full-protocol audit.
#   Do not load NGU201 / MJ1 raw experimental CSVs in this notebook stage.

from pathlib import Path
from datetime import datetime
import json
import re
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    """
    Walk upward until a directory containing both data/ and notebooks/ is found.
    Works whether the notebook is launched from repo root or notebooks/.
    """
    start = Path.cwd() if start is None else Path(start).resolve()

    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p

    raise FileNotFoundError(
        f"Could not find repo root from {start}. "
        "Expected a parent containing data/ and notebooks/."
    )

REPO = find_repo_root()
DATA = REPO / "data"
DOCS = REPO / "docs"
NB_DIR = REPO / "notebooks"

assert DATA.exists(), f"Missing data dir: {DATA}"
assert DOCS.exists(), f"Missing docs dir: {DOCS}"
assert NB_DIR.exists(), f"Missing notebooks dir: {NB_DIR}"

day19_required = [
    DATA / "day19A_step6_evidence_register.csv",
    DATA / "day19A_step6_final_verdict_summary.csv",
    DOCS / "day19A_retrospective_audit.md",
]

for p in day19_required:
    assert p.exists(), f"Missing Day19A closure artifact: {p}"

print("=" * 72)
print("Day20 — PyBaMM full-protocol segmented audit")
print("=" * 72)
print(f"CWD   = {Path.cwd().resolve()}")
print(f"REPO  = {REPO}")
print(f"DATA  = {DATA}")
print(f"DOCS  = {DOCS}")
print(f"NB_DIR= {NB_DIR}")
print(f"Start = {datetime.now().isoformat(timespec='seconds')}")
print()
print("Day19A closure artifacts verified.")
print("Scope: PyBaMM simulation outputs only. NGU201/MJ1 raw experimental CSVs are out of scope for this notebook stage.")

Day20 — PyBaMM full-protocol segmented audit
CWD   = /Users/louislu/pybamm-dcac-superimposed/notebooks
REPO  = /Users/louislu/pybamm-dcac-superimposed
DATA  = /Users/louislu/pybamm-dcac-superimposed/data
DOCS  = /Users/louislu/pybamm-dcac-superimposed/docs
NB_DIR= /Users/louislu/pybamm-dcac-superimposed/notebooks
Start = 2026-05-06T14:35:53

Day19A closure artifacts verified.
Scope: PyBaMM simulation outputs only. NGU201/MJ1 raw experimental CSVs are out of scope for this notebook stage.


In [3]:
# Cell 2 — Inventory PyBaMM full-protocol / voltage-boundary / CV-related outputs
#
# Purpose:
#   Locate existing PyBaMM simulation outputs relevant to Day20 full-protocol
#   segmented audit.
#
# This is NOT an NGU201 / experimental raw CSV search.
#
# Outputs:
#   data/day20_step1_pybamm_full_protocol_output_inventory.csv

import json
import numpy as np
import pandas as pd
from pathlib import Path

patterns = [
    "day18_step1*",
    "day18_step2*",
    "day18B*",
    "day16_step2*",
    "day16_step3a*",
    "*phase_audit*",
    "*anchor*",
    "*trajectory*",
    "*trajectories*",
    "*dt_Q*",
    "*dtQ*",
    "*Vmax*",
    "*Q_to_Vmax*",
    "*metadata*",
]

hits = {}
for pat in patterns:
    for p in DATA.glob(pat):
        if not p.is_file():
            continue
        if p.name.startswith("day19A_"):
            continue
        hits[p.resolve()] = p

rows = []

for p in sorted(hits.values(), key=lambda x: x.name):
    row = {
        "path": str(p.relative_to(REPO)),
        "name": p.name,
        "suffixes": "".join(p.suffixes),
        "size_kb": round(p.stat().st_size / 1024, 2),
        "mtime": pd.Timestamp.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        "read_ok": False,
        "n_rows": np.nan,
        "n_cols_or_keys": np.nan,
        "columns_or_keys": "",
        "role_guess": "",
        "has_time": False,
        "has_current": False,
        "has_voltage": False,
        "has_Q": False,
        "has_dt": False,
        "has_event": False,
        "has_cv_or_termination": False,
        "has_frequency": False,
        "error": "",
    }

    try:
        if p.suffix.lower() == ".csv" or p.name.endswith(".csv.gz"):
            df_head = pd.read_csv(p, compression="infer", nrows=5)
            cols = list(df_head.columns)
            lc_all = " ".join([p.name.lower()] + [c.lower() for c in cols])

            row["read_ok"] = True
            row["n_cols_or_keys"] = len(cols)
            row["columns_or_keys"] = json.dumps(cols, ensure_ascii=False)

            row["has_time"] = any(s in lc_all for s in ["time", "t_s", "t_dc", "t_dcac", "t_protocol", "t_end"])
            row["has_current"] = any(s in lc_all for s in ["current", "i_", "i_a", "i_py"])
            row["has_voltage"] = any(s in lc_all for s in ["voltage", "vmax", "v_max", "v_min", "v_init"])
            row["has_Q"] = any(s in lc_all for s in ["q_", "q_ah", "q_net", "q_to_vmax", "q_hi", "q_low"])
            row["has_dt"] = any(s in lc_all for s in ["dt", "delta_t", "dtq"])
            row["has_event"] = any(s in lc_all for s in ["event", "q_to_vmax", "t_to_vmax", "vmax"])
            row["has_cv_or_termination"] = any(s in lc_all for s in ["cv", "termination", "cutoff", "t_end", "term_reason"])
            row["has_frequency"] = any(s in lc_all for s in ["f_hz", "freq", "frequency", "t_anchor", "period"])

            if "trajector" in p.name.lower():
                row["role_guess"] = "trajectory_or_cache_csv"
            elif row["has_event"] and row["has_cv_or_termination"]:
                row["role_guess"] = "event_boundary_metadata"
            elif row["has_dt"]:
                row["role_guess"] = "dtQ_curve_or_summary"
            elif row["has_frequency"]:
                row["role_guess"] = "frequency_protocol_design"
            else:
                row["role_guess"] = "csv_auxiliary"

            # row count
            try:
                row["n_rows"] = sum(len(chunk) for chunk in pd.read_csv(p, compression="infer", chunksize=100000))
            except Exception:
                row["n_rows"] = np.nan

        elif p.suffix.lower() == ".npz":
            z = np.load(p, allow_pickle=True)
            keys = list(z.keys())
            lc_all = " ".join([p.name.lower()] + [k.lower() for k in keys])

            row["read_ok"] = True
            row["n_cols_or_keys"] = len(keys)
            row["columns_or_keys"] = json.dumps(keys, ensure_ascii=False)

            row["has_time"] = any(s in lc_all for s in ["time", "t", "t_s"])
            row["has_current"] = any(s in lc_all for s in ["current", "i_", "i_a", "i_py"])
            row["has_voltage"] = any(s in lc_all for s in ["voltage", "v", "v_terminal"])
            row["has_Q"] = any(s in lc_all for s in ["q", "q_net"])
            row["has_dt"] = any(s in lc_all for s in ["dt", "delta"])
            row["has_event"] = any(s in lc_all for s in ["event", "vmax", "q_to_vmax"])
            row["has_cv_or_termination"] = any(s in lc_all for s in ["cv", "termination", "cutoff", "term"])
            row["has_frequency"] = any(s in lc_all for s in ["f_hz", "freq", "period", "tau"])

            row["role_guess"] = "npz_trajectory_cache"

        elif p.suffix.lower() in [".json", ".txt", ".yaml", ".yml"]:
            txt = p.read_text(encoding="utf-8", errors="ignore")[:5000].lower()
            row["read_ok"] = True
            row["columns_or_keys"] = ""

            row["has_time"] = any(s in txt for s in ["time", "t_s", "t_end"])
            row["has_current"] = any(s in txt for s in ["current", "i_py", "i_a"])
            row["has_voltage"] = any(s in txt for s in ["voltage", "vmax", "v_max"])
            row["has_Q"] = any(s in txt for s in ["q_net", "q_to_vmax", "q_hi"])
            row["has_dt"] = any(s in txt for s in ["dt", "delta_t"])
            row["has_event"] = any(s in txt for s in ["event", "vmax", "q_to_vmax"])
            row["has_cv_or_termination"] = any(s in txt for s in ["cv", "termination", "cutoff"])
            row["has_frequency"] = any(s in txt for s in ["f_hz", "frequency", "period", "tau"])

            row["role_guess"] = "text_or_json_metadata"

    except Exception as e:
        row["error"] = str(e)[:500]

    rows.append(row)

inv = pd.DataFrame(rows)

out = DATA / "day20_step1_pybamm_full_protocol_output_inventory.csv"
inv.to_csv(out, index=False)

print(f"Wrote: {out}")
print(f"Candidate files found: {len(inv)}")

if len(inv):
    display_cols = [
        "path", "size_kb", "read_ok", "role_guess",
        "has_time", "has_current", "has_voltage", "has_Q",
        "has_dt", "has_event", "has_cv_or_termination",
        "has_frequency", "columns_or_keys", "error"
    ]
    display(inv[display_cols].sort_values(["role_guess", "path"]).head(120))

print("\nRole guess counts:")
if len(inv):
    print(inv["role_guess"].value_counts(dropna=False).to_string())

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step1_pybamm_full_protocol_output_inventory.csv
Candidate files found: 30


,path,size_kb,read_ok,role_guess,has_time,has_current,has_voltage,has_Q,has_dt,has_event,has_cv_or_termination,has_frequency,columns_or_keys,error
17,data/day18_step1c_first_passage_dV_audit_v3.csv,0.39,True,csv_auxiliary,False,True,True,True,False,False,False,False,"[""param_set"", ""Q_nom_Ah"", ""q_lo_Ah"", ""q_hi_Ah""...",
3,data/day16_step3a_dtQ_curves_long.csv.gz,129.47,True,dtQ_curve_or_summary,True,True,False,True,True,False,False,False,"[""param_set"", ""chem_tag"", ""condition"", ""pair_r...",
4,data/day16_step3a_dtQ_table.csv,13.40,True,dtQ_curve_or_summary,True,True,True,True,True,True,False,False,"[""param_set"", ""chem_tag"", ""condition"", ""pair_r...",
5,data/day18B_protocol_design_full.csv,20.65,True,dtQ_curve_or_summary,False,True,True,True,True,False,True,True,"[""param_set"", ""Q_nom_Ah"", ""tau95_eq_s"", ""V_max...",
6,data/day18B_protocol_design_smoke.csv,7.72,True,dtQ_curve_or_summary,False,True,True,True,True,False,True,True,"[""param_set"", ""Q_nom_Ah"", ""tau95_eq_s"", ""V_max...",
7,data/day18B_smoke_dtQ_resid_curves_long.csv.gz,40.77,True,dtQ_curve_or_summary,False,False,False,True,True,False,False,False,"[""pair_id"", ""param_set"", ""protocol_label"", ""DC...",
8,data/day18B_smoke_dtQ_resid_summary.csv,7.30,True,dtQ_curve_or_summary,False,True,True,True,True,True,False,False,"[""pair_id"", ""param_set"", ""protocol_label"", ""DC...",
20,data/day18_step2_dt_Q_curves_long_v3.csv.gz,27.42,True,dtQ_curve_or_summary,True,False,False,True,True,False,False,False,"[""param_set"", ""anchor_label"", ""Q_Ah"", ""Q_frac_...",
21,data/day18_step2_dt_Q_curves_long_v4_charge_fi...,16.06,True,dtQ_curve_or_summary,True,False,False,True,True,False,False,False,"[""param_set"", ""anchor_label"", ""phase_label"", ""...",
22,data/day18_step2_dt_Q_v3_vs_v4_common_window.csv,0.75,True,dtQ_curve_or_summary,False,True,False,True,True,False,False,False,"[""param_set"", ""v3_status"", ""common_q_hi_Ah"", ""...",



Role guess counts:
role_guess
dtQ_curve_or_summary       16
event_boundary_metadata     8
npz_trajectory_cache        4
trajectory_or_cache_csv     1
csv_auxiliary               1


In [4]:
# Cell 3 — Inspect Day18 v4 charge-first trajectory schema
#
# Purpose:
#   Inspect the main PyBaMM full-protocol trajectory cache for Day20:
#       data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
#
# Outputs:
#   data/day20_step1_day18_v4_trajectory_schema.csv
#
# Goal:
#   Determine which fields are available for segmented audit:
#     - time
#     - Q_net
#     - voltage
#     - current, if available
#     - protocol / param-set naming structure

import json
import numpy as np
import pandas as pd
from pathlib import Path

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"

z = np.load(TRAJ_V4, allow_pickle=True)
keys = list(z.keys())

print(f"Loaded: {TRAJ_V4}")
print(f"n_keys = {len(keys)}")

rows = []

for k in keys:
    arr = np.asarray(z[k])
    kl = k.lower()

    # Expected naming: <param_set>__<protocol>__<field>
    parts = k.split("__")
    if len(parts) >= 3:
        param_set = parts[0]
        protocol = parts[1]
        field = "__".join(parts[2:])
    elif len(parts) == 2:
        param_set = parts[0]
        protocol = ""
        field = parts[1]
    else:
        param_set = ""
        protocol = ""
        field = k

    role = "unknown"
    if field.lower() in ["t", "t_s", "time", "time_s"]:
        role = "time"
    elif "q" in field.lower():
        role = "Q"
    elif field.lower().startswith("v") or "voltage" in field.lower():
        role = "voltage"
    elif field.lower().startswith("i") or "current" in field.lower():
        role = "current"
    elif "soc" in field.lower():
        role = "soc"

    finite = np.isfinite(arr).all() if np.issubdtype(arr.dtype, np.number) else False

    rows.append({
        "key": k,
        "param_set": param_set,
        "protocol": protocol,
        "field": field,
        "role": role,
        "shape": str(arr.shape),
        "dtype": str(arr.dtype),
        "size": int(arr.size),
        "finite_all": bool(finite),
        "min": float(np.nanmin(arr)) if arr.size and np.issubdtype(arr.dtype, np.number) else np.nan,
        "max": float(np.nanmax(arr)) if arr.size and np.issubdtype(arr.dtype, np.number) else np.nan,
    })

schema_df = pd.DataFrame(rows)

out = DATA / "day20_step1_day18_v4_trajectory_schema.csv"
schema_df.to_csv(out, index=False)

print(f"Wrote: {out}")

print("\nRole counts:")
print(schema_df["role"].value_counts(dropna=False).to_string())

print("\nParam/protocol/field preview:")
display(
    schema_df[
        ["key", "param_set", "protocol", "field", "role", "shape", "min", "max"]
    ].head(80)
)

print("\nProtocol coverage:")
coverage = (
    schema_df
    .groupby(["param_set", "protocol"])["role"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="roles")
)
display(coverage)

Loaded: /Users/louislu/pybamm-dcac-superimposed/data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
n_keys = 24
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step1_day18_v4_trajectory_schema.csv

Role counts:
role
time       6
voltage    6
current    6
Q          6

Param/protocol/field preview:


,key,param_set,protocol,field,role,shape,min,max
0,Chen2020__DC_0p2C__t,Chen2020,DC_0p2C,t,time,"(9464,)",0.000000,16659.715676
1,Chen2020__DC_0p2C__V,Chen2020,DC_0p2C,V,voltage,"(9464,)",3.157897,4.200000
2,Chen2020__DC_0p2C__I,Chen2020,DC_0p2C,I,current,"(9464,)",-1.000000,-1.000000
3,Chen2020__DC_0p2C__Q_net,Chen2020,DC_0p2C,Q_net,Q,"(9464,)",0.000000,4.627699
4,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__t,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,t,time,"(8964,)",0.000000,12799.119672
5,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__V,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,V,voltage,"(8964,)",3.157897,4.200000
6,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__I,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,I,current,"(8964,)",-3.500000,1.500000
7,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__Q_net,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,Q_net,Q,"(8964,)",0.000000,3.770074
8,OKane2022__DC_0p2C__t,OKane2022,DC_0p2C,t,time,"(9403,)",0.000000,16548.641555
9,OKane2022__DC_0p2C__V,OKane2022,DC_0p2C,V,voltage,"(9403,)",3.177915,4.200000



Protocol coverage:


,param_set,protocol,roles
0,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
1,Chen2020,DC_0p2C,"[Q, current, time, voltage]"
2,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
3,OKane2022,DC_0p2C,"[Q, current, time, voltage]"
4,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
5,ORegan2022,DC_0p2C,"[Q, current, time, voltage]"


In [6]:
# Cell 4 — Day18 v4 charge-first event-boundary and segment audit
#
# Purpose:
#   Use trajectory-level t, V, I, Q_net arrays from
#       day18_step1_phase_audit_trajectories_v4_charge_first.npz
#   and metadata from
#       day18_step1_phase_audit_matrix_v4_charge_first.csv
#       day18_step2_dt_Q_audit_v4_charge_first.csv
#
#   to define available PyBaMM full-protocol / voltage-boundary segments.
#
# Important scope note:
#   The available v4 trajectory cache appears to be voltage-limit / phase-audit
#   trajectory data, not a full CC+CV trajectory with CV current decay.
#   This cell therefore audits event-boundary segmentation first.
#
# Outputs:
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv

import numpy as np
import pandas as pd
from pathlib import Path

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
MATRIX_V4 = DATA / "day18_step1_phase_audit_matrix_v4_charge_first.csv"
AUDIT_V4 = DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv"

assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"
assert MATRIX_V4.exists(), f"Missing: {MATRIX_V4}"
assert AUDIT_V4.exists(), f"Missing: {AUDIT_V4}"

z = np.load(TRAJ_V4, allow_pickle=True)
matrix = pd.read_csv(MATRIX_V4)
audit = pd.read_csv(AUDIT_V4)

print(f"Loaded trajectory cache: {TRAJ_V4.name}")
print(f"Loaded matrix metadata : {MATRIX_V4.name} shape={matrix.shape}")
print(f"Loaded dtQ audit table : {AUDIT_V4.name} shape={audit.shape}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def get_traj(param_set, protocol):
    """
    Extract t, V, I, Q arrays for one param_set / protocol.
    Expected keys:
        <param_set>__<protocol>__t
        <param_set>__<protocol>__V
        <param_set>__<protocol>__I
        <param_set>__<protocol>__Q_net
    Q_net is in Ah in the v4 trajectory cache.
    """
    keys = {
        "t": f"{param_set}__{protocol}__t",
        "V": f"{param_set}__{protocol}__V",
        "I": f"{param_set}__{protocol}__I",
        "Q": f"{param_set}__{protocol}__Q_net",
    }

    missing = [k for k in keys.values() if k not in z.files]
    if missing:
        raise KeyError(f"Missing trajectory keys for {param_set}/{protocol}: {missing}")

    out = {
        "t_s": np.asarray(z[keys["t"]], dtype=float),
        "V_V": np.asarray(z[keys["V"]], dtype=float),
        "I_A": np.asarray(z[keys["I"]], dtype=float),
        "Q_Ah": np.asarray(z[keys["Q"]], dtype=float),
    }

    n = len(out["t_s"])
    for name, arr in out.items():
        if len(arr) != n:
            raise ValueError(f"{param_set}/{protocol}: length mismatch for {name}: {len(arr)} vs {n}")
        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{param_set}/{protocol}: non-finite values in {name}")

    if n > 1 and not np.all(np.diff(out["t_s"]) > 0):
        raise ValueError(f"{param_set}/{protocol}: t_s not strictly increasing")

    return out


def first_crossing_event(t_s, y, q_Ah, threshold):
    """
    First y >= threshold event with linear interpolation in t and Q.
    Returns NaN fields if no crossing.
    """
    t_s = np.asarray(t_s, dtype=float)
    y = np.asarray(y, dtype=float)
    q_Ah = np.asarray(q_Ah, dtype=float)

    crossed = y >= threshold

    if not crossed.any():
        return {
            "event_found": False,
            "event_idx": np.nan,
            "t_event_s": np.nan,
            "Q_event_Ah": np.nan,
            "y_event": np.nan,
        }

    idx = int(np.argmax(crossed))

    if idx == 0:
        return {
            "event_found": True,
            "event_idx": idx,
            "t_event_s": float(t_s[0]),
            "Q_event_Ah": float(q_Ah[0]),
            "y_event": float(y[0]),
        }

    y0, y1 = y[idx - 1], y[idx]
    t0, t1 = t_s[idx - 1], t_s[idx]
    q0, q1 = q_Ah[idx - 1], q_Ah[idx]

    if y1 == y0:
        frac = 1.0
    else:
        frac = (threshold - y0) / (y1 - y0)
        frac = float(np.clip(frac, 0.0, 1.0))

    return {
        "event_found": True,
        "event_idx": idx,
        "t_event_s": float(t0 + frac * (t1 - t0)),
        "Q_event_Ah": float(q0 + frac * (q1 - q0)),
        "y_event": float(threshold),
    }


def trajectory_summary(param_set, protocol, traj, meta_row):
    """
    Summarize one trajectory and compare event crossing with stored metadata.
    """
    Vmax_cutoff = float(meta_row["V_max_cutoff"])
    event = first_crossing_event(
        t_s=traj["t_s"],
        y=traj["V_V"],
        q_Ah=traj["Q_Ah"],
        threshold=Vmax_cutoff,
    )

    t_meta = float(meta_row["t_to_Vmax_s"]) if pd.notna(meta_row["t_to_Vmax_s"]) else np.nan
    q_meta = float(meta_row["Q_to_Vmax_Ah"]) if pd.notna(meta_row["Q_to_Vmax_Ah"]) else np.nan

    post_mask = traj["t_s"] > event["t_event_s"] + 1e-9 if event["event_found"] else np.zeros_like(traj["t_s"], dtype=bool)
    n_post = int(post_mask.sum())

    # Heuristic: a true CV tail would have nontrivial samples after voltage event
    # and current drift after the event. In the v4 phase-audit cache this is expected false.
    if n_post >= 5:
        I_post = traj["I_A"][post_mask]
        I_post_range = float(np.nanmax(I_post) - np.nanmin(I_post))
    else:
        I_post_range = np.nan

    has_post_event_tail = n_post >= 5
    has_cv_like_current_decay = bool(has_post_event_tail and np.isfinite(I_post_range) and I_post_range > 0.05)

    return {
        "param_set": param_set,
        "protocol": protocol,
        "phase_label": meta_row.get("phase_label", np.nan),
        "anchor_label": meta_row.get("anchor_label", np.nan),

        "n_samples": len(traj["t_s"]),
        "t_start_s": float(traj["t_s"][0]),
        "t_end_s": float(traj["t_s"][-1]),
        "Q_start_Ah": float(traj["Q_Ah"][0]),
        "Q_end_Ah": float(traj["Q_Ah"][-1]),
        "V_min_V": float(np.nanmin(traj["V_V"])),
        "V_max_V": float(np.nanmax(traj["V_V"])),
        "I_min_A": float(np.nanmin(traj["I_A"])),
        "I_max_A": float(np.nanmax(traj["I_A"])),
        "I_initial_A": float(traj["I_A"][0]),
        "I_final_A": float(traj["I_A"][-1]),

        "V_max_cutoff": Vmax_cutoff,
        "event_found_calc": event["event_found"],
        "event_idx_calc": event["event_idx"],
        "t_to_Vmax_calc_s": event["t_event_s"],
        "Q_to_Vmax_calc_Ah": event["Q_event_Ah"],
        "t_to_Vmax_meta_s": t_meta,
        "Q_to_Vmax_meta_Ah": q_meta,
        "t_to_Vmax_err_s": event["t_event_s"] - t_meta if np.isfinite(t_meta) else np.nan,
        "Q_to_Vmax_err_Ah": event["Q_event_Ah"] - q_meta if np.isfinite(q_meta) else np.nan,

        "n_post_event_samples": n_post,
        "has_post_event_tail": has_post_event_tail,
        "I_post_event_range_A": I_post_range,
        "has_cv_like_current_decay": has_cv_like_current_decay,

        "termination_raw": meta_row.get("termination_raw", np.nan),
        "feasibility_verdict": meta_row.get("feasibility_verdict", np.nan),
        "frac_below_Vmin": meta_row.get("frac_below_Vmin", np.nan),
        "frac_above_Vmax": meta_row.get("frac_above_Vmax", np.nan),
    }


# ---------------------------------------------------------------------
# Build event-boundary audit table
# ---------------------------------------------------------------------

required_matrix_cols = {
    "param_set", "protocol", "phase_label", "anchor_label",
    "V_max_cutoff", "Q_to_Vmax_Ah", "t_to_Vmax_s",
    "termination_raw", "feasibility_verdict",
}
missing = required_matrix_cols - set(matrix.columns)
assert not missing, f"Missing matrix columns: {missing}"

event_rows = []

# Use only protocols present in the v4 trajectory cache
schema_path = DATA / "day20_step1_day18_v4_trajectory_schema.csv"
schema = pd.read_csv(schema_path)
coverage = schema.groupby(["param_set", "protocol"])["role"].apply(lambda x: set(x)).reset_index()

for _, row in coverage.iterrows():
    param_set = row["param_set"]
    protocol = row["protocol"]
    roles = row["role"]

    if not {"time", "voltage", "current", "Q"}.issubset(roles):
        continue

    m = matrix[(matrix["param_set"] == param_set) & (matrix["protocol"] == protocol)]
    if len(m) != 1:
        raise ValueError(f"Expected one metadata row for {param_set}/{protocol}, got {len(m)}")

    meta_row = m.iloc[0]
    traj = get_traj(param_set, protocol)
    event_rows.append(trajectory_summary(param_set, protocol, traj, meta_row))

event_df = pd.DataFrame(event_rows)

out_event = DATA / "day20_step2_v4_event_boundary_audit.csv"
event_df.to_csv(out_event, index=False)

print(f"Wrote: {out_event}")
display(event_df)


# ---------------------------------------------------------------------
# Pair-level segment boundaries
# ---------------------------------------------------------------------

audit_required = {"param_set", "anchor_label", "phase_label", "f_anchor_Hz", "q_lo_full_Ah", "q_lo_stable_added_Ah", "q_hi_Ah"}
missing_audit = audit_required - set(audit.columns)
assert not missing_audit, f"Missing audit columns: {missing_audit}"

segment_rows = []

for _, ar in audit.iterrows():
    param_set = ar["param_set"]
    anchor_label = ar["anchor_label"]
    phase_label = ar["phase_label"]

    pair_meta = matrix[
        (matrix["param_set"] == param_set)
        & (matrix["anchor_label"] == anchor_label)
        & (matrix["phase_label"] == phase_label)
    ]

    dc_rows = pair_meta[pair_meta["protocol"].str.startswith("DC_")]
    dcac_rows = pair_meta[pair_meta["protocol"].str.startswith("DCAC_")]

    if len(dc_rows) != 1 or len(dcac_rows) != 1:
        raise ValueError(
            f"Expected one DC and one DCAC row for {param_set}/{anchor_label}, "
            f"got DC={len(dc_rows)}, DCAC={len(dcac_rows)}"
        )

    dc = dc_rows.iloc[0]
    dcac = dcac_rows.iloc[0]

    Q_vmax_DC = float(dc["Q_to_Vmax_Ah"])
    Q_vmax_DCAC = float(dcac["Q_to_Vmax_Ah"])
    t_vmax_DC = float(dc["t_to_Vmax_s"])
    t_vmax_DCAC = float(dcac["t_to_Vmax_s"])

    q_A_lo = float(ar["q_lo_stable_added_Ah"])
    q_A_hi = float(ar["q_hi_Ah"])

    # Segment B is meaningful when DCAC reaches Vmax at lower Q than DC.
    q_B_lo = min(Q_vmax_DCAC, Q_vmax_DC)
    q_B_hi = max(Q_vmax_DCAC, Q_vmax_DC)
    boundary_span_Ah = q_B_hi - q_B_lo

    # If the stored stable Q window ends below the earliest Vmax, then A corresponds to
    # prescribed-current shared window and B is outside the stored dt(Q) window.
    A_before_earliest_event = q_A_hi <= min(Q_vmax_DC, Q_vmax_DCAC) + 1e-9

    segment_rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,
        "protocol_DC": dc["protocol"],
        "protocol_DCAC": dcac["protocol"],

        "f_anchor_Hz": float(ar["f_anchor_Hz"]),
        "Q_nom_Ah": float(ar["Q_nom_Ah"]),
        "T_period_s": float(ar["T_period_s"]),

        "Vmax_cutoff_DC": float(dc["V_max_cutoff"]),
        "Vmax_cutoff_DCAC": float(dcac["V_max_cutoff"]),

        "Q_to_Vmax_DC_Ah": Q_vmax_DC,
        "Q_to_Vmax_DCAC_Ah": Q_vmax_DCAC,
        "Q_to_Vmax_shift_Ah": Q_vmax_DC - Q_vmax_DCAC,
        "Q_to_Vmax_shift_pct_nom": (Q_vmax_DC - Q_vmax_DCAC) / float(ar["Q_nom_Ah"]) * 100.0,

        "t_to_Vmax_DC_s": t_vmax_DC,
        "t_to_Vmax_DCAC_s": t_vmax_DCAC,
        "t_to_Vmax_shift_s": t_vmax_DC - t_vmax_DCAC,

        "segment_A_name": "AC-on prescribed-current shared dt(Q) window",
        "segment_A_Q_lo_Ah": q_A_lo,
        "segment_A_Q_hi_Ah": q_A_hi,
        "segment_A_before_earliest_Vmax": A_before_earliest_event,

        "segment_B_name": "voltage-boundary event separation interval",
        "segment_B_Q_lo_Ah": q_B_lo,
        "segment_B_Q_hi_Ah": q_B_hi,
        "segment_B_span_Ah": boundary_span_Ah,
        "segment_B_available_in_current_npz": False,

        "has_cv_feedback_in_current_npz": bool(
            event_df[
                (event_df["param_set"] == param_set)
                & (event_df["protocol"].isin([dc["protocol"], dcac["protocol"]]))
            ]["has_cv_like_current_decay"].any()
        ),

        "day20_scope_for_this_pair": (
            "CC/event-boundary audit only; CV feedback requires additional full CC+CV trajectory"
        ),
    })

segment_df = pd.DataFrame(segment_rows)

out_segment = DATA / "day20_step2_v4_segment_boundaries.csv"
segment_df.to_csv(out_segment, index=False)

print(f"\nWrote: {out_segment}")
display(segment_df)

print("\nBoundary shift summary:")
display(
    segment_df[
        [
            "param_set",
            "Q_to_Vmax_DC_Ah",
            "Q_to_Vmax_DCAC_Ah",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "segment_A_before_earliest_Vmax",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Validation checks
# ---------------------------------------------------------------------
# Metadata event locations are authoritative. The trajectory-derived event
# time is a sampled-trajectory reconstruction and can differ by several
# seconds near solver termination. We therefore use a relaxed time tolerance
# and a tighter Q tolerance.

T_EVENT_RECON_TOL_S = 15.0
Q_EVENT_RECON_TOL_AH = 5e-3

event_df["event_time_reconstruction_ok"] = event_df["t_to_Vmax_err_s"].abs() < T_EVENT_RECON_TOL_S
event_df["event_Q_reconstruction_ok"] = event_df["Q_to_Vmax_err_Ah"].abs() < Q_EVENT_RECON_TOL_AH

# Re-save event table with validation flags
event_df.to_csv(out_event, index=False)

print("\nEvent reconstruction validation:")
print(
    event_df[
        [
            "param_set", "protocol",
            "t_to_Vmax_calc_s", "t_to_Vmax_meta_s", "t_to_Vmax_err_s",
            "Q_to_Vmax_calc_Ah", "Q_to_Vmax_meta_Ah", "Q_to_Vmax_err_Ah",
            "event_time_reconstruction_ok",
            "event_Q_reconstruction_ok",
        ]
    ].to_string(index=False)
)

assert event_df["event_found_calc"].all(), (
    "At least one trajectory did not reach Vmax cutoff."
)

assert event_df["event_time_reconstruction_ok"].all(), (
    f"Trajectory-derived t_to_Vmax differs from metadata by >{T_EVENT_RECON_TOL_S} s:\n"
    f"{event_df[~event_df['event_time_reconstruction_ok']][['param_set', 'protocol', 't_to_Vmax_calc_s', 't_to_Vmax_meta_s', 't_to_Vmax_err_s']]}"
)

assert event_df["event_Q_reconstruction_ok"].all(), (
    f"Trajectory-derived Q_to_Vmax differs from metadata by >{Q_EVENT_RECON_TOL_AH} Ah:\n"
    f"{event_df[~event_df['event_Q_reconstruction_ok']][['param_set', 'protocol', 'Q_to_Vmax_calc_Ah', 'Q_to_Vmax_meta_Ah', 'Q_to_Vmax_err_Ah']]}"
)

assert not event_df["has_cv_like_current_decay"].any(), (
    "Unexpected CV-like current decay found in current v4 trajectory cache."
)

print("\n" + "=" * 72)
print("Cell 4 PASSED — Day18 v4 event-boundary audit complete")
print("Scope confirmed: CC/event-boundary only; no CV feedback trajectory in current NPZ.")
print("=" * 72)

Loaded trajectory cache: day18_step1_phase_audit_trajectories_v4_charge_first.npz
Loaded matrix metadata : day18_step1_phase_audit_matrix_v4_charge_first.csv shape=(6, 36)
Loaded dtQ audit table : day18_step2_dt_Q_audit_v4_charge_first.csv shape=(3, 71)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step2_v4_event_boundary_audit.csv


,param_set,protocol,phase_label,anchor_label,n_samples,t_start_s,t_end_s,Q_start_Ah,Q_end_Ah,V_min_V,...,t_to_Vmax_err_s,Q_to_Vmax_err_Ah,n_post_event_samples,has_post_event_tail,I_post_event_range_A,has_cv_like_current_decay,termination_raw,feasibility_verdict,frac_below_Vmin,frac_above_Vmax
0,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,8964,0.0,12799.119672,0.0,3.770074,3.157897,...,1.181164e+00,1.130151e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000223
1,Chen2020,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,9464,0.0,16659.715676,0.0,4.627699,3.157897,...,9.715676e+00,2.698799e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000634
2,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,9164,0.0,13045.766501,0.0,3.790253,3.177915,...,3.759816e+00,3.430599e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000327
3,OKane2022,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,9403,0.0,16548.641555,0.0,4.596845,3.177915,...,8.641555e+00,2.400432e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000638
4,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,7798,0.0,9925.854020,0.0,3.210457,3.099282,...,-3.637979e-12,-8.881784e-16,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000128
5,ORegan2022,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,12378,0.0,17006.861901,0.0,4.724128,3.099282,...,0.000000e+00,0.000000e+00,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000081



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step2_v4_segment_boundaries.csv


,param_set,anchor_label,phase_label,protocol_DC,protocol_DCAC,f_anchor_Hz,Q_nom_Ah,T_period_s,Vmax_cutoff_DC,Vmax_cutoff_DCAC,...,segment_A_Q_lo_Ah,segment_A_Q_hi_Ah,segment_A_before_earliest_Vmax,segment_B_name,segment_B_Q_lo_Ah,segment_B_Q_hi_Ah,segment_B_span_Ah,segment_B_available_in_current_npz,has_cv_feedback_in_current_npz,day20_scope_for_this_pair
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000408,5.0,2453.513699,4.2,4.2,...,1.0,3.668944,True,voltage-boundary event separation interval,3.768944,4.625000,0.856056,False,False,CC/event-boundary audit only; CV feedback requ...
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000397,5.0,2516.435063,4.2,4.2,...,1.0,3.686822,True,voltage-boundary event separation interval,3.786822,4.594444,0.807622,False,False,CC/event-boundary audit only; CV feedback requ...
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000333,5.0,2998.832183,4.4,4.4,...,1.0,3.110457,True,voltage-boundary event separation interval,3.210457,4.724128,1.513671,False,False,CC/event-boundary audit only; CV feedback requ...



Boundary shift summary:


,param_set,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,Q_to_Vmax_shift_Ah,Q_to_Vmax_shift_pct_nom,t_to_Vmax_shift_s,segment_A_before_earliest_Vmax,has_cv_feedback_in_current_npz
0,Chen2020,4.625000,3.768944,0.856056,17.121118,3852.061493,True,False
1,OKane2022,4.594444,3.786822,0.807622,16.152443,3497.993316,True,False
2,ORegan2022,4.724128,3.210457,1.513671,30.273420,7081.007881,True,False



Event reconstruction validation:
 param_set                 protocol  t_to_Vmax_calc_s  t_to_Vmax_meta_s  t_to_Vmax_err_s  Q_to_Vmax_calc_Ah  Q_to_Vmax_meta_Ah  Q_to_Vmax_err_Ah  event_time_reconstruction_ok  event_Q_reconstruction_ok
  Chen2020 DCAC_DC0p2C_AC0p5C_10tau      12799.119672      12797.938507     1.181164e+00           3.770074           3.768944      1.130151e-03                          True                       True
  Chen2020                  DC_0p2C      16659.715676      16650.000000     9.715676e+00           4.627699           4.625000      2.698799e-03                          True                       True
 OKane2022 DCAC_DC0p2C_AC0p5C_10tau      13045.766501      13042.006684     3.759816e+00           3.790253           3.786822      3.430599e-03                          True                       True
 OKane2022                  DC_0p2C      16548.641555      16540.000000     8.641555e+00           4.596845           4.594444      2.400432e-03              

In [7]:
# Cell 5 — Segment A geometry / residual audit for Day18 v4 charge-first
#
# Purpose:
#   Audit Segment A:
#       AC-on prescribed-current shared-Q window before earliest Vmax event
#
# Inputs:
#   data/day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz
#   data/day18_step2_dt_Q_audit_v4_charge_first.csv
#   data/day20_step2_v4_segment_boundaries.csv
#
# Outputs:
#   data/day20_step3_segment_A_geometry_residual_curves.csv
#   data/day20_step3_segment_A_geometry_residual_summary.csv
#
# Interpretation:
#   If Segment A residual remains near zero, then the charge-first v4
#   prescribed-current shared-Q region is geometry-dominated.
#
# Scope boundary:
#   This does not audit voltage-boundary interval, AC-off transition, or CV feedback.

import numpy as np
import pandas as pd

CURVES_V4 = DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz"
AUDIT_V4 = DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"

assert CURVES_V4.exists(), f"Missing: {CURVES_V4}"
assert AUDIT_V4.exists(), f"Missing: {AUDIT_V4}"
assert SEGMENTS.exists(), f"Missing: {SEGMENTS}"

curves = pd.read_csv(CURVES_V4, compression="gzip")
audit = pd.read_csv(AUDIT_V4)
segments = pd.read_csv(SEGMENTS)

required_curves = {
    "param_set", "anchor_label", "phase_label",
    "Q_Ah", "t_DC_s", "t_DCAC_s",
    "dtQ_s", "dtQ_geom_s", "dtQ_resid_s",
    "window_full", "window_stable_added",
    "protocol_DC", "protocol_DCAC", "status",
}
missing_curves = required_curves - set(curves.columns)
assert not missing_curves, f"Curves missing columns: {missing_curves}"

required_segments = {
    "param_set", "anchor_label", "phase_label",
    "protocol_DC", "protocol_DCAC",
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "segment_A_before_earliest_Vmax",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "t_to_Vmax_shift_s",
    "has_cv_feedback_in_current_npz",
}
missing_segments = required_segments - set(segments.columns)
assert not missing_segments, f"Segments missing columns: {missing_segments}"

required_audit = {
    "param_set", "anchor_label", "phase_label",
    "Q_nom_Ah", "f_anchor_Hz",
    "full_geom_mean_s", "full_resid_mean_s", "full_resid_max_abs_s",
    "stable_geom_mean_s", "stable_resid_mean_s", "stable_resid_max_abs_s",
}
missing_audit = required_audit - set(audit.columns)
assert not missing_audit, f"Audit table missing columns: {missing_audit}"


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def as_bool_series(s):
    """
    Robustly convert bool-like column values to bool.
    Handles True/False, 1/0, and string variants.
    """
    if s.dtype == bool:
        return s

    return (
        s.astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        })
        .fillna(False)
        .astype(bool)
    )


def sign_topology(x, threshold_s=1.0):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)

    if not finite.any():
        return "invalid"

    xx = x[finite]
    pos = np.any(xx > threshold_s)
    neg = np.any(xx < -threshold_s)

    if pos and neg:
        return "mixed"
    if pos:
        return "positive_only"
    if neg:
        return "negative_only"
    return "near_zero"


def safe_abs_ratio(num, den, eps=1e-12):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full_like(num, np.nan, dtype=float)
    mask = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > eps)
    out[mask] = np.abs(num[mask]) / np.abs(den[mask])
    return out


def segment_A_classification(max_abs_resid, p95_abs_resid, median_abs_resid):
    if not np.isfinite(max_abs_resid):
        return "invalid"
    if max_abs_resid < 0.1 and p95_abs_resid < 0.01:
        return "numerical_null"
    if max_abs_resid < 1.0:
        return "near_zero"
    if p95_abs_resid < 1.0:
        return "spiky_but_stable_near_zero"
    if p95_abs_resid < 10.0:
        return "weak_residual"
    return "requires_inspection"


# ---------------------------------------------------------------------
# Join curves with segment boundaries and audit metadata
# ---------------------------------------------------------------------

join_keys = ["param_set", "anchor_label", "phase_label", "protocol_DC", "protocol_DCAC"]

seg_cols = join_keys + [
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "segment_A_before_earliest_Vmax",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "Q_to_Vmax_shift_pct_nom",
    "t_to_Vmax_shift_s",
    "has_cv_feedback_in_current_npz",
]

segments_meta = segments[seg_cols].drop_duplicates(join_keys)

curves_join = curves.merge(
    segments_meta,
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_join = curves_join[curves_join["_merge"] != "both"]
assert missing_join.empty, (
    "Some curves failed to join segment boundaries:\n"
    f"{missing_join[join_keys].drop_duplicates().head(20)}"
)

curves_join = curves_join.drop(columns=["_merge"])

audit_meta_cols = [
    "param_set", "anchor_label", "phase_label",
    "Q_nom_Ah", "f_anchor_Hz", "T_period_s",
    "full_geom_mean_s", "full_resid_mean_s", "full_resid_max_abs_s",
    "stable_geom_mean_s", "stable_resid_mean_s", "stable_resid_max_abs_s",
]

audit_meta = audit[audit_meta_cols].drop_duplicates(["param_set", "anchor_label", "phase_label"])

curves_join = curves_join.merge(
    audit_meta,
    on=["param_set", "anchor_label", "phase_label"],
    how="left",
    validate="many_to_one",
)

# Numeric coercion
num_cols = [
    "Q_Ah", "t_DC_s", "t_DCAC_s",
    "dtQ_s", "dtQ_geom_s", "dtQ_resid_s",
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "t_to_Vmax_shift_s",
    "Q_nom_Ah", "f_anchor_Hz", "T_period_s",
]
for c in num_cols:
    curves_join[c] = pd.to_numeric(curves_join[c], errors="coerce")

curves_join["window_full_bool"] = as_bool_series(curves_join["window_full"])
curves_join["window_stable_added_bool"] = as_bool_series(curves_join["window_stable_added"])

curves_join["inside_segment_A_by_boundary"] = (
    (curves_join["Q_Ah"] >= curves_join["segment_A_Q_lo_Ah"] - 1e-12)
    & (curves_join["Q_Ah"] <= curves_join["segment_A_Q_hi_Ah"] + 1e-12)
)

curves_join["segment_A_row"] = (
    curves_join["inside_segment_A_by_boundary"]
    & curves_join["window_stable_added_bool"]
)

# Use segment_A_row as primary filter.
segA = curves_join[curves_join["segment_A_row"]].copy()

assert len(segA) > 0, "No Segment A rows found."

# ---------------------------------------------------------------------
# Row-level diagnostics
# ---------------------------------------------------------------------

segA["dtQ_identity_err_s"] = segA["dtQ_s"] - (
    segA["dtQ_geom_s"] + segA["dtQ_resid_s"]
)

segA["abs_resid_s"] = segA["dtQ_resid_s"].abs()
segA["abs_geom_s"] = segA["dtQ_geom_s"].abs()
segA["abs_model_s"] = segA["dtQ_s"].abs()
segA["resid_over_geom"] = safe_abs_ratio(segA["dtQ_resid_s"], segA["dtQ_geom_s"])
segA["resid_over_model"] = safe_abs_ratio(segA["dtQ_resid_s"], segA["dtQ_s"])

# ---------------------------------------------------------------------
# Summary per param_set / anchor
# ---------------------------------------------------------------------

summary_rows = []

for keys, g in segA.groupby(["param_set", "anchor_label", "phase_label"], sort=False):
    param_set, anchor_label, phase_label = keys

    dt_model = g["dtQ_s"].to_numpy(dtype=float)
    dt_geom = g["dtQ_geom_s"].to_numpy(dtype=float)
    dt_resid = g["dtQ_resid_s"].to_numpy(dtype=float)
    abs_resid = np.abs(dt_resid)

    resid_over_geom = g["resid_over_geom"].to_numpy(dtype=float)
    resid_over_model = g["resid_over_model"].to_numpy(dtype=float)

    finite = np.isfinite(dt_model) & np.isfinite(dt_geom) & np.isfinite(dt_resid)

    max_abs_resid = float(np.nanmax(abs_resid)) if len(abs_resid) else np.nan
    p95_abs_resid = float(np.nanquantile(abs_resid, 0.95)) if len(abs_resid) else np.nan
    median_abs_resid = float(np.nanmedian(abs_resid)) if len(abs_resid) else np.nan

    row0 = g.iloc[0]

    summary_rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,

        "n_rows_segment_A": len(g),
        "n_finite": int(finite.sum()),
        "n_nonfinite": int((~finite).sum()),

        "Q_lo_segment_A_Ah": float(row0["segment_A_Q_lo_Ah"]),
        "Q_hi_segment_A_Ah": float(row0["segment_A_Q_hi_Ah"]),
        "Q_min_Ah": float(g["Q_Ah"].min()),
        "Q_max_Ah": float(g["Q_Ah"].max()),
        "segment_A_before_earliest_Vmax": bool(row0["segment_A_before_earliest_Vmax"]),

        "Q_to_Vmax_DC_Ah": float(row0["Q_to_Vmax_DC_Ah"]),
        "Q_to_Vmax_DCAC_Ah": float(row0["Q_to_Vmax_DCAC_Ah"]),
        "Q_to_Vmax_shift_Ah": float(row0["Q_to_Vmax_shift_Ah"]),
        "Q_to_Vmax_shift_pct_nom": float(row0["Q_to_Vmax_shift_pct_nom"]),
        "t_to_Vmax_shift_s": float(row0["t_to_Vmax_shift_s"]),
        "has_cv_feedback_in_current_npz": bool(row0["has_cv_feedback_in_current_npz"]),

        "dt_model_mean_s": float(np.nanmean(dt_model)),
        "dt_model_median_s": float(np.nanmedian(dt_model)),
        "dt_model_min_s": float(np.nanmin(dt_model)),
        "dt_model_max_s": float(np.nanmax(dt_model)),
        "dt_model_sign_topology": sign_topology(dt_model),

        "dt_geom_mean_s": float(np.nanmean(dt_geom)),
        "dt_geom_median_s": float(np.nanmedian(dt_geom)),
        "dt_geom_min_s": float(np.nanmin(dt_geom)),
        "dt_geom_max_s": float(np.nanmax(dt_geom)),
        "dt_geom_sign_topology": sign_topology(dt_geom),

        "dt_resid_mean_s": float(np.nanmean(dt_resid)),
        "dt_resid_median_s": float(np.nanmedian(dt_resid)),
        "dt_resid_min_s": float(np.nanmin(dt_resid)),
        "dt_resid_max_s": float(np.nanmax(dt_resid)),
        "dt_resid_mean_abs_s": float(np.nanmean(abs_resid)),
        "dt_resid_median_abs_s": median_abs_resid,
        "dt_resid_p95_abs_s": p95_abs_resid,
        "dt_resid_max_abs_s": max_abs_resid,
        "dt_resid_sign_topology": sign_topology(dt_resid),

        "resid_over_geom_median": float(np.nanmedian(resid_over_geom)),
        "resid_over_geom_p95": float(np.nanquantile(resid_over_geom, 0.95)),
        "resid_over_geom_max": float(np.nanmax(resid_over_geom)),

        "resid_over_model_median": float(np.nanmedian(resid_over_model)),
        "resid_over_model_p95": float(np.nanquantile(resid_over_model, 0.95)),
        "resid_over_model_max": float(np.nanmax(resid_over_model)),

        "dtQ_identity_err_max_abs_s": float(np.nanmax(segA["dtQ_identity_err_s"].abs())),

        "Q_nom_Ah": float(row0["Q_nom_Ah"]),
        "f_anchor_Hz": float(row0["f_anchor_Hz"]),
        "T_period_s": float(row0["T_period_s"]),

        "segment_A_status": segment_A_classification(
            max_abs_resid=max_abs_resid,
            p95_abs_resid=p95_abs_resid,
            median_abs_resid=median_abs_resid,
        ),

        "interpretation": (
            "Segment A is prescribed-current geometry-dominated"
            if max_abs_resid < 1.0
            else "Segment A residual requires inspection"
        ),
    })

summary_df = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_curves = DATA / "day20_step3_segment_A_geometry_residual_curves.csv"
out_summary = DATA / "day20_step3_segment_A_geometry_residual_summary.csv"

segA.to_csv(out_curves, index=False)
summary_df.to_csv(out_summary, index=False)

print(f"Wrote: {out_curves} ({len(segA)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")

display(summary_df)

print("\nSegment A status counts:")
print(summary_df["segment_A_status"].value_counts(dropna=False).to_string())

print("\nKey Segment A metrics:")
display(
    summary_df[
        [
            "param_set",
            "dt_model_median_s",
            "dt_geom_median_s",
            "dt_resid_median_s",
            "dt_resid_p95_abs_s",
            "dt_resid_max_abs_s",
            "resid_over_geom_median",
            "Q_to_Vmax_shift_Ah",
            "t_to_Vmax_shift_s",
            "segment_A_status",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert summary_df["segment_A_before_earliest_Vmax"].all(), (
    "Segment A is not entirely before earliest Vmax for some pair."
)

assert not summary_df["has_cv_feedback_in_current_npz"].any(), (
    "Unexpected CV feedback detected in current v4 trajectory cache."
)

assert (summary_df["n_nonfinite"] == 0).all(), (
    "Non-finite Segment A values detected:\n"
    f"{summary_df[summary_df['n_nonfinite'] != 0][['param_set', 'n_nonfinite']]}"
)

assert (summary_df["dtQ_identity_err_max_abs_s"] < 1e-9).all(), (
    "dtQ_s != dtQ_geom_s + dtQ_resid_s identity violation."
)

print("\n" + "=" * 72)
print("Cell 5 PASSED — Segment A geometry/residual audit complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step3_segment_A_geometry_residual_curves.csv (183 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step3_segment_A_geometry_residual_summary.csv (3 rows)


,param_set,anchor_label,phase_label,n_rows_segment_A,n_finite,n_nonfinite,Q_lo_segment_A_Ah,Q_hi_segment_A_Ah,Q_min_Ah,Q_max_Ah,...,resid_over_geom_max,resid_over_model_median,resid_over_model_p95,resid_over_model_max,dtQ_identity_err_max_abs_s,Q_nom_Ah,f_anchor_Hz,T_period_s,segment_A_status,interpretation
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,62,62,0,1.0,3.668944,1.029000,3.668944,...,0.000009,1.516077e-06,0.000005,0.000009,4.547474e-13,5.0,0.000408,2453.513699,numerical_null,Segment A is prescribed-current geometry-domin...
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,62,62,0,1.0,3.686822,1.033073,3.686822,...,0.000008,1.114545e-06,0.000003,0.000008,4.547474e-13,5.0,0.000397,2516.435063,numerical_null,Segment A is prescribed-current geometry-domin...
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,59,59,0,1.0,3.110457,1.010375,3.110457,...,0.000006,4.892002e-07,0.000003,0.000006,4.547474e-13,5.0,0.000333,2998.832183,numerical_null,Segment A is prescribed-current geometry-domin...



Segment A status counts:
segment_A_status
numerical_null    3

Key Segment A metrics:


,param_set,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,dt_resid_p95_abs_s,dt_resid_max_abs_s,resid_over_geom_median,Q_to_Vmax_shift_Ah,t_to_Vmax_shift_s,segment_A_status,has_cv_feedback_in_current_npz
0,Chen2020,1302.892568,1302.894455,-0.001338,0.007722,0.014259,1.516074e-06,0.856056,3852.061493,numerical_null,False
1,OKane2022,1345.949033,1345.949270,-0.000882,0.003514,0.005144,1.114545e-06,0.807622,3497.993316,numerical_null,False
2,ORegan2022,1378.560563,1378.560482,0.000279,0.005973,0.014275,4.892004e-07,1.513671,7081.007881,numerical_null,False



Cell 5 PASSED — Segment A geometry/residual audit complete


## Segment B — voltage-boundary event separation

Segment A has shown that the charge-first prescribed-current shared-Q window is geometry-dominated:

$$
\Delta t_{\mathrm{model}}(Q)
\approx
\Delta t_{\mathrm{geom}}(Q),
\qquad
\Delta t_{\mathrm{resid}}(Q)
\approx
0
$$

The next question is whether DCAC reaches the voltage boundary substantially earlier than the DC reference.

This cell audits only the voltage-boundary event separation. It does not include AC-off transition or CV feedback.

For later full-protocol simulations, the MJ1 experimental cutoff current must not be transferred as an absolute 50 mA value. It must be normalized as

$$
C_{\mathrm{cutoff}}
=
\frac{0.05}{3.4}
\approx
0.0147C
$$

and applied to each PyBaMM parameter set as

$$
I_{\mathrm{cutoff}}
=
C_{\mathrm{cutoff}}
\cdot
Q_{\mathrm{nom}}
$$

In [8]:
# Cell 6 — Segment B voltage-boundary contribution audit
#
# Purpose:
#   Quantify Segment B:
#       voltage-boundary event separation interval
#
# Segment B is defined by:
#       Q_to_Vmax_DCAC  →  Q_to_Vmax_DC
#
# Important:
#   Current v4 trajectory cache terminates at Vmax and contains no CV feedback tail.
#   Therefore Segment B is not a full t_DCAC(Q) curve beyond Vmax.
#   It is an event-boundary separation audit:
#
#       - At Q where DCAC reaches Vmax, where is DC?
#       - How much Q and time remain before DC reaches Vmax?
#       - How large is the event-level boundary shift?
#
# Inputs:
#   data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv
#
# Output:
#   data/day20_step4_segment_B_voltage_boundary_audit.csv

import numpy as np
import pandas as pd

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
EVENT_AUDIT = DATA / "day20_step2_v4_event_boundary_audit.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"

assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"
assert EVENT_AUDIT.exists(), f"Missing: {EVENT_AUDIT}"
assert SEGMENTS.exists(), f"Missing: {SEGMENTS}"

z = np.load(TRAJ_V4, allow_pickle=True)
event_df = pd.read_csv(EVENT_AUDIT)
segments = pd.read_csv(SEGMENTS)

print(f"Loaded trajectory cache: {TRAJ_V4.name}")
print(f"Loaded event audit      : {EVENT_AUDIT.name} shape={event_df.shape}")
print(f"Loaded segment table    : {SEGMENTS.name} shape={segments.shape}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def get_traj(param_set, protocol):
    keys = {
        "t": f"{param_set}__{protocol}__t",
        "V": f"{param_set}__{protocol}__V",
        "I": f"{param_set}__{protocol}__I",
        "Q": f"{param_set}__{protocol}__Q_net",
    }

    missing = [k for k in keys.values() if k not in z.files]
    if missing:
        raise KeyError(f"Missing trajectory keys for {param_set}/{protocol}: {missing}")

    out = {
        "t_s": np.asarray(z[keys["t"]], dtype=float),
        "V_V": np.asarray(z[keys["V"]], dtype=float),
        "I_A": np.asarray(z[keys["I"]], dtype=float),
        "Q_Ah": np.asarray(z[keys["Q"]], dtype=float),
    }

    n = len(out["t_s"])
    for name, arr in out.items():
        if len(arr) != n:
            raise ValueError(f"{param_set}/{protocol}: length mismatch for {name}")
        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{param_set}/{protocol}: non-finite values in {name}")

    if n > 1 and not np.all(np.diff(out["t_s"]) > 0):
        raise ValueError(f"{param_set}/{protocol}: t_s not strictly increasing")

    return out


def first_passage_by_Q(t_s, Q_Ah, target_Q_Ah, extra_arrays=None):
    """
    Raw first-passage interpolation by Q.

    Returns t(Q*) and interpolated extra variables, e.g. V(Q*), I(Q*).
    """
    t_s = np.asarray(t_s, dtype=float)
    Q_Ah = np.asarray(Q_Ah, dtype=float)

    if extra_arrays is None:
        extra_arrays = {}

    crossed = Q_Ah >= target_Q_Ah

    if not crossed.any():
        out = {
            "found": False,
            "idx": np.nan,
            "t_s": np.nan,
            "Q_Ah": np.nan,
        }
        for name in extra_arrays:
            out[name] = np.nan
        return out

    idx = int(np.argmax(crossed))

    if idx == 0:
        out = {
            "found": True,
            "idx": idx,
            "t_s": float(t_s[0]),
            "Q_Ah": float(Q_Ah[0]),
        }
        for name, arr in extra_arrays.items():
            arr = np.asarray(arr, dtype=float)
            out[name] = float(arr[0])
        return out

    q0, q1 = Q_Ah[idx - 1], Q_Ah[idx]
    t0, t1 = t_s[idx - 1], t_s[idx]

    if q1 == q0:
        frac = 1.0
    else:
        frac = (target_Q_Ah - q0) / (q1 - q0)
        frac = float(np.clip(frac, 0.0, 1.0))

    out = {
        "found": True,
        "idx": idx,
        "t_s": float(t0 + frac * (t1 - t0)),
        "Q_Ah": float(target_Q_Ah),
    }

    for name, arr in extra_arrays.items():
        arr = np.asarray(arr, dtype=float)
        y0, y1 = arr[idx - 1], arr[idx]
        out[name] = float(y0 + frac * (y1 - y0))

    return out


# ---------------------------------------------------------------------
# Validate scope from Cell 4
# ---------------------------------------------------------------------

assert not event_df["has_cv_like_current_decay"].any(), (
    "Current v4 trajectory cache unexpectedly contains CV-like current decay."
)

assert not segments["has_cv_feedback_in_current_npz"].any(), (
    "Segment table reports CV feedback available, but Day20 current scope assumes no CV tail."
)


# ---------------------------------------------------------------------
# Segment B audit
# ---------------------------------------------------------------------

rows = []

for _, seg in segments.iterrows():
    param_set = seg["param_set"]
    protocol_DC = seg["protocol_DC"]
    protocol_DCAC = seg["protocol_DCAC"]

    dc = get_traj(param_set, protocol_DC)
    dcac = get_traj(param_set, protocol_DCAC)

    Q_vmax_DC = float(seg["Q_to_Vmax_DC_Ah"])
    Q_vmax_DCAC = float(seg["Q_to_Vmax_DCAC_Ah"])
    t_vmax_DC = float(seg["t_to_Vmax_DC_s"])
    t_vmax_DCAC = float(seg["t_to_Vmax_DCAC_s"])

    Vmax_DC = float(seg["Vmax_cutoff_DC"])
    Vmax_DCAC = float(seg["Vmax_cutoff_DCAC"])

    Q_nom_Ah = float(seg["Q_nom_Ah"])

    # State of DC arm at the Q where DCAC reaches Vmax
    dc_at_Q_dcac_vmax = first_passage_by_Q(
        t_s=dc["t_s"],
        Q_Ah=dc["Q_Ah"],
        target_Q_Ah=Q_vmax_DCAC,
        extra_arrays={
            "V_DC_at_Q_DCAC_Vmax_V": dc["V_V"],
            "I_DC_at_Q_DCAC_Vmax_A": dc["I_A"],
        },
    )

    # State of DCAC arm at its own Vmax Q, reconstructed from its own trajectory
    dcac_at_Q_dcac_vmax = first_passage_by_Q(
        t_s=dcac["t_s"],
        Q_Ah=dcac["Q_Ah"],
        target_Q_Ah=Q_vmax_DCAC,
        extra_arrays={
            "V_DCAC_at_Q_DCAC_Vmax_V": dcac["V_V"],
            "I_DCAC_at_Q_DCAC_Vmax_A": dcac["I_A"],
        },
    )

    # Decompose event timing shift:
    # total_event_shift =
    #   state_equivalent_gain_at_DCAC_boundary
    #   + DC_remaining_time_from_Q_DCAC_Vmax_to_Q_DC_Vmax
    dt_state_at_DCAC_boundary_s = dc_at_Q_dcac_vmax["t_s"] - t_vmax_DCAC
    dc_remaining_time_to_own_Vmax_s = t_vmax_DC - dc_at_Q_dcac_vmax["t_s"]
    total_event_shift_s = t_vmax_DC - t_vmax_DCAC

    identity_err_s = total_event_shift_s - (
        dt_state_at_DCAC_boundary_s + dc_remaining_time_to_own_Vmax_s
    )

    Q_shift_Ah = Q_vmax_DC - Q_vmax_DCAC
    Q_shift_pct_nom = Q_shift_Ah / Q_nom_Ah * 100.0

    V_DC_at_Q_DCAC = dc_at_Q_dcac_vmax["V_DC_at_Q_DCAC_Vmax_V"]
    voltage_headroom_DC_V = Vmax_DC - V_DC_at_Q_DCAC

    # DC current estimate around the same-Q point
    I_DC_at_Q_DCAC = abs(dc_at_Q_dcac_vmax["I_DC_at_Q_DCAC_Vmax_A"])
    if np.isfinite(I_DC_at_Q_DCAC) and I_DC_at_Q_DCAC > 0:
        dc_remaining_time_from_Q_shift_constant_I_s = Q_shift_Ah * 3600.0 / I_DC_at_Q_DCAC
    else:
        dc_remaining_time_from_Q_shift_constant_I_s = np.nan

    constant_I_time_err_s = dc_remaining_time_to_own_Vmax_s - dc_remaining_time_from_Q_shift_constant_I_s

    rows.append({
        "param_set": param_set,
        "anchor_label": seg["anchor_label"],
        "phase_label": seg["phase_label"],
        "protocol_DC": protocol_DC,
        "protocol_DCAC": protocol_DCAC,

        "Q_nom_Ah": Q_nom_Ah,
        "f_anchor_Hz": float(seg["f_anchor_Hz"]),
        "T_period_s": float(seg["T_period_s"]),

        "Vmax_cutoff_DC": Vmax_DC,
        "Vmax_cutoff_DCAC": Vmax_DCAC,

        "Q_to_Vmax_DC_Ah": Q_vmax_DC,
        "Q_to_Vmax_DCAC_Ah": Q_vmax_DCAC,
        "Q_to_Vmax_shift_Ah": Q_shift_Ah,
        "Q_to_Vmax_shift_pct_nom": Q_shift_pct_nom,

        "t_to_Vmax_DC_s": t_vmax_DC,
        "t_to_Vmax_DCAC_s": t_vmax_DCAC,
        "t_to_Vmax_shift_s": total_event_shift_s,

        "DC_at_Q_DCAC_Vmax_found": dc_at_Q_dcac_vmax["found"],
        "t_DC_at_Q_DCAC_Vmax_s": dc_at_Q_dcac_vmax["t_s"],
        "V_DC_at_Q_DCAC_Vmax_V": V_DC_at_Q_DCAC,
        "I_DC_at_Q_DCAC_Vmax_A": dc_at_Q_dcac_vmax["I_DC_at_Q_DCAC_Vmax_A"],

        "DCAC_at_Q_DCAC_Vmax_found": dcac_at_Q_dcac_vmax["found"],
        "t_DCAC_at_Q_DCAC_Vmax_s_reconstructed": dcac_at_Q_dcac_vmax["t_s"],
        "V_DCAC_at_Q_DCAC_Vmax_V_reconstructed": dcac_at_Q_dcac_vmax["V_DCAC_at_Q_DCAC_Vmax_V"],
        "I_DCAC_at_Q_DCAC_Vmax_A_reconstructed": dcac_at_Q_dcac_vmax["I_DCAC_at_Q_DCAC_Vmax_A"],

        "voltage_headroom_DC_at_DCAC_Vmax_Q_V": voltage_headroom_DC_V,

        "dt_state_at_DCAC_boundary_s": dt_state_at_DCAC_boundary_s,
        "dc_remaining_time_to_own_Vmax_s": dc_remaining_time_to_own_Vmax_s,
        "event_shift_decomposition_identity_err_s": identity_err_s,

        "dc_remaining_time_from_Q_shift_constant_I_s": dc_remaining_time_from_Q_shift_constant_I_s,
        "constant_I_time_err_s": constant_I_time_err_s,

        "segment_A_Q_lo_Ah": float(seg["segment_A_Q_lo_Ah"]),
        "segment_A_Q_hi_Ah": float(seg["segment_A_Q_hi_Ah"]),
        "segment_A_before_earliest_Vmax": bool(seg["segment_A_before_earliest_Vmax"]),

        "segment_B_Q_lo_Ah": float(seg["segment_B_Q_lo_Ah"]),
        "segment_B_Q_hi_Ah": float(seg["segment_B_Q_hi_Ah"]),
        "segment_B_span_Ah": float(seg["segment_B_span_Ah"]),
        "segment_B_available_in_current_npz": bool(seg["segment_B_available_in_current_npz"]),
        "has_cv_feedback_in_current_npz": bool(seg["has_cv_feedback_in_current_npz"]),

        "interpretation": (
            "large voltage-boundary shift; current NPZ stops at Vmax and does not contain CV feedback"
        ),
    })

boundary_df = pd.DataFrame(rows)

# Verdict flags
boundary_df["Q_shift_positive"] = boundary_df["Q_to_Vmax_shift_Ah"] > 0
boundary_df["t_shift_positive"] = boundary_df["t_to_Vmax_shift_s"] > 0
boundary_df["DC_below_Vmax_at_DCAC_boundary"] = boundary_df["voltage_headroom_DC_at_DCAC_Vmax_Q_V"] > 0

boundary_df["boundary_shift_class"] = np.select(
    [
        boundary_df["Q_to_Vmax_shift_pct_nom"] >= 20.0,
        boundary_df["Q_to_Vmax_shift_pct_nom"] >= 10.0,
        boundary_df["Q_to_Vmax_shift_pct_nom"] > 0.0,
    ],
    [
        "very_large_boundary_shift",
        "large_boundary_shift",
        "positive_boundary_shift",
    ],
    default="no_positive_boundary_shift",
)

out = DATA / "day20_step4_segment_B_voltage_boundary_audit.csv"
boundary_df.to_csv(out, index=False)

print(f"Wrote: {out}")
display(boundary_df)

print("\nSegment B boundary shift summary:")
display(
    boundary_df[
        [
            "param_set",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "dt_state_at_DCAC_boundary_s",
            "dc_remaining_time_to_own_Vmax_s",
            "voltage_headroom_DC_at_DCAC_Vmax_Q_V",
            "boundary_shift_class",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert boundary_df["Q_shift_positive"].all(), (
    "Expected DC to reach Vmax at higher Q than DCAC for all v4 pairs."
)

assert boundary_df["t_shift_positive"].all(), (
    "Expected DC to reach Vmax later than DCAC for all v4 pairs."
)

assert boundary_df["DC_below_Vmax_at_DCAC_boundary"].all(), (
    "Expected DC to remain below Vmax at Q where DCAC reaches Vmax."
)

assert (boundary_df["event_shift_decomposition_identity_err_s"].abs() < 1e-6).all(), (
    "Event shift decomposition identity failed:\n"
    f"{boundary_df[['param_set', 'event_shift_decomposition_identity_err_s']]}"
)

assert not boundary_df["has_cv_feedback_in_current_npz"].any(), (
    "Current v4 NPZ unexpectedly contains CV feedback."
)

print("\n" + "=" * 72)
print("Cell 6 PASSED — Segment B voltage-boundary contribution audit complete")
print("=" * 72)

Loaded trajectory cache: day18_step1_phase_audit_trajectories_v4_charge_first.npz
Loaded event audit      : day20_step2_v4_event_boundary_audit.csv shape=(6, 34)
Loaded segment table    : day20_step2_v4_segment_boundaries.csv shape=(3, 28)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step4_segment_B_voltage_boundary_audit.csv


,param_set,anchor_label,phase_label,protocol_DC,protocol_DCAC,Q_nom_Ah,f_anchor_Hz,T_period_s,Vmax_cutoff_DC,Vmax_cutoff_DCAC,...,segment_B_Q_lo_Ah,segment_B_Q_hi_Ah,segment_B_span_Ah,segment_B_available_in_current_npz,has_cv_feedback_in_current_npz,interpretation,Q_shift_positive,t_shift_positive,DC_below_Vmax_at_DCAC_boundary,boundary_shift_class
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000408,2453.513699,4.2,4.2,...,3.768944,4.625000,0.856056,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,large_boundary_shift
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000397,2516.435063,4.2,4.2,...,3.786822,4.594444,0.807622,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,large_boundary_shift
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000333,2998.832183,4.4,4.4,...,3.210457,4.724128,1.513671,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,very_large_boundary_shift



Segment B boundary shift summary:


,param_set,Q_to_Vmax_shift_Ah,Q_to_Vmax_shift_pct_nom,t_to_Vmax_shift_s,dt_state_at_DCAC_boundary_s,dc_remaining_time_to_own_Vmax_s,voltage_headroom_DC_at_DCAC_Vmax_Q_V,boundary_shift_class,has_cv_feedback_in_current_npz
0,Chen2020,0.856056,17.121118,3852.061493,770.260282,3081.801211,0.113419,large_boundary_shift,False
1,OKane2022,0.807622,16.152443,3497.993316,590.553554,2907.439762,0.106144,large_boundary_shift,False
2,ORegan2022,1.513671,30.273420,7081.007881,1631.792248,5449.215633,0.454153,very_large_boundary_shift,False



Cell 6 PASSED — Segment B voltage-boundary contribution audit complete


In [10]:
# Cell 7 — Day20A interim summary: Segment A/B pre-CV audit
#
# Purpose:
#   Close Day20A pre-CV / event-boundary audit.
#
# Inputs:
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv
#   data/day20_step3_segment_A_geometry_residual_summary.csv
#   data/day20_step4_segment_B_voltage_boundary_audit.csv
#
# Outputs:
#   data/day20_step5_interim_segment_AB_summary.csv
#   docs/day20A_preCV_event_boundary_audit.md
#
# Scope:
#   This closes the current existing-trajectory audit only.
#   It does NOT claim full-protocol CC+CV behavior because current NPZ
#   terminates at Vmax and contains no CV feedback tail.

import numpy as np
import pandas as pd
from pathlib import Path

EVENT_AUDIT = DATA / "day20_step2_v4_event_boundary_audit.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"
SEG_A = DATA / "day20_step3_segment_A_geometry_residual_summary.csv"
SEG_B = DATA / "day20_step4_segment_B_voltage_boundary_audit.csv"

for p in [EVENT_AUDIT, SEGMENTS, SEG_A, SEG_B]:
    assert p.exists(), f"Missing required Day20A file: {p}"

event_df = pd.read_csv(EVENT_AUDIT)
segments_df = pd.read_csv(SEGMENTS)
segA_df = pd.read_csv(SEG_A)
segB_df = pd.read_csv(SEG_B)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def pick_col(row, base, preferred="segB"):
    """
    Pick a possibly suffixed column after pandas merge.

    Preference:
      base_preferred -> base -> base_segA -> base_segB
    """
    candidates = [
        f"{base}_{preferred}",
        base,
        f"{base}_segA",
        f"{base}_segB",
    ]
    for c in candidates:
        if c in row.index:
            return row[c]
    raise KeyError(f"Cannot find column for base={base!r}; tried {candidates}")


def to_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return bool(x)


def safe_float(x):
    if pd.isna(x):
        return np.nan
    return float(x)


# ---------------------------------------------------------------------
# Build pair-level interim summary
# ---------------------------------------------------------------------

join_keys = ["param_set", "anchor_label", "phase_label"]

merged = segA_df.merge(
    segB_df,
    on=join_keys,
    how="inner",
    suffixes=("_segA", "_segB"),
    validate="one_to_one",
)

assert len(merged) == len(segA_df) == len(segB_df), (
    f"Expected one-to-one merge between Segment A and B summaries; "
    f"got segA={len(segA_df)}, segB={len(segB_df)}, merged={len(merged)}"
)

rows = []

for _, r in merged.iterrows():
    param_set = r["param_set"]

    # Segment A metrics
    segA_status = r["segment_A_status"]
    segA_resid_max = safe_float(r["dt_resid_max_abs_s"])
    segA_resid_p95 = safe_float(r["dt_resid_p95_abs_s"])
    segA_resid_median = safe_float(r["dt_resid_median_s"])
    segA_geom_median = safe_float(r["dt_geom_median_s"])
    segA_model_median = safe_float(r["dt_model_median_s"])
    resid_over_geom_median = safe_float(r["resid_over_geom_median"])

    # Segment B metrics
    q_shift_Ah = safe_float(pick_col(r, "Q_to_Vmax_shift_Ah", preferred="segB"))
    q_shift_pct = safe_float(pick_col(r, "Q_to_Vmax_shift_pct_nom", preferred="segB"))
    t_shift_s = safe_float(pick_col(r, "t_to_Vmax_shift_s", preferred="segB"))
    dt_state_boundary = safe_float(pick_col(r, "dt_state_at_DCAC_boundary_s", preferred="segB"))
    dc_remaining = safe_float(pick_col(r, "dc_remaining_time_to_own_Vmax_s", preferred="segB"))
    voltage_headroom = safe_float(pick_col(r, "voltage_headroom_DC_at_DCAC_Vmax_Q_V", preferred="segB"))
    boundary_class = pick_col(r, "boundary_shift_class", preferred="segB")

    # Contribution fractions for event timing shift
    if np.isfinite(t_shift_s) and abs(t_shift_s) > 1e-12:
        state_gain_fraction = dt_state_boundary / t_shift_s
        dc_remaining_fraction = dc_remaining / t_shift_s
    else:
        state_gain_fraction = np.nan
        dc_remaining_fraction = np.nan

    # Scope flags
    has_cv_feedback = to_bool(pick_col(r, "has_cv_feedback_in_current_npz", preferred="segB"))

    # Interpretive verdict
    if segA_status == "numerical_null" and q_shift_Ah > 0 and not has_cv_feedback:
        day20A_verdict = "segment_A_geometry_dominated_segment_B_boundary_shift_large_no_CV_tail"
    elif segA_status == "numerical_null" and q_shift_Ah > 0 and has_cv_feedback:
        day20A_verdict = "segment_A_geometry_dominated_boundary_shift_large_with_CV_tail_available"
    else:
        day20A_verdict = "requires_review"

    rows.append({
        "param_set": param_set,
        "anchor_label": r["anchor_label"],
        "phase_label": r["phase_label"],

        "protocol_DC": pick_col(r, "protocol_DC", preferred="segB"),
        "protocol_DCAC": pick_col(r, "protocol_DCAC", preferred="segB"),
        "Q_nom_Ah": safe_float(pick_col(r, "Q_nom_Ah", preferred="segB")),
        "f_anchor_Hz": safe_float(pick_col(r, "f_anchor_Hz", preferred="segB")),
        "T_period_s": safe_float(pick_col(r, "T_period_s", preferred="segB")),

        # Segment A
        "segment_A_status": segA_status,
        "segment_A_n_rows": int(r["n_rows_segment_A"]),
        "segment_A_Q_lo_Ah": safe_float(r["Q_lo_segment_A_Ah"]),
        "segment_A_Q_hi_Ah": safe_float(r["Q_hi_segment_A_Ah"]),
        "segment_A_dt_model_median_s": segA_model_median,
        "segment_A_dt_geom_median_s": segA_geom_median,
        "segment_A_dt_resid_median_s": segA_resid_median,
        "segment_A_dt_resid_p95_abs_s": segA_resid_p95,
        "segment_A_dt_resid_max_abs_s": segA_resid_max,
        "segment_A_resid_over_geom_median": resid_over_geom_median,

        # Segment B
        "Q_to_Vmax_DC_Ah": safe_float(pick_col(r, "Q_to_Vmax_DC_Ah", preferred="segB")),
        "Q_to_Vmax_DCAC_Ah": safe_float(pick_col(r, "Q_to_Vmax_DCAC_Ah", preferred="segB")),
        "Q_to_Vmax_shift_Ah": q_shift_Ah,
        "Q_to_Vmax_shift_pct_nom": q_shift_pct,
        "t_to_Vmax_DC_s": safe_float(pick_col(r, "t_to_Vmax_DC_s", preferred="segB")),
        "t_to_Vmax_DCAC_s": safe_float(pick_col(r, "t_to_Vmax_DCAC_s", preferred="segB")),
        "t_to_Vmax_shift_s": t_shift_s,
        "dt_state_at_DCAC_boundary_s": dt_state_boundary,
        "dc_remaining_time_to_own_Vmax_s": dc_remaining,
        "state_gain_fraction_of_event_shift": state_gain_fraction,
        "dc_remaining_fraction_of_event_shift": dc_remaining_fraction,
        "voltage_headroom_DC_at_DCAC_Vmax_Q_V": voltage_headroom,
        "boundary_shift_class": boundary_class,

        # Scope
        "has_cv_feedback_in_current_npz": has_cv_feedback,
        "current_npz_scope": "pre_CV_event_boundary_only",
        "day20A_verdict": day20A_verdict,
        "next_required_step": (
            "new full CC+CV PyBaMM protocol with AC-off at Vmax and normalized cutoff current"
            if not has_cv_feedback else
            "audit available CV feedback tail"
        ),
    })

summary_df = pd.DataFrame(rows)

out_summary = DATA / "day20_step5_interim_segment_AB_summary.csv"
summary_df.to_csv(out_summary, index=False)

print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
display(summary_df)

print("\nDay20A verdict counts:")
print(summary_df["day20A_verdict"].value_counts(dropna=False).to_string())

print("\nKey compact summary:")
display(
    summary_df[
        [
            "param_set",
            "segment_A_status",
            "segment_A_dt_resid_p95_abs_s",
            "segment_A_dt_resid_max_abs_s",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "state_gain_fraction_of_event_shift",
            "dc_remaining_fraction_of_event_shift",
            "voltage_headroom_DC_at_DCAC_Vmax_Q_V",
            "boundary_shift_class",
            "has_cv_feedback_in_current_npz",
            "day20A_verdict",
        ]
    ]
)

# ---------------------------------------------------------------------
# Write Markdown note
# ---------------------------------------------------------------------

DOCS.mkdir(exist_ok=True)
out_doc = DOCS / "day20A_preCV_event_boundary_audit.md"

lines = []
lines.append("# Day20A — PyBaMM pre-CV / voltage-boundary segmented audit")
lines.append("")
lines.append("Status: interim audit closed")
lines.append("")
lines.append("## Scope")
lines.append("")
lines.append("Day20A audits the existing Day18 v4 charge-first PyBaMM trajectory cache:")
lines.append("")
lines.append("```text")
lines.append("data/day18_step1_phase_audit_trajectories_v4_charge_first.npz")
lines.append("```")
lines.append("")
lines.append("The cache contains `t`, `V`, `I`, and `Q_net` for DC and DCAC arms, but it terminates at the maximum-voltage event. It does not contain an AC-off transition or a CV feedback tail.")
lines.append("")
lines.append("Therefore Day20A covers:")
lines.append("")
lines.append("- Segment A: AC-on prescribed-current shared-Q window")
lines.append("- Segment B: voltage-boundary event separation")
lines.append("")
lines.append("Day20A does not cover:")
lines.append("")
lines.append("- Segment C: AC-off transition")
lines.append("- Segment D: pure DC-CV feedback to cutoff")
lines.append("")
lines.append("## Segment A finding")
lines.append("")
lines.append("Segment A is geometry-dominated for all audited parameter sets.")
lines.append("")
lines.append("| param_set | median Δt_model [s] | median Δt_geom [s] | median Δt_resid [s] | p95 |Δt_resid| [s] | max |Δt_resid| [s] |")
lines.append("|---|---:|---:|---:|---:|---:|")

for _, r in summary_df.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{r['segment_A_dt_model_median_s']:.3f} | "
        f"{r['segment_A_dt_geom_median_s']:.3f} | "
        f"{r['segment_A_dt_resid_median_s']:.6f} | "
        f"{r['segment_A_dt_resid_p95_abs_s']:.6f} | "
        f"{r['segment_A_dt_resid_max_abs_s']:.6f} |"
    )

lines.append("")
lines.append("Interpretation:")
lines.append("")
lines.append("```text")
lines.append("Segment A raw Δt(Q) is explained by current geometry. Non-geometric residual is numerical-null.")
lines.append("```")
lines.append("")
lines.append("## Segment B finding")
lines.append("")
lines.append("Segment B shows large voltage-boundary separation. DCAC reaches the voltage boundary at substantially lower Q and earlier time than DC.")
lines.append("")
lines.append("| param_set | Q shift [Ah] | Q shift [% nominal] | time shift [s] | state gain at DCAC boundary [s] | DC remaining time [s] | DC voltage headroom [V] | class |")
lines.append("|---|---:|---:|---:|---:|---:|---:|---|")

for _, r in summary_df.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{r['Q_to_Vmax_shift_Ah']:.6f} | "
        f"{r['Q_to_Vmax_shift_pct_nom']:.3f} | "
        f"{r['t_to_Vmax_shift_s']:.3f} | "
        f"{r['dt_state_at_DCAC_boundary_s']:.3f} | "
        f"{r['dc_remaining_time_to_own_Vmax_s']:.3f} | "
        f"{r['voltage_headroom_DC_at_DCAC_Vmax_Q_V']:.6f} | "
        f"{r['boundary_shift_class']} |"
    )

lines.append("")
lines.append("The event timing shift decomposes as:")
lines.append("")
lines.append("```text")
lines.append("t_DC,Vmax − t_DCAC,Vmax")
lines.append("=")
lines.append("[t_DC(Q_DCAC,Vmax) − t_DCAC,Vmax]")
lines.append("+")
lines.append("[t_DC,Vmax − t_DC(Q_DCAC,Vmax)]")
lines.append("```")
lines.append("")
lines.append("The second term dominates: at the Q where DCAC reaches the voltage boundary, the DC arm is still below Vmax and still requires substantial additional charging time.")
lines.append("")
lines.append("## Cutoff normalization rule for future full-protocol simulations")
lines.append("")
lines.append("The MJ1 experimental cutoff current of 50 mA must not be transferred as an absolute current to all PyBaMM parameter sets.")
lines.append("")
lines.append("It must be normalized as:")
lines.append("")
lines.append("```text")
lines.append("C_cutoff = 0.05 / 3.4 ≈ 0.0147059 C")
lines.append("I_cutoff = C_cutoff · Q_nom")
lines.append("```")
lines.append("")
lines.append("## Day20A conclusion")
lines.append("")
lines.append("```text")
lines.append("The existing charge-first PyBaMM v4 trajectory cache supports a pre-CV audit only.")
lines.append("Segment A is current-geometry dominated with numerical-null residual.")
lines.append("Segment B shows large voltage-boundary separation.")
lines.append("The current cache does not contain AC-off transition or CV feedback.")
lines.append("A new full CC+CV PyBaMM protocol is required for Segment C/D.")
lines.append("```")
lines.append("")

out_doc.write_text("\n".join(lines), encoding="utf-8")

print(f"\nWrote: {out_doc}")

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert (summary_df["segment_A_status"] == "numerical_null").all(), (
    "Segment A is not numerical-null for all parameter sets."
)

assert (summary_df["Q_to_Vmax_shift_Ah"] > 0).all(), (
    "Expected positive Q_to_Vmax shift for all parameter sets."
)

assert (summary_df["t_to_Vmax_shift_s"] > 0).all(), (
    "Expected positive t_to_Vmax shift for all parameter sets."
)

assert not summary_df["has_cv_feedback_in_current_npz"].any(), (
    "Unexpected CV feedback in current NPZ."
)

print("\n" + "=" * 72)
print("Cell 7 PASSED — Day20A interim Segment A/B summary complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step5_interim_segment_AB_summary.csv (3 rows)


,param_set,anchor_label,phase_label,protocol_DC,protocol_DCAC,Q_nom_Ah,f_anchor_Hz,T_period_s,segment_A_status,segment_A_n_rows,...,dt_state_at_DCAC_boundary_s,dc_remaining_time_to_own_Vmax_s,state_gain_fraction_of_event_shift,dc_remaining_fraction_of_event_shift,voltage_headroom_DC_at_DCAC_Vmax_Q_V,boundary_shift_class,has_cv_feedback_in_current_npz,current_npz_scope,day20A_verdict,next_required_step
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000408,2453.513699,numerical_null,62,...,770.260282,3081.801211,0.199961,0.800039,0.113419,large_boundary_shift,False,pre_CV_event_boundary_only,segment_A_geometry_dominated_segment_B_boundar...,new full CC+CV PyBaMM protocol with AC-off at ...
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000397,2516.435063,numerical_null,62,...,590.553554,2907.439762,0.168826,0.831174,0.106144,large_boundary_shift,False,pre_CV_event_boundary_only,segment_A_geometry_dominated_segment_B_boundar...,new full CC+CV PyBaMM protocol with AC-off at ...
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000333,2998.832183,numerical_null,59,...,1631.792248,5449.215633,0.230446,0.769554,0.454153,very_large_boundary_shift,False,pre_CV_event_boundary_only,segment_A_geometry_dominated_segment_B_boundar...,new full CC+CV PyBaMM protocol with AC-off at ...



Day20A verdict counts:
day20A_verdict
segment_A_geometry_dominated_segment_B_boundary_shift_large_no_CV_tail    3

Key compact summary:


,param_set,segment_A_status,segment_A_dt_resid_p95_abs_s,segment_A_dt_resid_max_abs_s,Q_to_Vmax_shift_Ah,Q_to_Vmax_shift_pct_nom,t_to_Vmax_shift_s,state_gain_fraction_of_event_shift,dc_remaining_fraction_of_event_shift,voltage_headroom_DC_at_DCAC_Vmax_Q_V,boundary_shift_class,has_cv_feedback_in_current_npz,day20A_verdict
0,Chen2020,numerical_null,0.007722,0.014259,0.856056,17.121118,3852.061493,0.199961,0.800039,0.113419,large_boundary_shift,False,segment_A_geometry_dominated_segment_B_boundar...
1,OKane2022,numerical_null,0.003514,0.005144,0.807622,16.152443,3497.993316,0.168826,0.831174,0.106144,large_boundary_shift,False,segment_A_geometry_dominated_segment_B_boundar...
2,ORegan2022,numerical_null,0.005973,0.014275,1.513671,30.273420,7081.007881,0.230446,0.769554,0.454153,very_large_boundary_shift,False,segment_A_geometry_dominated_segment_B_boundar...



Wrote: /Users/louislu/pybamm-dcac-superimposed/docs/day20A_preCV_event_boundary_audit.md

Cell 7 PASSED — Day20A interim Segment A/B summary complete


## Day20B — Full CC+CV PyBaMM protocol design

Day20A closed the available pre-CV audit:

- Segment A is current-geometry dominated with numerical-null residual.
- Segment B shows large voltage-boundary separation.
- The existing Day18 v4 trajectory cache terminates at the voltage-limit event.
- It does not contain AC-off transition or CV feedback.

Day20B therefore defines a new PyBaMM full-protocol simulation that matches the MJ1 experimental control logic at the protocol level:

1. DC reference:

   - Constant-current charge until $V_{\max}$
   - Then CV at $V_{\max}$
   - Terminate when the current magnitude reaches the normalized cutoff

2. DCAC arm:

   - Charge-first DC+AC during CC
   - Stop AC at first $V_{\max}$
   - Then pure DC-CV at $V_{\max}$
   - Terminate when the current magnitude reaches the normalized cutoff

The MJ1 experimental cutoff current of 50 mA is not transferred as an absolute current. It is normalized as

$$
C_{\mathrm{cutoff}}
=
\frac{0.05}{3.4}
\approx
0.0147059C
$$

and applied to each PyBaMM parameter set as

$$
I_{\mathrm{cutoff}}
=
C_{\mathrm{cutoff}}
\cdot
Q_{\mathrm{nom}}
$$

This cell only builds the protocol design table. It does not run the full simulation yet.

In [12]:
# Cell 8 — Day20B full CC+CV protocol design table
#
# Purpose:
#   Build the protocol design table for the next full CC+CV PyBaMM simulation.
#
# This cell does NOT run PyBaMM yet.
#
# It defines, for each parameter set:
#   - DC reference full protocol
#   - DCAC full protocol with charge-first AC during CC and AC-off at Vmax
#   - pure DC-CV after Vmax
#   - normalized cutoff current:
#         I_cutoff = (0.05 / 3.4) * Q_nom
#
# Inputs:
#   data/day20_step5_interim_segment_AB_summary.csv
#   data/day20_step4_segment_B_voltage_boundary_audit.csv
#
# Output:
#   data/day20B_step0_full_protocol_design.csv

import re
import numpy as np
import pandas as pd

SEG_AB = DATA / "day20_step5_interim_segment_AB_summary.csv"
SEG_B = DATA / "day20_step4_segment_B_voltage_boundary_audit.csv"

assert SEG_AB.exists(), f"Missing: {SEG_AB}"
assert SEG_B.exists(), f"Missing: {SEG_B}"

seg_ab = pd.read_csv(SEG_AB)
seg_b = pd.read_csv(SEG_B)

MJ1_I_NOM_A = 3.4
MJ1_I_CUTOFF_A = 0.050
CUTOFF_C_RATE = MJ1_I_CUTOFF_A / MJ1_I_NOM_A

PHASE_FULL_PROTOCOL = "charge_first"
FULL_PROTOCOL_SCOPE = "CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_until_normalized_cutoff"

print("=" * 72)
print("Day20B — full CC+CV protocol design")
print("=" * 72)
print(f"MJ1 cutoff reference : {MJ1_I_CUTOFF_A:.6f} A / {MJ1_I_NOM_A:.6f} A")
print(f"Normalized cutoff    : {CUTOFF_C_RATE:.9f} C")
print()


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def pick_col(row, base, preferred="boundary"):
    """
    Pick a possibly suffixed column after pandas merge.

    Preference:
      base_preferred -> base -> base_summary -> base_boundary
    """
    candidates = [
        f"{base}_{preferred}",
        base,
        f"{base}_summary",
        f"{base}_boundary",
    ]
    for c in candidates:
        if c in row.index:
            return row[c]
    raise KeyError(f"Cannot find column for base={base!r}; tried {candidates}")


def safe_float(x):
    if pd.isna(x):
        return np.nan
    return float(x)


def parse_dcac_label(label):
    """
    Parse labels like:
        DCAC_DC0p2C_AC0p5C_10tau
    """
    s = str(label)
    if s.upper().startswith("DCAC_"):
        s = s[5:]

    pat = re.compile(
        r"DC(?P<dc>[0-9p.]+)C[_+]?AC(?P<ac>[0-9p.]+)C[_-]?(?P<tau>[0-9p.]+)tau",
        re.IGNORECASE,
    )
    m = pat.search(s)
    if not m:
        raise ValueError(f"Cannot parse DCAC label: {label!r}")

    def pfloat(x):
        return float(str(x).replace("p", "."))

    return {
        "DC_C": pfloat(m.group("dc")),
        "AC_C": pfloat(m.group("ac")),
        "n_tau": pfloat(m.group("tau")),
    }


def parse_dc_label(label):
    """
    Parse labels like:
        DC_0p2C
    """
    s = str(label)
    pat = re.compile(r"DC[_-]?(?P<dc>[0-9p.]+)C", re.IGNORECASE)
    m = pat.search(s)
    if not m:
        raise ValueError(f"Cannot parse DC label: {label!r}")

    return float(m.group("dc").replace("p", "."))


# ---------------------------------------------------------------------
# Join design sources
# ---------------------------------------------------------------------

join_keys = ["param_set", "anchor_label", "phase_label", "protocol_DC", "protocol_DCAC"]

required_ab = set(join_keys + ["Q_nom_Ah", "f_anchor_Hz", "T_period_s"])
missing_ab = required_ab - set(seg_ab.columns)
assert not missing_ab, f"Segment AB summary missing columns: {missing_ab}"

required_b = set(join_keys + [
    "Vmax_cutoff_DC",
    "Vmax_cutoff_DCAC",
    "Q_to_Vmax_DC_Ah",
    "Q_to_Vmax_DCAC_Ah",
    "t_to_Vmax_DC_s",
    "t_to_Vmax_DCAC_s",
])
missing_b = required_b - set(seg_b.columns)
assert not missing_b, f"Segment B audit missing columns: {missing_b}"

merged = seg_ab.merge(
    seg_b,
    on=join_keys,
    how="inner",
    suffixes=("_summary", "_boundary"),
    validate="one_to_one",
)

assert len(merged) == len(seg_ab) == len(seg_b), (
    f"Expected one-to-one merge; seg_ab={len(seg_ab)}, seg_b={len(seg_b)}, merged={len(merged)}"
)

print("Merged design source rows:", len(merged))
print("Merged columns include:", [c for c in merged.columns if "Q_to_Vmax" in c or "Vmax" in c])


# ---------------------------------------------------------------------
# Build design rows
# ---------------------------------------------------------------------

rows = []

for _, r in merged.iterrows():
    param_set = r["param_set"]
    anchor_label = r["anchor_label"]
    phase_label = r["phase_label"]
    protocol_DC = r["protocol_DC"]
    protocol_DCAC = r["protocol_DCAC"]

    Q_nom_Ah = safe_float(pick_col(r, "Q_nom_Ah", preferred="summary"))
    f_anchor_Hz = safe_float(pick_col(r, "f_anchor_Hz", preferred="summary"))
    T_period_s = safe_float(pick_col(r, "T_period_s", preferred="summary"))

    Vmax_cutoff = safe_float(pick_col(r, "Vmax_cutoff_DCAC", preferred="boundary"))
    Vmax_cutoff_DC = safe_float(pick_col(r, "Vmax_cutoff_DC", preferred="boundary"))

    assert abs(Vmax_cutoff - Vmax_cutoff_DC) < 1e-12, (
        f"{param_set}: DC and DCAC Vmax cutoffs differ: {Vmax_cutoff_DC} vs {Vmax_cutoff}"
    )

    dc_C_from_dc_label = parse_dc_label(protocol_DC)
    dcac_parsed = parse_dcac_label(protocol_DCAC)

    DC_C = dcac_parsed["DC_C"]
    AC_C = dcac_parsed["AC_C"]
    n_tau = dcac_parsed["n_tau"]

    assert abs(DC_C - dc_C_from_dc_label) < 1e-12, (
        f"{param_set}: DC_C mismatch between DC protocol and DCAC label"
    )

    I_DC_A = DC_C * Q_nom_Ah
    I_AC_A = AC_C * Q_nom_Ah
    I_cutoff_A = CUTOFF_C_RATE * Q_nom_Ah

    I_peak_charge_A = I_DC_A + I_AC_A
    I_peak_discharge_A = max(I_AC_A - I_DC_A, 0.0)

    Q_to_Vmax_DC_Ah = safe_float(pick_col(r, "Q_to_Vmax_DC_Ah", preferred="boundary"))
    Q_to_Vmax_DCAC_Ah = safe_float(pick_col(r, "Q_to_Vmax_DCAC_Ah", preferred="boundary"))
    t_to_Vmax_DC_s = safe_float(pick_col(r, "t_to_Vmax_DC_s", preferred="boundary"))
    t_to_Vmax_DCAC_s = safe_float(pick_col(r, "t_to_Vmax_DCAC_s", preferred="boundary"))

    # -----------------------------------------------------------------
    # DC reference arm
    # -----------------------------------------------------------------
    rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,

        "arm": "DC_reference",
        "protocol_label": f"{anchor_label}__DC_full_CC_CV",
        "base_protocol_from": protocol_DC,

        "Q_nom_Ah": Q_nom_Ah,
        "Vmax_cutoff_V": Vmax_cutoff,
        "C_cutoff": CUTOFF_C_RATE,
        "I_cutoff_A": I_cutoff_A,

        "DC_C": DC_C,
        "AC_C": 0.0,
        "n_tau": np.nan,
        "f_Hz": np.nan,
        "T_period_s": np.nan,

        "I_DC_A": I_DC_A,
        "I_AC_A": 0.0,
        "I_peak_charge_A": I_DC_A,
        "I_peak_discharge_A": 0.0,

        "cc_current_formula_pybamm": "I_py = -I_DC_A",
        "cv_stage_definition": "hold V=Vmax_cutoff_V until |I_py| <= I_cutoff_A",
        "termination_condition": "CV current magnitude <= normalized cutoff",
        "full_protocol_scope": FULL_PROTOCOL_SCOPE,

        "source_Q_to_Vmax_Ah": Q_to_Vmax_DC_Ah,
        "source_t_to_Vmax_s": t_to_Vmax_DC_s,
        "source_event_type": "DC reaches Vmax then CV",
    })

    # -----------------------------------------------------------------
    # DCAC full protocol arm
    # -----------------------------------------------------------------
    rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,

        "arm": "DCAC_full_protocol",
        "protocol_label": f"{anchor_label}__DCAC_full_CC_ACoff_CV",
        "base_protocol_from": protocol_DCAC,

        "Q_nom_Ah": Q_nom_Ah,
        "Vmax_cutoff_V": Vmax_cutoff,
        "C_cutoff": CUTOFF_C_RATE,
        "I_cutoff_A": I_cutoff_A,

        "DC_C": DC_C,
        "AC_C": AC_C,
        "n_tau": n_tau,
        "f_Hz": f_anchor_Hz,
        "T_period_s": T_period_s,

        "I_DC_A": I_DC_A,
        "I_AC_A": I_AC_A,
        "I_peak_charge_A": I_peak_charge_A,
        "I_peak_discharge_A": I_peak_discharge_A,

        "cc_current_formula_pybamm": "I_py = -I_DC_A - I_AC_A*sin(2*pi*f_Hz*t)",
        "cv_stage_definition": "AC off; hold V=Vmax_cutoff_V until |I_py| <= I_cutoff_A",
        "termination_condition": "CV current magnitude <= normalized cutoff",
        "full_protocol_scope": FULL_PROTOCOL_SCOPE,

        "source_Q_to_Vmax_Ah": Q_to_Vmax_DCAC_Ah,
        "source_t_to_Vmax_s": t_to_Vmax_DCAC_s,
        "source_event_type": "DCAC reaches Vmax; AC off; then pure DC-CV",
    })

design_df = pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Derived sanity columns
# ---------------------------------------------------------------------

design_df["I_cutoff_mA"] = design_df["I_cutoff_A"] * 1000.0
design_df["I_cutoff_over_Qnom_C"] = design_df["I_cutoff_A"] / design_df["Q_nom_Ah"]
design_df["peak_charge_C"] = design_df["I_peak_charge_A"] / design_df["Q_nom_Ah"]
design_df["peak_discharge_C"] = design_df["I_peak_discharge_A"] / design_df["Q_nom_Ah"]


# ---------------------------------------------------------------------
# Assertions
# ---------------------------------------------------------------------

assert (design_df["Q_nom_Ah"] > 0).all()
assert (design_df["Vmax_cutoff_V"] > 0).all()
assert np.allclose(design_df["I_cutoff_over_Qnom_C"], CUTOFF_C_RATE)
assert (design_df["I_cutoff_A"] > 0).all()

dc_rows = design_df[design_df["arm"].eq("DC_reference")]
dcac_rows = design_df[design_df["arm"].eq("DCAC_full_protocol")]

assert len(dc_rows) == len(dcac_rows) == len(merged)
assert (dcac_rows["phase_label"] == PHASE_FULL_PROTOCOL).all()
assert (dcac_rows["f_Hz"] > 0).all()
assert (dcac_rows["I_AC_A"] > 0).all()
assert (dcac_rows["AC_C"] > 0).all()

# Peak current note:
# These designs preserve the historical Day18 v4 AC0p5C protocol:
# DC=0.2C, AC=0.5C, peak charge=0.7C, peak discharge=0.3C.
assert (dcac_rows["peak_charge_C"] <= 1.0 + 1e-12).all(), (
    "Peak charge current exceeds 1C constraint."
)


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out = DATA / "day20B_step0_full_protocol_design.csv"
design_df.to_csv(out, index=False)

print(f"Wrote: {out} ({len(design_df)} rows)")
display(design_df)

print("\nCutoff current by parameter set:")
display(
    design_df[
        [
            "param_set", "arm", "Q_nom_Ah", "C_cutoff",
            "I_cutoff_A", "I_cutoff_mA",
            "DC_C", "AC_C", "peak_charge_C", "peak_discharge_C",
            "Vmax_cutoff_V", "f_Hz", "T_period_s",
        ]
    ]
)

print("\n" + "=" * 72)
print("Cell 8 PASSED — Day20B full-protocol design table created")
print("=" * 72)

Day20B — full CC+CV protocol design
MJ1 cutoff reference : 0.050000 A / 3.400000 A
Normalized cutoff    : 0.014705882 C

Merged design source rows: 3
Merged columns include: ['Q_to_Vmax_DC_Ah_summary', 'Q_to_Vmax_DCAC_Ah_summary', 'Q_to_Vmax_shift_Ah_summary', 'Q_to_Vmax_shift_pct_nom_summary', 't_to_Vmax_DC_s_summary', 't_to_Vmax_DCAC_s_summary', 't_to_Vmax_shift_s_summary', 'dc_remaining_time_to_own_Vmax_s_summary', 'voltage_headroom_DC_at_DCAC_Vmax_Q_V_summary', 'Vmax_cutoff_DC', 'Vmax_cutoff_DCAC', 'Q_to_Vmax_DC_Ah_boundary', 'Q_to_Vmax_DCAC_Ah_boundary', 'Q_to_Vmax_shift_Ah_boundary', 'Q_to_Vmax_shift_pct_nom_boundary', 't_to_Vmax_DC_s_boundary', 't_to_Vmax_DCAC_s_boundary', 't_to_Vmax_shift_s_boundary', 'DC_at_Q_DCAC_Vmax_found', 't_DC_at_Q_DCAC_Vmax_s', 'V_DC_at_Q_DCAC_Vmax_V', 'I_DC_at_Q_DCAC_Vmax_A', 'DCAC_at_Q_DCAC_Vmax_found', 't_DCAC_at_Q_DCAC_Vmax_s_reconstructed', 'V_DCAC_at_Q_DCAC_Vmax_V_reconstructed', 'I_DCAC_at_Q_DCAC_Vmax_A_reconstructed', 'voltage_headroom_DC_at_DCA

,param_set,anchor_label,phase_label,arm,protocol_label,base_protocol_from,Q_nom_Ah,Vmax_cutoff_V,C_cutoff,I_cutoff_A,...,cv_stage_definition,termination_condition,full_protocol_scope,source_Q_to_Vmax_Ah,source_t_to_Vmax_s,source_event_type,I_cutoff_mA,I_cutoff_over_Qnom_C,peak_charge_C,peak_discharge_C
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_reference,Route2_AC0p5C_10tau_native_charge_first__DC_fu...,DC_0p2C,5.0,4.2,0.014706,0.073529,...,hold V=Vmax_cutoff_V until |I_py| <= I_cutoff_A,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,4.625000,16650.000000,DC reaches Vmax then CV,73.529412,0.014706,0.2,0.0
1,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DCAC_full_protocol,Route2_AC0p5C_10tau_native_charge_first__DCAC_...,DCAC_DC0p2C_AC0p5C_10tau,5.0,4.2,0.014706,0.073529,...,AC off; hold V=Vmax_cutoff_V until |I_py| <= I...,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,3.768944,12797.938507,DCAC reaches Vmax; AC off; then pure DC-CV,73.529412,0.014706,0.7,0.3
2,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_reference,Route2_AC0p5C_10tau_native_charge_first__DC_fu...,DC_0p2C,5.0,4.2,0.014706,0.073529,...,hold V=Vmax_cutoff_V until |I_py| <= I_cutoff_A,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,4.594444,16540.000000,DC reaches Vmax then CV,73.529412,0.014706,0.2,0.0
3,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DCAC_full_protocol,Route2_AC0p5C_10tau_native_charge_first__DCAC_...,DCAC_DC0p2C_AC0p5C_10tau,5.0,4.2,0.014706,0.073529,...,AC off; hold V=Vmax_cutoff_V until |I_py| <= I...,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,3.786822,13042.006684,DCAC reaches Vmax; AC off; then pure DC-CV,73.529412,0.014706,0.7,0.3
4,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_reference,Route2_AC0p5C_10tau_native_charge_first__DC_fu...,DC_0p2C,5.0,4.4,0.014706,0.073529,...,hold V=Vmax_cutoff_V until |I_py| <= I_cutoff_A,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,4.724128,17006.861901,DC reaches Vmax then CV,73.529412,0.014706,0.2,0.0
5,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DCAC_full_protocol,Route2_AC0p5C_10tau_native_charge_first__DCAC_...,DCAC_DC0p2C_AC0p5C_10tau,5.0,4.4,0.014706,0.073529,...,AC off; hold V=Vmax_cutoff_V until |I_py| <= I...,CV current magnitude <= normalized cutoff,CC_DCAC_until_Vmax_then_AC_off_pure_DC_CV_unti...,3.210457,9925.854020,DCAC reaches Vmax; AC off; then pure DC-CV,73.529412,0.014706,0.7,0.3



Cutoff current by parameter set:


,param_set,arm,Q_nom_Ah,C_cutoff,I_cutoff_A,I_cutoff_mA,DC_C,AC_C,peak_charge_C,peak_discharge_C,Vmax_cutoff_V,f_Hz,T_period_s
0,Chen2020,DC_reference,5.0,0.014706,0.073529,73.529412,0.2,0.0,0.2,0.0,4.2,NaN,NaN
1,Chen2020,DCAC_full_protocol,5.0,0.014706,0.073529,73.529412,0.2,0.5,0.7,0.3,4.2,0.000408,2453.513699
2,OKane2022,DC_reference,5.0,0.014706,0.073529,73.529412,0.2,0.0,0.2,0.0,4.2,NaN,NaN
3,OKane2022,DCAC_full_protocol,5.0,0.014706,0.073529,73.529412,0.2,0.5,0.7,0.3,4.2,0.000397,2516.435063
4,ORegan2022,DC_reference,5.0,0.014706,0.073529,73.529412,0.2,0.0,0.2,0.0,4.4,NaN,NaN
5,ORegan2022,DCAC_full_protocol,5.0,0.014706,0.073529,73.529412,0.2,0.5,0.7,0.3,4.4,0.000333,2998.832183



Cell 8 PASSED — Day20B full-protocol design table created


In [16]:
# Cell 9 — Day20B Chen2020 full CC+CV smoke simulation
#
# Purpose:
#   Run the first full CC+CV PyBaMM smoke simulation for Chen2020 only.
#
# Protocol:
#   DC reference:
#       CC: I_py = -I_DC_A until Vmax
#       CV: hold Vmax until |I_py| <= I_cutoff_A
#
#   DCAC:
#       CC: I_py = -I_DC_A - I_AC_A*sin(2*pi*f_Hz*t) until Vmax
#       CV: AC off; hold Vmax until |I_py| <= I_cutoff_A
#
# Cutoff:
#   I_cutoff_A = (0.05 / 3.4) * Q_nom
#
# Outputs:
#   data/day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv
#
# Scope:
#   Smoke test only. Do not run all parameter sets yet.

import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid

try:
    import pybamm
except Exception as e:
    raise RuntimeError(f"PyBaMM import failed: {e}")

DESIGN = DATA / "day20B_step0_full_protocol_design.csv"
assert DESIGN.exists(), f"Missing design table: {DESIGN}"

design = pd.read_csv(DESIGN)

PARAM_SET = "Chen2020"
INITIAL_SOC = 0.05

OUTPUT_PERIOD_S = 10.0
MAX_CC_DURATION_S = 60000.0
MAX_CV_DURATION_S = 50000.0

df_ch = design[design["param_set"].eq(PARAM_SET)].copy()
assert len(df_ch) == 2, f"Expected two Chen2020 design rows, got {len(df_ch)}"

dc_row = df_ch[df_ch["arm"].eq("DC_reference")].iloc[0].to_dict()
dcac_row = df_ch[df_ch["arm"].eq("DCAC_full_protocol")].iloc[0].to_dict()

print("=" * 72)
print("Cell 9 — Chen2020 full CC+CV smoke simulation")
print("=" * 72)
print(f"Parameter set     : {PARAM_SET}")
print(f"Initial SOC       : {INITIAL_SOC}")
print(f"Vmax              : {dc_row['Vmax_cutoff_V']} V")
print(f"I_cutoff          : {dc_row['I_cutoff_A']:.9f} A ({dc_row['I_cutoff_mA']:.3f} mA)")
print(f"DC current        : {dc_row['I_DC_A']:.6f} A")
print(f"DCAC I_DC         : {dcac_row['I_DC_A']:.6f} A")
print(f"DCAC I_AC         : {dcac_row['I_AC_A']:.6f} A")
print(f"DCAC f            : {dcac_row['f_Hz']:.9g} Hz")
print(f"DCAC T            : {dcac_row['T_period_s']:.3f} s")
print()


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def make_dfn_simulation(param_set, experiment):
    model = pybamm.lithium_ion.DFN()
    pv = pybamm.ParameterValues(param_set)

    solver = pybamm.CasadiSolver(
        mode="safe",
        rtol=1e-6,
        atol=1e-8,
        dt_max=120.0,
    )

    return pybamm.Simulation(
        model,
        parameter_values=pv,
        experiment=experiment,
        solver=solver,
    )


def build_dc_experiment(row, cv_termination):
    I_DC_A = float(row["I_DC_A"])
    Vmax = float(row["Vmax_cutoff_V"])

    steps = [
        pybamm.step.current(
            -I_DC_A,
            duration=f"{MAX_CC_DURATION_S} seconds",
            termination=f"{Vmax} V",
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=cv_termination,
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def build_dcac_experiment(row, cv_termination):
    I_DC_A = float(row["I_DC_A"])
    I_AC_A = float(row["I_AC_A"])
    f_Hz = float(row["f_Hz"])
    Vmax = float(row["Vmax_cutoff_V"])

    def dcac_current(t):
        # PyBaMM sign: I > 0 discharge, I < 0 charge.
        # charge-first convention:
        #   I_py = -I_DC - I_AC sin(2πft)
        return -I_DC_A - I_AC_A * pybamm.sin(2 * np.pi * f_Hz * t)

    steps = [
        pybamm.step.current(
            dcac_current,
            duration=f"{MAX_CC_DURATION_S} seconds",
            termination=f"{Vmax} V",
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=cv_termination,
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def solve_with_cutoff_termination(arm, row):
    """
    Try current cutoff termination variants. PyBaMM versions differ in how they
    parse current termination strings for voltage steps.
    """
    I_cutoff_A = float(row["I_cutoff_A"])

    termination_candidates = [
        f"{I_cutoff_A} A",
        f"{I_cutoff_A * 1000.0} mA",
    ]

    last_err = None

    for cv_term in termination_candidates:
        print(f"[{arm}] trying CV termination: {cv_term}")

        try:
            if arm == "DC_reference":
                exp = build_dc_experiment(row, cv_term)
            elif arm == "DCAC_full_protocol":
                exp = build_dcac_experiment(row, cv_term)
            else:
                raise ValueError(f"Unknown arm: {arm}")

            sim = make_dfn_simulation(PARAM_SET, exp)

            try:
                sol = sim.solve(initial_soc=INITIAL_SOC)
            except TypeError:
                # Older PyBaMM fallback. Should normally not be used.
                sol = sim.solve()

            print(f"[{arm}] solve succeeded with termination {cv_term}")
            return sol, cv_term

        except Exception as e:
            last_err = e
            print(f"[{arm}] failed with termination {cv_term}: {e}")

    raise RuntimeError(f"{arm}: all cutoff termination variants failed. Last error: {last_err}")


def extract_solution(sol, arm, row, cv_termination_used):
    t = np.asarray(sol["Time [s]"].entries, dtype=float)
    V = np.asarray(sol["Terminal voltage [V]"].entries, dtype=float)
    I = np.asarray(sol["Current [A]"].entries, dtype=float)

    if not (len(t) == len(V) == len(I)):
        raise ValueError(f"{arm}: length mismatch t/V/I")

    # strict-net charge in Ah
    Q_Ah = -cumulative_trapezoid(I, t, initial=0.0) / 3600.0

    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])

    # First Vmax event by sampled trajectory
    crossed = V >= Vmax - 1e-6
    if crossed.any():
        idx_vmax = int(np.argmax(crossed))
        t_vmax = float(t[idx_vmax])
        Q_vmax = float(Q_Ah[idx_vmax])
    else:
        idx_vmax = np.nan
        t_vmax = np.nan
        Q_vmax = np.nan

    stage = np.where(t <= t_vmax + 1e-9, "CC_until_Vmax", "CV_after_Vmax")

    # CV diagnostics
    cv_mask = stage == "CV_after_Vmax"
    n_cv = int(cv_mask.sum())

    if n_cv > 0:
        I_cv = I[cv_mask]
        V_cv = V[cv_mask]
        I_cv_abs_initial = float(abs(I_cv[0]))
        I_cv_abs_final = float(abs(I_cv[-1]))
        I_cv_abs_range = float(np.nanmax(np.abs(I_cv)) - np.nanmin(np.abs(I_cv)))
        V_cv_max = float(np.nanmax(V_cv))
        V_cv_min = float(np.nanmin(V_cv))
    else:
        I_cv_abs_initial = np.nan
        I_cv_abs_final = np.nan
        I_cv_abs_range = np.nan
        V_cv_max = np.nan
        V_cv_min = np.nan

    traj = pd.DataFrame({
        "arm": arm,
        "param_set": PARAM_SET,
        "t_s": t,
        "V_V": V,
        "I_A": I,
        "Q_Ah": Q_Ah,
        "stage": stage,
        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "cv_termination_used": cv_termination_used,
    })

    summary = {
        "arm": arm,
        "param_set": PARAM_SET,
        "cv_termination_used": cv_termination_used,
        "n_samples": len(t),

        "t_end_s": float(t[-1]),
        "Q_end_Ah": float(Q_Ah[-1]),
        "V_end_V": float(V[-1]),
        "I_end_A": float(I[-1]),
        "I_end_abs_A": float(abs(I[-1])),

        "V_min_V": float(np.nanmin(V)),
        "V_max_V": float(np.nanmax(V)),
        "I_min_A": float(np.nanmin(I)),
        "I_max_A": float(np.nanmax(I)),

        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "I_cutoff_mA": I_cutoff_A * 1000.0,

        "idx_vmax_sample": idx_vmax,
        "t_to_Vmax_sample_s": t_vmax,
        "Q_to_Vmax_sample_Ah": Q_vmax,

        "n_cv_samples": n_cv,
        "has_cv_tail": n_cv > 5,
        "I_cv_abs_initial_A": I_cv_abs_initial,
        "I_cv_abs_final_A": I_cv_abs_final,
        "I_cv_abs_range_A": I_cv_abs_range,
        "V_cv_min_V": V_cv_min,
        "V_cv_max_V": V_cv_max,

        "cutoff_reached_by_final_current": bool(abs(I[-1]) <= I_cutoff_A * 1.05),
        "Vmax_violation_max_over_V": float(np.nanmax(V) - Vmax),
    }

    return traj, summary


# ---------------------------------------------------------------------
# Run simulations
# ---------------------------------------------------------------------

solutions = {}
traj_list = []
summary_list = []

for arm, row in [
    ("DC_reference", dc_row),
    ("DCAC_full_protocol", dcac_row),
]:
    sol, term_used = solve_with_cutoff_termination(arm, row)
    solutions[arm] = sol

    traj, summary = extract_solution(sol, arm, row, term_used)
    traj_list.append(traj)
    summary_list.append(summary)

traj_df = pd.concat(traj_list, ignore_index=True)
summary_df = pd.DataFrame(summary_list)

# ---------------------------------------------------------------------
# Pair-level comparison
# ---------------------------------------------------------------------

dc_sum = summary_df[summary_df["arm"].eq("DC_reference")].iloc[0]
dcac_sum = summary_df[summary_df["arm"].eq("DCAC_full_protocol")].iloc[0]

pair_summary = {
    "param_set": PARAM_SET,
    "initial_soc": INITIAL_SOC,

    "t_end_DC_s": dc_sum["t_end_s"],
    "t_end_DCAC_s": dcac_sum["t_end_s"],
    "dt_total_s": dc_sum["t_end_s"] - dcac_sum["t_end_s"],
    "dt_total_min": (dc_sum["t_end_s"] - dcac_sum["t_end_s"]) / 60.0,

    "Q_end_DC_Ah": dc_sum["Q_end_Ah"],
    "Q_end_DCAC_Ah": dcac_sum["Q_end_Ah"],
    "Q_end_diff_Ah": dc_sum["Q_end_Ah"] - dcac_sum["Q_end_Ah"],

    "t_to_Vmax_DC_s": dc_sum["t_to_Vmax_sample_s"],
    "t_to_Vmax_DCAC_s": dcac_sum["t_to_Vmax_sample_s"],
    "dt_to_Vmax_s": dc_sum["t_to_Vmax_sample_s"] - dcac_sum["t_to_Vmax_sample_s"],

    "Q_to_Vmax_DC_Ah": dc_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_DCAC_Ah": dcac_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_shift_Ah": dc_sum["Q_to_Vmax_sample_Ah"] - dcac_sum["Q_to_Vmax_sample_Ah"],

    "DC_cutoff_reached": dc_sum["cutoff_reached_by_final_current"],
    "DCAC_cutoff_reached": dcac_sum["cutoff_reached_by_final_current"],

    "DC_has_cv_tail": dc_sum["has_cv_tail"],
    "DCAC_has_cv_tail": dcac_sum["has_cv_tail"],

    "DC_Vmax_violation_max_over_V": dc_sum["Vmax_violation_max_over_V"],
    "DCAC_Vmax_violation_max_over_V": dcac_sum["Vmax_violation_max_over_V"],
}

pair_df = pd.DataFrame([pair_summary])

# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_traj = DATA / "day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv"
out_summary = DATA / "day20B_step1_Chen2020_full_protocol_smoke_summary.csv"
out_pair = DATA / "day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv"

traj_df.to_csv(out_traj, index=False)
summary_df.to_csv(out_summary, index=False)
pair_df.to_csv(out_pair, index=False)

print(f"\nWrote: {out_traj} ({len(traj_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
print(f"Wrote: {out_pair} ({len(pair_df)} rows)")

display(summary_df)
display(pair_df)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert summary_df["has_cv_tail"].all(), (
    "Expected both arms to contain CV tail. Full-protocol simulation did not enter CV correctly."
)

assert summary_df["cutoff_reached_by_final_current"].all(), (
    "Expected both arms to terminate near normalized cutoff current."
)

assert (summary_df["Vmax_violation_max_over_V"] < 0.02).all(), (
    "Voltage exceeded Vmax by more than 20 mV."
)

assert pair_df["DC_has_cv_tail"].iloc[0] and pair_df["DCAC_has_cv_tail"].iloc[0]

print("\n" + "=" * 72)
print("Cell 9 PASSED — Chen2020 full CC+CV smoke simulation complete")
print("=" * 72)

Cell 9 — Chen2020 full CC+CV smoke simulation
Parameter set     : Chen2020
Initial SOC       : 0.05
Vmax              : 4.2 V
I_cutoff          : 0.073529412 A (73.529 mA)
DC current        : 1.000000 A
DCAC I_DC         : 1.000000 A
DCAC I_AC         : 2.500000 A
DCAC f            : 0.000407578731 Hz
DCAC T            : 2453.514 s

[DC_reference] trying CV termination: 0.0735294117647058 A
[DC_reference] solve succeeded with termination 0.0735294117647058 A
[DCAC_full_protocol] trying CV termination: 0.0735294117647058 A
[DCAC_full_protocol] failed with termination 0.0735294117647058 A: module 'pybamm' has no attribute 'isfinite'
[DCAC_full_protocol] trying CV termination: 73.5294117647058 mA
[DCAC_full_protocol] failed with termination 73.5294117647058 mA: module 'pybamm' has no attribute 'isfinite'


RuntimeError: DCAC_full_protocol: all cutoff termination variants failed. Last error: module 'pybamm' has no attribute 'isfinite'

In [17]:
# Cell 9B — Re-run Chen2020 full CC+CV smoke with tabulated DCAC drive-cycle current
#
# Reason:
#   The previous DCAC step.current(callable) path failed with:
#       module 'pybamm' has no attribute 'isfinite'
#
#   This cell avoids callable current entirely. It replaces the DCAC CC step by
#   a tabulated drive-cycle array:
#       [time_s, current_A]
#
# Requirements:
#   This cell assumes Cell 9 already defined:
#       pybamm, np, pd, cumulative_trapezoid
#       PARAM_SET, INITIAL_SOC
#       OUTPUT_PERIOD_S, MAX_CC_DURATION_S, MAX_CV_DURATION_S
#       dc_row, dcac_row
#       make_dfn_simulation()
#       build_dc_experiment()
#       solve_with_cutoff_termination()
#       extract_solution()
#
# Outputs overwrite the previous Cell 9 output files if successful.

import numpy as np
import pandas as pd

print("=" * 72)
print("Cell 9B — Chen2020 full CC+CV smoke, drive-cycle DCAC current")
print("=" * 72)


def build_dcac_experiment(row, cv_termination):
    """
    DCAC full protocol using tabulated drive-cycle current instead of callable.

    CC:
        I_py(t) = -I_DC_A - I_AC_A * sin(2*pi*f_Hz*t)
        until Vmax

    CV:
        AC off
        hold Vmax until |I_py| <= I_cutoff_A
    """
    I_DC_A = float(row["I_DC_A"])
    I_AC_A = float(row["I_AC_A"])
    f_Hz = float(row["f_Hz"])
    Vmax = float(row["Vmax_cutoff_V"])
    T_period_s = float(row["T_period_s"])

    # Use a sufficiently fine drive-cycle grid.
    # 10 s gives ~245 points per 10tau period for Chen2020 here.
    dt_drive_s = min(float(OUTPUT_PERIOD_S), T_period_s / 200.0)
    dt_drive_s = max(dt_drive_s, 1.0)

    t_drive = np.arange(0.0, float(MAX_CC_DURATION_S) + dt_drive_s, dt_drive_s)

    I_drive = -I_DC_A - I_AC_A * np.sin(2.0 * np.pi * f_Hz * t_drive)

    drive_cycle = np.column_stack([t_drive, I_drive])

    print(
        f"[DCAC drive-cycle] dt={dt_drive_s:.3f}s, n={len(t_drive)}, "
        f"I_min={I_drive.min():.6f} A, I_max={I_drive.max():.6f} A"
    )

    steps = [
        pybamm.step.current(
            drive_cycle,
            termination=f"{Vmax} V",
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=cv_termination,
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


print("Patched build_dcac_experiment(): drive-cycle array version loaded.")
print("Now rerunning DC and DCAC simulations...")


# ---------------------------------------------------------------------
# Re-run simulations
# ---------------------------------------------------------------------

solutions = {}
traj_list = []
summary_list = []

for arm, row in [
    ("DC_reference", dc_row),
    ("DCAC_full_protocol", dcac_row),
]:
    sol, term_used = solve_with_cutoff_termination(arm, row)
    solutions[arm] = sol

    traj, summary = extract_solution(sol, arm, row, term_used)
    traj_list.append(traj)
    summary_list.append(summary)

traj_df = pd.concat(traj_list, ignore_index=True)
summary_df = pd.DataFrame(summary_list)


# ---------------------------------------------------------------------
# Pair-level comparison
# ---------------------------------------------------------------------

dc_sum = summary_df[summary_df["arm"].eq("DC_reference")].iloc[0]
dcac_sum = summary_df[summary_df["arm"].eq("DCAC_full_protocol")].iloc[0]

pair_summary = {
    "param_set": PARAM_SET,
    "initial_soc": INITIAL_SOC,

    "t_end_DC_s": dc_sum["t_end_s"],
    "t_end_DCAC_s": dcac_sum["t_end_s"],
    "dt_total_s": dc_sum["t_end_s"] - dcac_sum["t_end_s"],
    "dt_total_min": (dc_sum["t_end_s"] - dcac_sum["t_end_s"]) / 60.0,

    "Q_end_DC_Ah": dc_sum["Q_end_Ah"],
    "Q_end_DCAC_Ah": dcac_sum["Q_end_Ah"],
    "Q_end_diff_Ah": dc_sum["Q_end_Ah"] - dcac_sum["Q_end_Ah"],

    "t_to_Vmax_DC_s": dc_sum["t_to_Vmax_sample_s"],
    "t_to_Vmax_DCAC_s": dcac_sum["t_to_Vmax_sample_s"],
    "dt_to_Vmax_s": dc_sum["t_to_Vmax_sample_s"] - dcac_sum["t_to_Vmax_sample_s"],

    "Q_to_Vmax_DC_Ah": dc_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_DCAC_Ah": dcac_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_shift_Ah": dc_sum["Q_to_Vmax_sample_Ah"] - dcac_sum["Q_to_Vmax_sample_Ah"],

    "DC_cutoff_reached": dc_sum["cutoff_reached_by_final_current"],
    "DCAC_cutoff_reached": dcac_sum["cutoff_reached_by_final_current"],

    "DC_has_cv_tail": dc_sum["has_cv_tail"],
    "DCAC_has_cv_tail": dcac_sum["has_cv_tail"],

    "DC_Vmax_violation_max_over_V": dc_sum["Vmax_violation_max_over_V"],
    "DCAC_Vmax_violation_max_over_V": dcac_sum["Vmax_violation_max_over_V"],
}

pair_df = pd.DataFrame([pair_summary])


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_traj = DATA / "day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv"
out_summary = DATA / "day20B_step1_Chen2020_full_protocol_smoke_summary.csv"
out_pair = DATA / "day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv"

traj_df.to_csv(out_traj, index=False)
summary_df.to_csv(out_summary, index=False)
pair_df.to_csv(out_pair, index=False)

print(f"\nWrote: {out_traj} ({len(traj_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
print(f"Wrote: {out_pair} ({len(pair_df)} rows)")

display(summary_df)
display(pair_df)


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert summary_df["has_cv_tail"].all(), (
    "Expected both arms to contain CV tail. Full-protocol simulation did not enter CV correctly."
)

assert summary_df["cutoff_reached_by_final_current"].all(), (
    "Expected both arms to terminate near normalized cutoff current."
)

assert (summary_df["Vmax_violation_max_over_V"] < 0.02).all(), (
    "Voltage exceeded Vmax by more than 20 mV."
)

assert pair_df["DC_has_cv_tail"].iloc[0] and pair_df["DCAC_has_cv_tail"].iloc[0]

print("\n" + "=" * 72)
print("Cell 9B PASSED — Chen2020 full CC+CV smoke simulation complete")
print("=" * 72)

Cell 9B — Chen2020 full CC+CV smoke, drive-cycle DCAC current
Patched build_dcac_experiment(): drive-cycle array version loaded.
Now rerunning DC and DCAC simulations...
[DC_reference] trying CV termination: 0.0735294117647058 A
[DC_reference] solve succeeded with termination 0.0735294117647058 A
[DCAC_full_protocol] trying CV termination: 0.0735294117647058 A
[DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[DCAC_full_protocol] failed with termination 0.0735294117647058 A: Termination must include an operator when using InputParameter.
[DCAC_full_protocol] trying CV termination: 73.5294117647058 mA
[DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[DCAC_full_protocol] failed with termination 73.5294117647058 mA: Termination must include an operator when using InputParameter.


RuntimeError: DCAC_full_protocol: all cutoff termination variants failed. Last error: Termination must include an operator when using InputParameter.

In [18]:
# Cell 9C — Chen2020 full CC+CV smoke using CustomTermination
#
# Purpose:
#   Avoid PyBaMM string-termination parser issues with drive-cycle / InputParameter current.
#
# Previous failures:
#   9  : callable current + pybamm.sin -> pybamm.isfinite issue
#   9B : drive-cycle current + string termination -> "Termination must include an operator..."
#
# Strategy:
#   Use:
#     - drive-cycle tabulated DCAC current during CC
#     - pybamm.step.CustomTermination for Vmax event
#     - pybamm.step.CustomTermination for CV current cutoff
#
# Protocol:
#   DC reference:
#       CC: I_py = -I_DC_A until Vmax
#       CV: hold Vmax until -I_py <= I_cutoff_A
#
#   DCAC:
#       CC: I_py = -I_DC_A - I_AC_A*sin(2*pi*f_Hz*t) until Vmax
#       CV: AC off, hold Vmax until -I_py <= I_cutoff_A
#
# Outputs:
#   data/day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv

import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid

try:
    import pybamm
except Exception as e:
    raise RuntimeError(f"PyBaMM import failed: {e}")

DESIGN = DATA / "day20B_step0_full_protocol_design.csv"
assert DESIGN.exists(), f"Missing design table: {DESIGN}"

design = pd.read_csv(DESIGN)

PARAM_SET = "Chen2020"
INITIAL_SOC = 0.05

OUTPUT_PERIOD_S = 10.0
MAX_CC_DURATION_S = 60000.0
MAX_CV_DURATION_S = 50000.0

df_ch = design[design["param_set"].eq(PARAM_SET)].copy()
assert len(df_ch) == 2, f"Expected two Chen2020 design rows, got {len(df_ch)}"

dc_row = df_ch[df_ch["arm"].eq("DC_reference")].iloc[0].to_dict()
dcac_row = df_ch[df_ch["arm"].eq("DCAC_full_protocol")].iloc[0].to_dict()

print("=" * 72)
print("Cell 9C — Chen2020 full CC+CV smoke with CustomTermination")
print("=" * 72)
print(f"Parameter set     : {PARAM_SET}")
print(f"Initial SOC       : {INITIAL_SOC}")
print(f"Vmax              : {dc_row['Vmax_cutoff_V']} V")
print(f"I_cutoff          : {dc_row['I_cutoff_A']:.9f} A ({dc_row['I_cutoff_mA']:.3f} mA)")
print(f"DC current        : {dc_row['I_DC_A']:.6f} A")
print(f"DCAC I_DC         : {dcac_row['I_DC_A']:.6f} A")
print(f"DCAC I_AC         : {dcac_row['I_AC_A']:.6f} A")
print(f"DCAC f            : {dcac_row['f_Hz']:.9g} Hz")
print(f"DCAC T            : {dcac_row['T_period_s']:.3f} s")
print()


# ---------------------------------------------------------------------
# Custom termination helpers
# ---------------------------------------------------------------------

def make_vmax_termination(Vmax):
    """
    Event function positive before Vmax and reaches zero at Vmax.
    """
    Vmax = float(Vmax)

    def event_function(variables):
        return Vmax - variables["Terminal voltage [V]"]

    return pybamm.step.CustomTermination(
        name=f"Vmax_{Vmax:.4f}V",
        event_function=event_function,
    )


def make_charge_current_cutoff_termination(I_cutoff_A):
    """
    CV charge cutoff.

    PyBaMM sign:
        I < 0 during charge.
    Terminate when |I| <= I_cutoff, i.e.
        -I - I_cutoff = 0
    """
    I_cutoff_A = float(I_cutoff_A)

    def event_function(variables):
        return -variables["Current [A]"] - I_cutoff_A

    return pybamm.step.CustomTermination(
        name=f"charge_current_cutoff_{I_cutoff_A:.6f}A",
        event_function=event_function,
    )


# ---------------------------------------------------------------------
# Simulation builders
# ---------------------------------------------------------------------

def make_dfn_simulation(param_set, experiment):
    model = pybamm.lithium_ion.DFN()
    pv = pybamm.ParameterValues(param_set)

    solver = pybamm.CasadiSolver(
        mode="safe",
        rtol=1e-6,
        atol=1e-8,
        dt_max=120.0,
    )

    return pybamm.Simulation(
        model,
        parameter_values=pv,
        experiment=experiment,
        solver=solver,
    )


def build_dc_experiment(row):
    I_DC_A = float(row["I_DC_A"])
    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])

    vmax_term = make_vmax_termination(Vmax)
    cutoff_term = make_charge_current_cutoff_termination(I_cutoff_A)

    steps = [
        pybamm.step.current(
            -I_DC_A,
            duration=f"{MAX_CC_DURATION_S} seconds",
            termination=[vmax_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=[cutoff_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def build_dcac_experiment(row):
    I_DC_A = float(row["I_DC_A"])
    I_AC_A = float(row["I_AC_A"])
    f_Hz = float(row["f_Hz"])
    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])
    T_period_s = float(row["T_period_s"])

    vmax_term = make_vmax_termination(Vmax)
    cutoff_term = make_charge_current_cutoff_termination(I_cutoff_A)

    # Drive-cycle time resolution:
    # Keep output period 10 s, and ensure at least ~200 points per AC period.
    dt_drive_s = min(float(OUTPUT_PERIOD_S), T_period_s / 200.0)
    dt_drive_s = max(dt_drive_s, 1.0)

    t_drive = np.arange(0.0, float(MAX_CC_DURATION_S) + dt_drive_s, dt_drive_s)
    I_drive = -I_DC_A - I_AC_A * np.sin(2.0 * np.pi * f_Hz * t_drive)

    drive_cycle = np.column_stack([t_drive, I_drive])

    print(
        f"[DCAC drive-cycle] dt={dt_drive_s:.3f}s, n={len(t_drive)}, "
        f"I_min={I_drive.min():.6f} A, I_max={I_drive.max():.6f} A"
    )

    steps = [
        pybamm.step.current(
            drive_cycle,
            termination=[vmax_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=[cutoff_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def solve_arm(arm, row):
    print(f"[{arm}] building experiment")

    if arm == "DC_reference":
        exp = build_dc_experiment(row)
    elif arm == "DCAC_full_protocol":
        exp = build_dcac_experiment(row)
    else:
        raise ValueError(f"Unknown arm: {arm}")

    sim = make_dfn_simulation(PARAM_SET, exp)

    print(f"[{arm}] solving...")
    try:
        sol = sim.solve(initial_soc=INITIAL_SOC)
    except TypeError:
        sol = sim.solve()

    print(f"[{arm}] solve succeeded")
    return sol


# ---------------------------------------------------------------------
# Extraction helpers
# ---------------------------------------------------------------------

def extract_solution(sol, arm, row):
    t = np.asarray(sol["Time [s]"].entries, dtype=float)
    V = np.asarray(sol["Terminal voltage [V]"].entries, dtype=float)
    I = np.asarray(sol["Current [A]"].entries, dtype=float)

    if not (len(t) == len(V) == len(I)):
        raise ValueError(f"{arm}: length mismatch t/V/I")

    Q_Ah = -cumulative_trapezoid(I, t, initial=0.0) / 3600.0

    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])

    crossed = V >= Vmax - 1e-6
    if crossed.any():
        idx_vmax = int(np.argmax(crossed))
        t_vmax = float(t[idx_vmax])
        Q_vmax = float(Q_Ah[idx_vmax])
    else:
        idx_vmax = np.nan
        t_vmax = np.nan
        Q_vmax = np.nan

    if np.isfinite(t_vmax):
        stage = np.where(t <= t_vmax + 1e-9, "CC_until_Vmax", "CV_after_Vmax")
    else:
        stage = np.array(["unknown"] * len(t), dtype=object)

    cv_mask = stage == "CV_after_Vmax"
    n_cv = int(cv_mask.sum())

    if n_cv > 0:
        I_cv = I[cv_mask]
        V_cv = V[cv_mask]
        I_cv_abs_initial = float(abs(I_cv[0]))
        I_cv_abs_final = float(abs(I_cv[-1]))
        I_cv_abs_range = float(np.nanmax(np.abs(I_cv)) - np.nanmin(np.abs(I_cv)))
        V_cv_max = float(np.nanmax(V_cv))
        V_cv_min = float(np.nanmin(V_cv))
    else:
        I_cv_abs_initial = np.nan
        I_cv_abs_final = np.nan
        I_cv_abs_range = np.nan
        V_cv_max = np.nan
        V_cv_min = np.nan

    traj = pd.DataFrame({
        "arm": arm,
        "param_set": PARAM_SET,
        "t_s": t,
        "V_V": V,
        "I_A": I,
        "Q_Ah": Q_Ah,
        "stage": stage,
        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "termination_method": "CustomTermination",
    })

    summary = {
        "arm": arm,
        "param_set": PARAM_SET,
        "termination_method": "CustomTermination",
        "n_samples": len(t),

        "t_end_s": float(t[-1]),
        "Q_end_Ah": float(Q_Ah[-1]),
        "V_end_V": float(V[-1]),
        "I_end_A": float(I[-1]),
        "I_end_abs_A": float(abs(I[-1])),

        "V_min_V": float(np.nanmin(V)),
        "V_max_V": float(np.nanmax(V)),
        "I_min_A": float(np.nanmin(I)),
        "I_max_A": float(np.nanmax(I)),

        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "I_cutoff_mA": I_cutoff_A * 1000.0,

        "idx_vmax_sample": idx_vmax,
        "t_to_Vmax_sample_s": t_vmax,
        "Q_to_Vmax_sample_Ah": Q_vmax,

        "n_cv_samples": n_cv,
        "has_cv_tail": n_cv > 5,
        "I_cv_abs_initial_A": I_cv_abs_initial,
        "I_cv_abs_final_A": I_cv_abs_final,
        "I_cv_abs_range_A": I_cv_abs_range,
        "V_cv_min_V": V_cv_min,
        "V_cv_max_V": V_cv_max,

        "cutoff_reached_by_final_current": bool(abs(I[-1]) <= I_cutoff_A * 1.05),
        "Vmax_violation_max_over_V": float(np.nanmax(V) - Vmax),
    }

    return traj, summary


# ---------------------------------------------------------------------
# Run simulations
# ---------------------------------------------------------------------

solutions = {}
traj_list = []
summary_list = []

for arm, row in [
    ("DC_reference", dc_row),
    ("DCAC_full_protocol", dcac_row),
]:
    sol = solve_arm(arm, row)
    solutions[arm] = sol

    traj, summary = extract_solution(sol, arm, row)
    traj_list.append(traj)
    summary_list.append(summary)

traj_df = pd.concat(traj_list, ignore_index=True)
summary_df = pd.DataFrame(summary_list)


# ---------------------------------------------------------------------
# Pair-level comparison
# ---------------------------------------------------------------------

dc_sum = summary_df[summary_df["arm"].eq("DC_reference")].iloc[0]
dcac_sum = summary_df[summary_df["arm"].eq("DCAC_full_protocol")].iloc[0]

pair_summary = {
    "param_set": PARAM_SET,
    "initial_soc": INITIAL_SOC,

    "t_end_DC_s": dc_sum["t_end_s"],
    "t_end_DCAC_s": dcac_sum["t_end_s"],
    "dt_total_s": dc_sum["t_end_s"] - dcac_sum["t_end_s"],
    "dt_total_min": (dc_sum["t_end_s"] - dcac_sum["t_end_s"]) / 60.0,

    "Q_end_DC_Ah": dc_sum["Q_end_Ah"],
    "Q_end_DCAC_Ah": dcac_sum["Q_end_Ah"],
    "Q_end_diff_Ah": dc_sum["Q_end_Ah"] - dcac_sum["Q_end_Ah"],

    "t_to_Vmax_DC_s": dc_sum["t_to_Vmax_sample_s"],
    "t_to_Vmax_DCAC_s": dcac_sum["t_to_Vmax_sample_s"],
    "dt_to_Vmax_s": dc_sum["t_to_Vmax_sample_s"] - dcac_sum["t_to_Vmax_sample_s"],

    "Q_to_Vmax_DC_Ah": dc_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_DCAC_Ah": dcac_sum["Q_to_Vmax_sample_Ah"],
    "Q_to_Vmax_shift_Ah": dc_sum["Q_to_Vmax_sample_Ah"] - dcac_sum["Q_to_Vmax_sample_Ah"],

    "DC_cutoff_reached": dc_sum["cutoff_reached_by_final_current"],
    "DCAC_cutoff_reached": dcac_sum["cutoff_reached_by_final_current"],

    "DC_has_cv_tail": dc_sum["has_cv_tail"],
    "DCAC_has_cv_tail": dcac_sum["has_cv_tail"],

    "DC_Vmax_violation_max_over_V": dc_sum["Vmax_violation_max_over_V"],
    "DCAC_Vmax_violation_max_over_V": dcac_sum["Vmax_violation_max_over_V"],
}

pair_df = pd.DataFrame([pair_summary])


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_traj = DATA / "day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv"
out_summary = DATA / "day20B_step1_Chen2020_full_protocol_smoke_summary.csv"
out_pair = DATA / "day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv"

traj_df.to_csv(out_traj, index=False)
summary_df.to_csv(out_summary, index=False)
pair_df.to_csv(out_pair, index=False)

print(f"\nWrote: {out_traj} ({len(traj_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
print(f"Wrote: {out_pair} ({len(pair_df)} rows)")

display(summary_df)
display(pair_df)


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert summary_df["has_cv_tail"].all(), (
    "Expected both arms to contain CV tail. Full-protocol simulation did not enter CV correctly."
)

assert summary_df["cutoff_reached_by_final_current"].all(), (
    "Expected both arms to terminate near normalized cutoff current."
)

assert (summary_df["Vmax_violation_max_over_V"] < 0.02).all(), (
    "Voltage exceeded Vmax by more than 20 mV."
)

assert pair_df["DC_has_cv_tail"].iloc[0] and pair_df["DCAC_has_cv_tail"].iloc[0]

print("\n" + "=" * 72)
print("Cell 9C PASSED — Chen2020 full CC+CV smoke simulation complete")
print("=" * 72)

Cell 9C — Chen2020 full CC+CV smoke with CustomTermination
Parameter set     : Chen2020
Initial SOC       : 0.05
Vmax              : 4.2 V
I_cutoff          : 0.073529412 A (73.529 mA)
DC current        : 1.000000 A
DCAC I_DC         : 1.000000 A
DCAC I_AC         : 2.500000 A
DCAC f            : 0.000407578731 Hz
DCAC T            : 2453.514 s

[DC_reference] building experiment
[DC_reference] solving...
[DC_reference] solve succeeded
[DCAC_full_protocol] building experiment
[DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[DCAC_full_protocol] solving...
[DCAC_full_protocol] solve succeeded

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv (3651 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv (2 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv (1 rows)


,arm,param_set,termination_method,n_samples,t_end_s,Q_end_Ah,V_end_V,I_end_A,I_end_abs_A,V_min_V,...,Q_to_Vmax_sample_Ah,n_cv_samples,has_cv_tail,I_cv_abs_initial_A,I_cv_abs_final_A,I_cv_abs_range_A,V_cv_min_V,V_cv_max_V,cutoff_reached_by_final_current,Vmax_violation_max_over_V
0,DC_reference,Chen2020,CustomTermination,1938,19352.284482,4.872815,4.2,-0.073539,0.073539,3.157897,...,4.627667,270,True,0.979562,0.073539,0.906023,4.2,4.2,True,2.309264e-14
1,DCAC_full_protocol,Chen2020,CustomTermination,1713,17105.917746,4.872588,4.2,-0.073539,0.073539,3.157897,...,3.770222,431,True,3.376611,0.073539,3.303071,4.2,4.2,True,4.529710e-14


,param_set,initial_soc,t_end_DC_s,t_end_DCAC_s,dt_total_s,dt_total_min,Q_end_DC_Ah,Q_end_DCAC_Ah,Q_end_diff_Ah,t_to_Vmax_DC_s,...,dt_to_Vmax_s,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,Q_to_Vmax_shift_Ah,DC_cutoff_reached,DCAC_cutoff_reached,DC_has_cv_tail,DCAC_has_cv_tail,DC_Vmax_violation_max_over_V,DCAC_Vmax_violation_max_over_V
0,Chen2020,0.05,19352.284482,17105.917746,2246.366737,37.439446,4.872815,4.872588,0.000226,16659.601329,...,3860.315546,4.627667,3.770222,0.857446,True,True,True,True,2.309264e-14,4.529710e-14



Cell 9C PASSED — Chen2020 full CC+CV smoke simulation complete


In [20]:
# Cell 10 — Chen2020 full-protocol Δt(Q) segment decomposition
#
# Purpose:
#   Compute full-protocol Δt(Q) from Cell 9C Chen2020 trajectories and classify
#   each Q target into event-defined segments:
#
#       Segment A: Q <= Q_DCAC,Vmax
#                  shared pre-DCAC-Vmax region
#
#       Segment B: Q_DCAC,Vmax < Q <= Q_DC,Vmax
#                  voltage-boundary / control-state split region
#                  DCAC already in CV, DC still approaching Vmax
#
#       Segment D: Q > Q_DC,Vmax
#                  late region after DC reaches Vmax; CV feedback dominated
#
# Inputs:
#   data/day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv
#
# Outputs:
#   data/day20B_step2_Chen2020_full_protocol_dtQ_segments.csv
#   data/day20B_step2_Chen2020_full_protocol_segment_summary.csv
#
# Note:
#   This is full-protocol raw Δt(Q), not yet geometry-corrected beyond Segment A.
#   Segment A geometry residual was already shown to be numerical-null in Day20A.

import numpy as np
import pandas as pd

TRAJ = DATA / "day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv"
SUMMARY = DATA / "day20B_step1_Chen2020_full_protocol_smoke_summary.csv"
PAIR = DATA / "day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv"

assert TRAJ.exists(), f"Missing: {TRAJ}"
assert SUMMARY.exists(), f"Missing: {SUMMARY}"
assert PAIR.exists(), f"Missing: {PAIR}"

traj = pd.read_csv(TRAJ)
summary = pd.read_csv(SUMMARY)
pair = pd.read_csv(PAIR).iloc[0]

PARAM_SET = "Chen2020"
N_Q = 120

print("=" * 72)
print("Cell 10 — Chen2020 full-protocol Δt(Q) segment decomposition")
print("=" * 72)
print(f"Trajectory rows: {len(traj)}")
print(f"Summary rows   : {len(summary)}")
print()


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def first_passage_time_by_Q(df, Q_target_Ah, value_cols=None):
    """
    Raw first-passage by Q_Ah with linear interpolation in t and requested variables.

    Returns dict:
        found, t_s, Q_Ah, <interpolated value cols>
    """
    if value_cols is None:
        value_cols = []

    q = df["Q_Ah"].to_numpy(dtype=float)
    t = df["t_s"].to_numpy(dtype=float)

    crossed = q >= Q_target_Ah

    if not crossed.any():
        out = {
            "found": False,
            "t_s": np.nan,
            "Q_Ah": Q_target_Ah,
        }
        for c in value_cols:
            out[c] = np.nan
        return out

    idx = int(np.argmax(crossed))

    if idx == 0:
        out = {
            "found": True,
            "t_s": float(t[0]),
            "Q_Ah": Q_target_Ah,
        }
        for c in value_cols:
            out[c] = float(df[c].iloc[0])
        return out

    q0, q1 = q[idx - 1], q[idx]
    t0, t1 = t[idx - 1], t[idx]

    if q1 == q0:
        frac = 1.0
    else:
        frac = (Q_target_Ah - q0) / (q1 - q0)
        frac = float(np.clip(frac, 0.0, 1.0))

    out = {
        "found": True,
        "t_s": float(t0 + frac * (t1 - t0)),
        "Q_Ah": Q_target_Ah,
    }

    for c in value_cols:
        y = df[c].to_numpy()
        y0, y1 = y[idx - 1], y[idx]

        # stage is string; choose post-crossing stage if nonnumeric
        if not np.issubdtype(np.asarray(y).dtype, np.number):
            out[c] = str(y1)
        else:
            out[c] = float(y0 + frac * (y1 - y0))

    return out


def classify_segment(Q_Ah, Q_DCAC_vmax_Ah, Q_DC_vmax_Ah):
    if Q_Ah <= Q_DCAC_vmax_Ah + 1e-12:
        return "A_pre_DCAC_Vmax"
    if Q_Ah <= Q_DC_vmax_Ah + 1e-12:
        return "B_between_DCAC_Vmax_and_DC_Vmax"
    return "D_late_CV_feedback_region"


def segment_interpretation(segment):
    if segment == "A_pre_DCAC_Vmax":
        return "shared pre-Vmax region; Segment A geometry-dominated"
    if segment == "B_between_DCAC_Vmax_and_DC_Vmax":
        return "DCAC already in CV/control-limited state while DC still approaches Vmax"
    if segment == "D_late_CV_feedback_region":
        return "late post-DC-Vmax region; both branches have reached voltage boundary"
    return "unknown"


# ---------------------------------------------------------------------
# Split arms and validate
# ---------------------------------------------------------------------

dc = traj[traj["arm"].eq("DC_reference")].copy().sort_values("t_s").reset_index(drop=True)
dcac = traj[traj["arm"].eq("DCAC_full_protocol")].copy().sort_values("t_s").reset_index(drop=True)

assert len(dc) > 0, "No DC_reference rows"
assert len(dcac) > 0, "No DCAC_full_protocol rows"

# ---------------------------------------------------------------------
# Validate trajectories
# ---------------------------------------------------------------------
# Time must be monotone.
# Q_Ah is allowed to be non-monotone under DCAC because AC > DC can create
# instantaneous discharge half-cycles. Day20 uses raw strict-net first-passage,
# not cummax and not monotonic forcing.

q_diag_rows = []

for name, df in [("DC", dc), ("DCAC", dcac)]:
    t_arr = df["t_s"].to_numpy(dtype=float)
    q_arr = df["Q_Ah"].to_numpy(dtype=float)
    i_arr = df["I_A"].to_numpy(dtype=float)

    assert np.all(np.diff(t_arr) >= 0), f"{name}: t not monotone"

    dQ = np.diff(q_arr)
    neg_dQ = dQ < -1e-9

    q_diag_rows.append({
        "arm": name,
        "n_samples": len(df),
        "Q_start_Ah": float(q_arr[0]),
        "Q_end_Ah": float(q_arr[-1]),
        "Q_min_Ah": float(np.nanmin(q_arr)),
        "Q_max_Ah": float(np.nanmax(q_arr)),
        "n_negative_dQ_steps": int(np.sum(neg_dQ)),
        "min_dQ_Ah": float(np.nanmin(dQ)) if len(dQ) else np.nan,
        "I_min_A": float(np.nanmin(i_arr)),
        "I_max_A": float(np.nanmax(i_arr)),
        "has_positive_discharge_current": bool(np.nanmax(i_arr) > 0),
        "Q_monotonic_required": False if name == "DCAC" else True,
    })

q_diag_df = pd.DataFrame(q_diag_rows)

out_qdiag = DATA / "day20B_step2_Chen2020_Q_monotonicity_diagnostic.csv"
q_diag_df.to_csv(out_qdiag, index=False)

print(f"Wrote: {out_qdiag}")
display(q_diag_df)

# DC should be monotone charging. DCAC may be non-monotone due to AC reversal.
dc_q_diag = q_diag_df[q_diag_df["arm"].eq("DC")].iloc[0]
assert dc_q_diag["n_negative_dQ_steps"] == 0, (
    "DC reference Q should be monotone; negative dQ found."
)

dcac_q_diag = q_diag_df[q_diag_df["arm"].eq("DCAC")].iloc[0]
assert dcac_q_diag["has_positive_discharge_current"], (
    "Expected DCAC to include positive current half-cycles because AC_C > DC_C."
)

Q_end_DC = float(dc["Q_Ah"].iloc[-1])
Q_end_DCAC = float(dcac["Q_Ah"].iloc[-1])
Q_common_max = min(Q_end_DC, Q_end_DCAC)

Q_DC_vmax = float(pair["Q_to_Vmax_DC_Ah"])
Q_DCAC_vmax = float(pair["Q_to_Vmax_DCAC_Ah"])

Q_lo = max(0.05 * Q_common_max, 0.05)  # avoid very early near-zero interpolation
Q_hi = Q_common_max

Q_grid = np.linspace(Q_lo, Q_hi, N_Q)

print(f"Q_end_DC      = {Q_end_DC:.6f} Ah")
print(f"Q_end_DCAC    = {Q_end_DCAC:.6f} Ah")
print(f"Q_common_max  = {Q_common_max:.6f} Ah")
print(f"Q_DCAC,Vmax   = {Q_DCAC_vmax:.6f} Ah")
print(f"Q_DC,Vmax     = {Q_DC_vmax:.6f} Ah")
print(f"Q_grid        = [{Q_lo:.6f}, {Q_hi:.6f}] Ah, n={N_Q}")


# ---------------------------------------------------------------------
# Compute full-protocol dt(Q)
# ---------------------------------------------------------------------

rows = []

for Q in Q_grid:
    dc_fp = first_passage_time_by_Q(
        dc,
        Q,
        value_cols=["V_V", "I_A", "stage"],
    )
    dcac_fp = first_passage_time_by_Q(
        dcac,
        Q,
        value_cols=["V_V", "I_A", "stage"],
    )

    found = dc_fp["found"] and dcac_fp["found"]

    if found:
        t_dc = dc_fp["t_s"]
        t_dcac = dcac_fp["t_s"]
        dt = t_dc - t_dcac
    else:
        t_dc = np.nan
        t_dcac = np.nan
        dt = np.nan

    seg = classify_segment(Q, Q_DCAC_vmax, Q_DC_vmax)

    rows.append({
        "param_set": PARAM_SET,
        "Q_Ah": float(Q),
        "Q_frac_common": float(Q / Q_common_max),
        "Q_frac_nominal_5Ah": float(Q / 5.0),

        "segment": seg,
        "segment_interpretation": segment_interpretation(seg),

        "t_DC_s": float(t_dc),
        "t_DCAC_s": float(t_dcac),
        "dt_full_s": float(dt),
        "dt_full_min": float(dt / 60.0) if np.isfinite(dt) else np.nan,

        "V_DC_V": dc_fp["V_V"],
        "I_DC_A": dc_fp["I_A"],
        "stage_DC": dc_fp["stage"],

        "V_DCAC_V": dcac_fp["V_V"],
        "I_DCAC_A": dcac_fp["I_A"],
        "stage_DCAC": dcac_fp["stage"],

        "Q_to_Vmax_DC_Ah": Q_DC_vmax,
        "Q_to_Vmax_DCAC_Ah": Q_DCAC_vmax,

        "DCAC_has_reached_Vmax": bool(Q >= Q_DCAC_vmax),
        "DC_has_reached_Vmax": bool(Q >= Q_DC_vmax),

        "found_both": bool(found),
    })

dtq_df = pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Segment summary
# ---------------------------------------------------------------------

summary_rows = []

for seg, g in dtq_df.groupby("segment", sort=False):
    dt = g["dt_full_s"].to_numpy(dtype=float)
    finite = np.isfinite(dt)

    if not finite.any():
        continue

    summary_rows.append({
        "param_set": PARAM_SET,
        "segment": seg,
        "segment_interpretation": segment_interpretation(seg),
        "n_points": int(len(g)),
        "n_finite": int(finite.sum()),

        "Q_lo_Ah": float(g["Q_Ah"].min()),
        "Q_hi_Ah": float(g["Q_Ah"].max()),

        "dt_full_mean_s": float(np.nanmean(dt)),
        "dt_full_median_s": float(np.nanmedian(dt)),
        "dt_full_min_s": float(np.nanmin(dt)),
        "dt_full_max_s": float(np.nanmax(dt)),
        "dt_full_mean_min": float(np.nanmean(dt) / 60.0),
        "dt_full_median_min": float(np.nanmedian(dt) / 60.0),
        "dt_full_min_min": float(np.nanmin(dt) / 60.0),
        "dt_full_max_min": float(np.nanmax(dt) / 60.0),

        "stage_DC_modes": "; ".join(g["stage_DC"].value_counts().index.astype(str).tolist()),
        "stage_DCAC_modes": "; ".join(g["stage_DCAC"].value_counts().index.astype(str).tolist()),
        "V_DC_median_V": float(np.nanmedian(g["V_DC_V"])),
        "V_DCAC_median_V": float(np.nanmedian(g["V_DCAC_V"])),
        "I_DC_median_A": float(np.nanmedian(g["I_DC_A"])),
        "I_DCAC_median_A": float(np.nanmedian(g["I_DCAC_A"])),
    })

seg_summary = pd.DataFrame(summary_rows)


# ---------------------------------------------------------------------
# Additional anchor points: Q80/Q90 of common final Q and nominal 5Ah
# ---------------------------------------------------------------------

anchor_targets = []

for label, Q in [
    ("Q80_common", 0.80 * Q_common_max),
    ("Q90_common", 0.90 * Q_common_max),
    ("Q80_nominal_5Ah", 0.80 * 5.0),
    ("Q90_nominal_5Ah", 0.90 * 5.0),
]:
    if Q <= Q_common_max:
        dc_fp = first_passage_time_by_Q(dc, Q, value_cols=["V_V", "I_A", "stage"])
        dcac_fp = first_passage_time_by_Q(dcac, Q, value_cols=["V_V", "I_A", "stage"])
        dt = dc_fp["t_s"] - dcac_fp["t_s"] if dc_fp["found"] and dcac_fp["found"] else np.nan
        seg = classify_segment(Q, Q_DCAC_vmax, Q_DC_vmax)

        anchor_targets.append({
            "anchor_label": label,
            "Q_Ah": Q,
            "segment": seg,
            "t_DC_s": dc_fp["t_s"],
            "t_DCAC_s": dcac_fp["t_s"],
            "dt_full_s": dt,
            "dt_full_min": dt / 60.0 if np.isfinite(dt) else np.nan,
            "stage_DC": dc_fp["stage"],
            "stage_DCAC": dcac_fp["stage"],
            "V_DC_V": dc_fp["V_V"],
            "V_DCAC_V": dcac_fp["V_V"],
            "I_DC_A": dc_fp["I_A"],
            "I_DCAC_A": dcac_fp["I_A"],
        })

anchors_df = pd.DataFrame(anchor_targets)


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_curves = DATA / "day20B_step2_Chen2020_full_protocol_dtQ_segments.csv"
out_summary = DATA / "day20B_step2_Chen2020_full_protocol_segment_summary.csv"
out_anchors = DATA / "day20B_step2_Chen2020_full_protocol_anchor_points.csv"

dtq_df.to_csv(out_curves, index=False)
seg_summary.to_csv(out_summary, index=False)
anchors_df.to_csv(out_anchors, index=False)

print(f"\nWrote: {out_curves} ({len(dtq_df)} rows)")
print(f"Wrote: {out_summary} ({len(seg_summary)} rows)")
print(f"Wrote: {out_anchors} ({len(anchors_df)} rows)")

print("\nSegment summary:")
display(seg_summary)

print("\nAnchor points:")
display(anchors_df)

print("\nFirst / last rows:")
display(dtq_df.head(5))
display(dtq_df.tail(5))


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert dtq_df["found_both"].all(), "Some Q targets not reached by both arms."

assert "A_pre_DCAC_Vmax" in set(dtq_df["segment"]), "Missing Segment A."
assert "B_between_DCAC_Vmax_and_DC_Vmax" in set(dtq_df["segment"]), "Missing Segment B."
assert "D_late_CV_feedback_region" in set(dtq_df["segment"]), "Missing Segment D."

assert (dtq_df["dt_full_s"] > 0).any(), "No positive full-protocol gain found."

print("\n" + "=" * 72)
print("Cell 10 PASSED — Chen2020 full-protocol Δt(Q) segment decomposition complete")
print("=" * 72)

Cell 10 — Chen2020 full-protocol Δt(Q) segment decomposition
Trajectory rows: 3651
Summary rows   : 2

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step2_Chen2020_Q_monotonicity_diagnostic.csv


,arm,n_samples,Q_start_Ah,Q_end_Ah,Q_min_Ah,Q_max_Ah,n_negative_dQ_steps,min_dQ_Ah,I_min_A,I_max_A,has_positive_discharge_current,Q_monotonic_required
0,DC,1938,-0.0,4.872815,-0.0,4.872815,0,8.881784e-16,-1.000270,-0.073539,False,True
1,DCAC,1713,-0.0,4.872588,-0.0,4.872588,453,-4.166082e-03,-3.499999,1.500000,True,False


Q_end_DC      = 4.872815 Ah
Q_end_DCAC    = 4.872588 Ah
Q_common_max  = 4.872588 Ah
Q_DCAC,Vmax   = 3.770222 Ah
Q_DC,Vmax     = 4.627667 Ah
Q_grid        = [0.243629, 4.872588] Ah, n=120

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step2_Chen2020_full_protocol_dtQ_segments.csv (120 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step2_Chen2020_full_protocol_segment_summary.csv (3 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step2_Chen2020_full_protocol_anchor_points.csv (4 rows)

Segment summary:


,param_set,segment,segment_interpretation,n_points,n_finite,Q_lo_Ah,Q_hi_Ah,dt_full_mean_s,dt_full_median_s,dt_full_min_s,...,dt_full_mean_min,dt_full_median_min,dt_full_min_min,dt_full_max_min,stage_DC_modes,stage_DCAC_modes,V_DC_median_V,V_DCAC_median_V,I_DC_median_A,I_DCAC_median_A
0,Chen2020,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,91,91,0.243629,3.744523,1262.728313,1266.180118,466.605875,...,21.045472,21.103002,7.776765,32.537896,CC_until_Vmax,CC_until_Vmax,3.751356,3.85367,-1.000000,-3.243297
1,Chen2020,B_between_DCAC_Vmax_and_DC_Vmax,DCAC already in CV/control-limited state while...,22,22,3.783422,4.600297,1687.112966,1752.128662,807.030373,...,28.118549,29.202144,13.450506,38.331120,CC_until_Vmax,CV_after_Vmax,4.136063,4.20000,-1.000000,-2.377669
2,Chen2020,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,7,7,4.639196,4.872588,2260.960989,2259.923735,2235.361097,...,37.682683,37.665396,37.256018,38.130892,CV_after_Vmax,CV_after_Vmax,4.200000,4.20000,-0.462265,-0.453075



Anchor points:


,anchor_label,Q_Ah,segment,t_DC_s,t_DCAC_s,dt_full_s,dt_full_min,stage_DC,stage_DCAC,V_DC_V,V_DCAC_V,I_DC_A,I_DCAC_A
0,Q80_common,3.898071,B_between_DCAC_Vmax_and_DC_Vmax,14033.054738,12944.723653,1088.331085,18.138851,CC_until_Vmax,CV_after_Vmax,4.108356,4.2,-1.0,-2.986573
1,Q90_common,4.385330,B_between_DCAC_Vmax_and_DC_Vmax,15787.186580,13676.462153,2110.724427,35.178740,CC_until_Vmax,CV_after_Vmax,4.149006,4.2,-1.0,-1.745372
2,Q80_nominal_5Ah,4.000000,B_between_DCAC_Vmax_and_DC_Vmax,14400.000000,13071.972570,1328.027430,22.133790,CC_until_Vmax,CV_after_Vmax,4.121730,4.2,-1.0,-2.789010
3,Q90_nominal_5Ah,4.500000,B_between_DCAC_Vmax_and_DC_Vmax,16200.000000,13946.289732,2253.710268,37.561838,CC_until_Vmax,CV_after_Vmax,4.166696,4.2,-1.0,-1.333211



First / last rows:


,param_set,Q_Ah,Q_frac_common,Q_frac_nominal_5Ah,segment,segment_interpretation,t_DC_s,t_DCAC_s,dt_full_s,dt_full_min,...,I_DC_A,stage_DC,V_DCAC_V,I_DCAC_A,stage_DCAC,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,DCAC_has_reached_Vmax,DC_has_reached_Vmax,found_both
0,Chen2020,0.243629,0.050000,0.048726,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,877.065921,402.589151,474.476771,7.907946,...,-1.0,CC_until_Vmax,3.474913,-3.144382,CC_until_Vmax,4.627667,3.770222,False,False,True
1,Chen2020,0.282528,0.057983,0.056506,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,1017.101656,446.192113,570.909543,9.515159,...,-1.0,CC_until_Vmax,3.506858,-3.274167,CC_until_Vmax,4.627667,3.770222,False,False,True
2,Chen2020,0.321427,0.065966,0.064285,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,1157.137392,488.304077,668.833315,11.147222,...,-1.0,CC_until_Vmax,3.534875,-3.372742,CC_until_Vmax,4.627667,3.770222,False,False,True
3,Chen2020,0.360326,0.073950,0.072065,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,1297.173127,529.376854,767.796273,12.796605,...,-1.0,CC_until_Vmax,3.558298,-3.442331,CC_until_Vmax,4.627667,3.770222,False,False,True
4,Chen2020,0.399225,0.081933,0.079845,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,1437.208862,569.786640,867.422222,14.457037,...,-1.0,CC_until_Vmax,3.577065,-3.484422,CC_until_Vmax,4.627667,3.770222,False,False,True


,param_set,Q_Ah,Q_frac_common,Q_frac_nominal_5Ah,segment,segment_interpretation,t_DC_s,t_DCAC_s,dt_full_s,dt_full_min,...,I_DC_A,stage_DC,V_DCAC_V,I_DCAC_A,stage_DCAC,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,DCAC_has_reached_Vmax,DC_has_reached_Vmax,found_both
115,Chen2020,4.716993,0.968067,0.943399,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,17078.738458,14812.016629,2266.721829,37.778697,...,-0.601532,CV_after_Vmax,4.2,-0.582391,CV_after_Vmax,4.627667,3.770222,True,True,True
116,Chen2020,4.755892,0.976050,0.951178,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,17343.903668,15083.979933,2259.923735,37.665396,...,-0.462265,CV_after_Vmax,4.2,-0.453075,CV_after_Vmax,4.627667,3.770222,True,True,True
117,Chen2020,4.794791,0.984034,0.958958,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,17701.248648,15447.241090,2254.007558,37.566793,...,-0.329748,CV_after_Vmax,4.2,-0.325188,CV_after_Vmax,4.627667,3.770222,True,True,True
118,Chen2020,4.833690,0.992017,0.966738,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,18240.125861,15992.859323,2247.266538,37.454442,...,-0.201006,CV_after_Vmax,4.2,-0.198693,CV_after_Vmax,4.627667,3.770222,True,True,True
119,Chen2020,4.872588,1.000000,0.974518,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,19341.278843,17105.917746,2235.361097,37.256018,...,-0.074272,CV_after_Vmax,4.2,-0.073539,CV_after_Vmax,4.627667,3.770222,True,True,True



Cell 10 PASSED — Chen2020 full-protocol Δt(Q) segment decomposition complete


In [21]:
# Cell 11 — Chen2020 full-protocol smoke verdict
#
# Purpose:
#   Close the single-parameter-set full-protocol smoke test for Chen2020.
#
# Inputs:
#   data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv
#   data/day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv
#   data/day20B_step2_Chen2020_full_protocol_segment_summary.csv
#   data/day20B_step2_Chen2020_full_protocol_anchor_points.csv
#   data/day20B_step2_Chen2020_Q_monotonicity_diagnostic.csv
#
# Outputs:
#   data/day20B_step3_Chen2020_full_protocol_smoke_verdict.csv
#   docs/day20B_Chen2020_full_protocol_smoke.md
#
# Interpretation:
#   This closes the Chen2020 smoke test only.
#   It does not yet generalize to OKane2022 / ORegan2022.

import numpy as np
import pandas as pd
from pathlib import Path

SMOKE_SUMMARY = DATA / "day20B_step1_Chen2020_full_protocol_smoke_summary.csv"
SMOKE_PAIR = DATA / "day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv"
SEG_SUMMARY = DATA / "day20B_step2_Chen2020_full_protocol_segment_summary.csv"
ANCHORS = DATA / "day20B_step2_Chen2020_full_protocol_anchor_points.csv"
Q_DIAG = DATA / "day20B_step2_Chen2020_Q_monotonicity_diagnostic.csv"

for p in [SMOKE_SUMMARY, SMOKE_PAIR, SEG_SUMMARY, ANCHORS, Q_DIAG]:
    assert p.exists(), f"Missing required file: {p}"

smoke = pd.read_csv(SMOKE_SUMMARY)
pair = pd.read_csv(SMOKE_PAIR).iloc[0]
seg = pd.read_csv(SEG_SUMMARY)
anchors = pd.read_csv(ANCHORS)
qdiag = pd.read_csv(Q_DIAG)

PARAM_SET = "Chen2020"

print("=" * 72)
print("Cell 11 — Chen2020 full-protocol smoke verdict")
print("=" * 72)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def ffloat(x):
    if pd.isna(x):
        return np.nan
    return float(x)


def fmt(x, nd=3):
    if pd.isna(x):
        return "nan"
    return f"{float(x):.{nd}f}"


def to_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return bool(x)


def get_seg(segment_name):
    out = seg[seg["segment"].eq(segment_name)]
    assert len(out) == 1, f"Expected one segment row for {segment_name}, got {len(out)}"
    return out.iloc[0]


def get_anchor(anchor_label):
    out = anchors[anchors["anchor_label"].eq(anchor_label)]
    if len(out) == 0:
        return None
    assert len(out) == 1, f"Expected one anchor row for {anchor_label}, got {len(out)}"
    return out.iloc[0]


def get_qdiag(arm):
    out = qdiag[qdiag["arm"].eq(arm)]
    assert len(out) == 1, f"Expected one qdiag row for {arm}, got {len(out)}"
    return out.iloc[0]


# ---------------------------------------------------------------------
# Extract key rows
# ---------------------------------------------------------------------

seg_A = get_seg("A_pre_DCAC_Vmax")
seg_B = get_seg("B_between_DCAC_Vmax_and_DC_Vmax")
seg_D = get_seg("D_late_CV_feedback_region")

dc_row = smoke[smoke["arm"].eq("DC_reference")].iloc[0]
dcac_row = smoke[smoke["arm"].eq("DCAC_full_protocol")].iloc[0]

qdiag_dc = get_qdiag("DC")
qdiag_dcac = get_qdiag("DCAC")

q80_common = get_anchor("Q80_common")
q90_common = get_anchor("Q90_common")
q80_nominal = get_anchor("Q80_nominal_5Ah")
q90_nominal = get_anchor("Q90_nominal_5Ah")


# ---------------------------------------------------------------------
# Verdict logic
# ---------------------------------------------------------------------

full_protocol_smoke_ok = (
    to_bool(pair["DC_cutoff_reached"])
    and to_bool(pair["DCAC_cutoff_reached"])
    and to_bool(pair["DC_has_cv_tail"])
    and to_bool(pair["DCAC_has_cv_tail"])
    and ffloat(pair["DC_Vmax_violation_max_over_V"]) < 0.02
    and ffloat(pair["DCAC_Vmax_violation_max_over_V"]) < 0.02
)

full_protocol_verdict = (
    "full_protocol_gain_positive_smoke_pass"
    if full_protocol_smoke_ok and ffloat(pair["dt_total_min"]) > 0
    else "requires_review"
)

mechanism_verdict = (
    "raw_full_protocol_gain_supported_but_not_non_geometric_Segment_A_acceleration"
)

q80_segment = q80_common["segment"] if q80_common is not None else ""
q90_segment = q90_common["segment"] if q90_common is not None else ""

anchors_in_B = (
    q80_segment == "B_between_DCAC_Vmax_and_DC_Vmax"
    and q90_segment == "B_between_DCAC_Vmax_and_DC_Vmax"
)


# ---------------------------------------------------------------------
# Build verdict table
# ---------------------------------------------------------------------

verdict = pd.DataFrame([{
    "param_set": PARAM_SET,
    "initial_soc": ffloat(pair["initial_soc"]),

    "full_protocol_smoke_ok": bool(full_protocol_smoke_ok),
    "full_protocol_verdict": full_protocol_verdict,
    "mechanism_verdict": mechanism_verdict,

    "t_end_DC_s": ffloat(pair["t_end_DC_s"]),
    "t_end_DCAC_s": ffloat(pair["t_end_DCAC_s"]),
    "dt_total_s": ffloat(pair["dt_total_s"]),
    "dt_total_min": ffloat(pair["dt_total_min"]),

    "Q_end_DC_Ah": ffloat(pair["Q_end_DC_Ah"]),
    "Q_end_DCAC_Ah": ffloat(pair["Q_end_DCAC_Ah"]),
    "Q_end_diff_Ah": ffloat(pair["Q_end_diff_Ah"]),

    "DC_cutoff_reached": to_bool(pair["DC_cutoff_reached"]),
    "DCAC_cutoff_reached": to_bool(pair["DCAC_cutoff_reached"]),
    "DC_has_cv_tail": to_bool(pair["DC_has_cv_tail"]),
    "DCAC_has_cv_tail": to_bool(pair["DCAC_has_cv_tail"]),
    "DC_Vmax_violation_max_over_V": ffloat(pair["DC_Vmax_violation_max_over_V"]),
    "DCAC_Vmax_violation_max_over_V": ffloat(pair["DCAC_Vmax_violation_max_over_V"]),

    "Q_to_Vmax_DC_Ah": ffloat(pair["Q_to_Vmax_DC_Ah"]),
    "Q_to_Vmax_DCAC_Ah": ffloat(pair["Q_to_Vmax_DCAC_Ah"]),
    "Q_to_Vmax_shift_Ah": ffloat(pair["Q_to_Vmax_shift_Ah"]),
    "dt_to_Vmax_s": ffloat(pair["dt_to_Vmax_s"]),

    "segment_A_verdict": "geometry_dominated",
    "segment_A_n_points": int(seg_A["n_points"]),
    "segment_A_Q_lo_Ah": ffloat(seg_A["Q_lo_Ah"]),
    "segment_A_Q_hi_Ah": ffloat(seg_A["Q_hi_Ah"]),
    "segment_A_dt_median_min": ffloat(seg_A["dt_full_median_min"]),
    "segment_A_dt_min_min": ffloat(seg_A["dt_full_min_min"]),
    "segment_A_dt_max_min": ffloat(seg_A["dt_full_max_min"]),

    "segment_B_verdict": "boundary_control_state_split",
    "segment_B_n_points": int(seg_B["n_points"]),
    "segment_B_Q_lo_Ah": ffloat(seg_B["Q_lo_Ah"]),
    "segment_B_Q_hi_Ah": ffloat(seg_B["Q_hi_Ah"]),
    "segment_B_dt_median_min": ffloat(seg_B["dt_full_median_min"]),
    "segment_B_dt_min_min": ffloat(seg_B["dt_full_min_min"]),
    "segment_B_dt_max_min": ffloat(seg_B["dt_full_max_min"]),

    "segment_D_verdict": "CV_feedback_preserves_gain",
    "segment_D_n_points": int(seg_D["n_points"]),
    "segment_D_Q_lo_Ah": ffloat(seg_D["Q_lo_Ah"]),
    "segment_D_Q_hi_Ah": ffloat(seg_D["Q_hi_Ah"]),
    "segment_D_dt_median_min": ffloat(seg_D["dt_full_median_min"]),
    "segment_D_dt_min_min": ffloat(seg_D["dt_full_min_min"]),
    "segment_D_dt_max_min": ffloat(seg_D["dt_full_max_min"]),

    "Q80_common_segment": q80_segment,
    "Q80_common_dt_min": ffloat(q80_common["dt_full_min"]) if q80_common is not None else np.nan,
    "Q90_common_segment": q90_segment,
    "Q90_common_dt_min": ffloat(q90_common["dt_full_min"]) if q90_common is not None else np.nan,

    "Q80_nominal_5Ah_segment": q80_nominal["segment"] if q80_nominal is not None else "",
    "Q80_nominal_5Ah_dt_min": ffloat(q80_nominal["dt_full_min"]) if q80_nominal is not None else np.nan,
    "Q90_nominal_5Ah_segment": q90_nominal["segment"] if q90_nominal is not None else "",
    "Q90_nominal_5Ah_dt_min": ffloat(q90_nominal["dt_full_min"]) if q90_nominal is not None else np.nan,

    "anchors_Q80_Q90_common_in_segment_B": bool(anchors_in_B),

    "DC_Q_nonmonotone_steps": int(qdiag_dc["n_negative_dQ_steps"]),
    "DCAC_Q_nonmonotone_steps": int(qdiag_dcac["n_negative_dQ_steps"]),
    "DCAC_has_positive_discharge_current": to_bool(qdiag_dcac["has_positive_discharge_current"]),

    "interpretation": (
        "Chen2020 full-protocol raw gain is positive and survives into late CV, "
        "but Q80/Q90 gains occur in Segment B where DCAC is already voltage-limited "
        "while DC remains in CC. This supports boundary/control-state explanation, "
        "not non-geometric Segment-A acceleration."
    ),
    "next_step": (
        "Run OKane2022 and ORegan2022 full-protocol smoke/batch using the same "
        "CustomTermination protocol."
    ),
}])

out_verdict = DATA / "day20B_step3_Chen2020_full_protocol_smoke_verdict.csv"
verdict.to_csv(out_verdict, index=False)

print(f"Wrote: {out_verdict}")
display(verdict)


# ---------------------------------------------------------------------
# Markdown documentation
# ---------------------------------------------------------------------

DOCS.mkdir(exist_ok=True)
out_doc = DOCS / "day20B_Chen2020_full_protocol_smoke.md"

doc_lines = []

doc_lines.append("# Day20B — Chen2020 full-protocol CC+CV smoke test")
doc_lines.append("")
doc_lines.append("Status: smoke test passed")
doc_lines.append("")
doc_lines.append("## Scope")
doc_lines.append("")
doc_lines.append("This document summarizes the first full CC+CV PyBaMM smoke simulation after Day20A.")
doc_lines.append("")
doc_lines.append("Parameter set:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append("Chen2020")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("Protocol logic:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append("DC reference:")
doc_lines.append("  CC at -0.2C until Vmax")
doc_lines.append("  CV at Vmax until |I| <= normalized cutoff")
doc_lines.append("")
doc_lines.append("DCAC:")
doc_lines.append("  charge-first DC+AC during CC")
doc_lines.append("  AC off at Vmax")
doc_lines.append("  pure DC-CV at Vmax until |I| <= normalized cutoff")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("Normalized cutoff:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append(f"C_cutoff = 0.05 / 3.4 = {0.05 / 3.4:.9f} C")
doc_lines.append("I_cutoff = C_cutoff · Q_nom")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("For Chen2020, `Q_nom = 5 Ah`, so `I_cutoff = 73.529 mA`.")
doc_lines.append("")
doc_lines.append("## Smoke result")
doc_lines.append("")
doc_lines.append("| metric | value |")
doc_lines.append("|---|---:|")
doc_lines.append(f"| DC end time [s] | {fmt(pair['t_end_DC_s'], 3)} |")
doc_lines.append(f"| DCAC end time [s] | {fmt(pair['t_end_DCAC_s'], 3)} |")
doc_lines.append(f"| total Δt [s] | {fmt(pair['dt_total_s'], 3)} |")
doc_lines.append(f"| total Δt [min] | {fmt(pair['dt_total_min'], 3)} |")
doc_lines.append(f"| DC final Q [Ah] | {fmt(pair['Q_end_DC_Ah'], 6)} |")
doc_lines.append(f"| DCAC final Q [Ah] | {fmt(pair['Q_end_DCAC_Ah'], 6)} |")
doc_lines.append(f"| DC cutoff reached | {to_bool(pair['DC_cutoff_reached'])} |")
doc_lines.append(f"| DCAC cutoff reached | {to_bool(pair['DCAC_cutoff_reached'])} |")
doc_lines.append(f"| DC CV tail | {to_bool(pair['DC_has_cv_tail'])} |")
doc_lines.append(f"| DCAC CV tail | {to_bool(pair['DCAC_has_cv_tail'])} |")
doc_lines.append("")
doc_lines.append("## Voltage-boundary result")
doc_lines.append("")
doc_lines.append("| metric | value |")
doc_lines.append("|---|---:|")
doc_lines.append(f"| Q_to_Vmax DC [Ah] | {fmt(pair['Q_to_Vmax_DC_Ah'], 6)} |")
doc_lines.append(f"| Q_to_Vmax DCAC [Ah] | {fmt(pair['Q_to_Vmax_DCAC_Ah'], 6)} |")
doc_lines.append(f"| Q_to_Vmax shift [Ah] | {fmt(pair['Q_to_Vmax_shift_Ah'], 6)} |")
doc_lines.append(f"| t_to_Vmax shift [s] | {fmt(pair['dt_to_Vmax_s'], 3)} |")
doc_lines.append("")
doc_lines.append("## Segment summary")
doc_lines.append("")
doc_lines.append("| segment | Q range [Ah] | median Δt [min] | min Δt [min] | max Δt [min] | interpretation |")
doc_lines.append("|---|---:|---:|---:|---:|---|")
doc_lines.append(
    f"| A | {fmt(seg_A['Q_lo_Ah'], 3)}–{fmt(seg_A['Q_hi_Ah'], 3)} | "
    f"{fmt(seg_A['dt_full_median_min'], 3)} | {fmt(seg_A['dt_full_min_min'], 3)} | "
    f"{fmt(seg_A['dt_full_max_min'], 3)} | geometry-dominated |"
)
doc_lines.append(
    f"| B | {fmt(seg_B['Q_lo_Ah'], 3)}–{fmt(seg_B['Q_hi_Ah'], 3)} | "
    f"{fmt(seg_B['dt_full_median_min'], 3)} | {fmt(seg_B['dt_full_min_min'], 3)} | "
    f"{fmt(seg_B['dt_full_max_min'], 3)} | boundary / control-state split |"
)
doc_lines.append(
    f"| D | {fmt(seg_D['Q_lo_Ah'], 3)}–{fmt(seg_D['Q_hi_Ah'], 3)} | "
    f"{fmt(seg_D['dt_full_median_min'], 3)} | {fmt(seg_D['dt_full_min_min'], 3)} | "
    f"{fmt(seg_D['dt_full_max_min'], 3)} | late CV feedback region |"
)
doc_lines.append("")
doc_lines.append("## Anchor points")
doc_lines.append("")
doc_lines.append("| anchor | segment | Δt [min] | DC stage | DCAC stage |")
doc_lines.append("|---|---|---:|---|---|")

for _, a in anchors.iterrows():
    doc_lines.append(
        f"| {a['anchor_label']} | {a['segment']} | {fmt(a['dt_full_min'], 3)} | "
        f"{a['stage_DC']} | {a['stage_DCAC']} |"
    )

doc_lines.append("")
doc_lines.append("## Interpretation")
doc_lines.append("")
doc_lines.append("The Chen2020 full CC+CV smoke simulation successfully implements the intended MJ1-like control logic at the PyBaMM protocol level.")
doc_lines.append("")
doc_lines.append("The full-protocol raw gain is positive:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append(f"Δt_total = {fmt(pair['dt_total_min'], 3)} min")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("The gain also survives into the late CV region:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append(f"Segment D median Δt = {fmt(seg_D['dt_full_median_min'], 3)} min")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("However, Q80/Q90 gains occur in Segment B, where the DCAC branch is already voltage-limited while the DC reference remains in CC. This supports a voltage-boundary / control-state explanation rather than non-geometric Segment-A acceleration.")
doc_lines.append("")
doc_lines.append("## Verdict")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append("Full-protocol raw gain: supported for Chen2020 smoke test.")
doc_lines.append("Non-geometric Segment-A acceleration: not supported by this smoke test.")
doc_lines.append("Dominant interpretation: boundary/control-state split, with CV preserving earlier advantage.")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("## Next step")
doc_lines.append("")
doc_lines.append("Run the same full-protocol workflow for:")
doc_lines.append("")
doc_lines.append("```text")
doc_lines.append("OKane2022")
doc_lines.append("ORegan2022")
doc_lines.append("```")
doc_lines.append("")
doc_lines.append("Then compare whether the Segment B / late-CV pattern is consistent across the charge-first full-protocol batch.")
doc_lines.append("")

out_doc.write_text("\n".join(doc_lines), encoding="utf-8")
print(f"Wrote: {out_doc}")


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert bool(verdict["full_protocol_smoke_ok"].iloc[0]), "Smoke test not OK."
assert verdict["dt_total_min"].iloc[0] > 0, "Expected positive full-protocol gain."
assert bool(verdict["anchors_Q80_Q90_common_in_segment_B"].iloc[0]), (
    "Expected Q80/Q90 common anchors to lie in Segment B for this smoke test."
)
assert verdict["segment_D_dt_median_min"].iloc[0] > 0, (
    "Expected positive late-CV median gain."
)
assert verdict["DCAC_Q_nonmonotone_steps"].iloc[0] > 0, (
    "Expected DCAC strict-net Q to have non-monotone steps due to AC reversal."
)

print("\n" + "=" * 72)
print("Cell 11 PASSED — Chen2020 full-protocol smoke verdict complete")
print("=" * 72)

Cell 11 — Chen2020 full-protocol smoke verdict
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step3_Chen2020_full_protocol_smoke_verdict.csv


,param_set,initial_soc,full_protocol_smoke_ok,full_protocol_verdict,mechanism_verdict,t_end_DC_s,t_end_DCAC_s,dt_total_s,dt_total_min,Q_end_DC_Ah,...,Q80_nominal_5Ah_segment,Q80_nominal_5Ah_dt_min,Q90_nominal_5Ah_segment,Q90_nominal_5Ah_dt_min,anchors_Q80_Q90_common_in_segment_B,DC_Q_nonmonotone_steps,DCAC_Q_nonmonotone_steps,DCAC_has_positive_discharge_current,interpretation,next_step
0,Chen2020,0.05,True,full_protocol_gain_positive_smoke_pass,raw_full_protocol_gain_supported_but_not_non_g...,19352.284482,17105.917746,2246.366737,37.439446,4.872815,...,B_between_DCAC_Vmax_and_DC_Vmax,22.13379,B_between_DCAC_Vmax_and_DC_Vmax,37.561838,True,0,453,True,Chen2020 full-protocol raw gain is positive an...,Run OKane2022 and ORegan2022 full-protocol smo...


Wrote: /Users/louislu/pybamm-dcac-superimposed/docs/day20B_Chen2020_full_protocol_smoke.md

Cell 11 PASSED — Chen2020 full-protocol smoke verdict complete


In [24]:
# Cell 12 — Day20B full CC+CV batch runner
#
# Purpose:
#   Run full CC+CV PyBaMM simulations for:
#       Chen2020
#       OKane2022
#       ORegan2022
#
# Protocol:
#   DC reference:
#       CC at -I_DC until Vmax
#       CV at Vmax until |I| <= normalized cutoff
#
#   DCAC:
#       charge-first DC+AC during CC
#       AC off at Vmax
#       pure DC-CV at Vmax until |I| <= normalized cutoff
#
# Cutoff:
#   I_cutoff = (0.05 / 3.4) * Q_nom
#
# Outputs:
#   data/day20B_step4_full_protocol_batch_trajectories.csv
#   data/day20B_step4_full_protocol_batch_summary.csv
#   data/day20B_step4_full_protocol_batch_pair_summary.csv
#   data/day20B_step4_full_protocol_batch_errors.csv

import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid

try:
    import pybamm
except Exception as e:
    raise RuntimeError(f"PyBaMM import failed: {e}")

DESIGN = DATA / "day20B_step0_full_protocol_design.csv"
assert DESIGN.exists(), f"Missing design table: {DESIGN}"

design = pd.read_csv(DESIGN)

PARAM_SETS_TO_RUN = ["Chen2020", "OKane2022", "ORegan2022"]
INITIAL_SOC = 0.05

OUTPUT_PERIOD_S = 10.0
MAX_CC_DURATION_S = 60000.0
MAX_CV_DURATION_S = 50000.0

print("=" * 72)
print("Cell 12 — Day20B full CC+CV batch runner")
print("=" * 72)
print(f"Parameter sets: {PARAM_SETS_TO_RUN}")
print(f"Initial SOC   : {INITIAL_SOC}")
print()


# ---------------------------------------------------------------------
# Custom termination helpers
# ---------------------------------------------------------------------

def make_vmax_termination(Vmax):
    """
    Event function positive below Vmax and reaches zero at Vmax.
    """
    Vmax = float(Vmax)

    def event_function(variables):
        return Vmax - variables["Terminal voltage [V]"]

    return pybamm.step.CustomTermination(
        name=f"Vmax_{Vmax:.4f}V",
        event_function=event_function,
    )


def make_charge_current_cutoff_termination(I_cutoff_A):
    """
    CV charge cutoff.

    PyBaMM sign:
        I < 0 during charge.

    Terminate when:
        |I| <= I_cutoff
    For charging current:
        -I - I_cutoff = 0
    """
    I_cutoff_A = float(I_cutoff_A)

    def event_function(variables):
        return -variables["Current [A]"] - I_cutoff_A

    return pybamm.step.CustomTermination(
        name=f"charge_current_cutoff_{I_cutoff_A:.6f}A",
        event_function=event_function,
    )


# ---------------------------------------------------------------------
# Simulation builders
# ---------------------------------------------------------------------

def make_dfn_simulation(param_set, experiment):
    model = pybamm.lithium_ion.DFN()
    pv = pybamm.ParameterValues(param_set)

    solver = pybamm.CasadiSolver(
        mode="safe",
        rtol=1e-6,
        atol=1e-8,
        dt_max=120.0,
    )

    return pybamm.Simulation(
        model,
        parameter_values=pv,
        experiment=experiment,
        solver=solver,
    )


def build_dc_experiment(row):
    I_DC_A = float(row["I_DC_A"])
    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])

    vmax_term = make_vmax_termination(Vmax)
    cutoff_term = make_charge_current_cutoff_termination(I_cutoff_A)

    steps = [
        pybamm.step.current(
            -I_DC_A,
            duration=f"{MAX_CC_DURATION_S} seconds",
            termination=[vmax_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=[cutoff_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def build_dcac_experiment(row):
    I_DC_A = float(row["I_DC_A"])
    I_AC_A = float(row["I_AC_A"])
    f_Hz = float(row["f_Hz"])
    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])
    T_period_s = float(row["T_period_s"])

    vmax_term = make_vmax_termination(Vmax)
    cutoff_term = make_charge_current_cutoff_termination(I_cutoff_A)

    # Drive-cycle resolution:
    # - at least ~200 points per AC period
    # - no coarser than output period
    # - not finer than 1 s
    dt_drive_s = min(float(OUTPUT_PERIOD_S), T_period_s / 200.0)
    dt_drive_s = max(dt_drive_s, 1.0)

    t_drive = np.arange(0.0, float(MAX_CC_DURATION_S) + dt_drive_s, dt_drive_s)
    I_drive = -I_DC_A - I_AC_A * np.sin(2.0 * np.pi * f_Hz * t_drive)

    drive_cycle = np.column_stack([t_drive, I_drive])

    print(
        f"[{row['param_set']} DCAC drive-cycle] "
        f"dt={dt_drive_s:.3f}s, n={len(t_drive)}, "
        f"I_min={I_drive.min():.6f} A, I_max={I_drive.max():.6f} A"
    )

    steps = [
        pybamm.step.current(
            drive_cycle,
            termination=[vmax_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
        pybamm.step.voltage(
            Vmax,
            duration=f"{MAX_CV_DURATION_S} seconds",
            termination=[cutoff_term],
            period=f"{OUTPUT_PERIOD_S} seconds",
        ),
    ]

    return pybamm.Experiment(steps)


def solve_arm(param_set, arm, row):
    print(f"[{param_set} | {arm}] building experiment")

    if arm == "DC_reference":
        exp = build_dc_experiment(row)
    elif arm == "DCAC_full_protocol":
        exp = build_dcac_experiment(row)
    else:
        raise ValueError(f"Unknown arm: {arm}")

    sim = make_dfn_simulation(param_set, exp)

    print(f"[{param_set} | {arm}] solving...")
    try:
        sol = sim.solve(initial_soc=INITIAL_SOC)
    except TypeError:
        sol = sim.solve()

    print(f"[{param_set} | {arm}] solve succeeded")
    return sol


# ---------------------------------------------------------------------
# Extraction helper
# ---------------------------------------------------------------------

def extract_solution(sol, param_set, arm, row):
    t = np.asarray(sol["Time [s]"].entries, dtype=float)
    V = np.asarray(sol["Terminal voltage [V]"].entries, dtype=float)
    I = np.asarray(sol["Current [A]"].entries, dtype=float)

    if not (len(t) == len(V) == len(I)):
        raise ValueError(f"{param_set}/{arm}: length mismatch t/V/I")

    Q_Ah = -cumulative_trapezoid(I, t, initial=0.0) / 3600.0

    Vmax = float(row["Vmax_cutoff_V"])
    I_cutoff_A = float(row["I_cutoff_A"])

    crossed = V >= Vmax - 1e-6
    if crossed.any():
        idx_vmax = int(np.argmax(crossed))
        t_vmax = float(t[idx_vmax])
        Q_vmax = float(Q_Ah[idx_vmax])
    else:
        idx_vmax = np.nan
        t_vmax = np.nan
        Q_vmax = np.nan

    if np.isfinite(t_vmax):
        stage = np.where(t <= t_vmax + 1e-9, "CC_until_Vmax", "CV_after_Vmax")
    else:
        stage = np.array(["unknown"] * len(t), dtype=object)

    cv_mask = stage == "CV_after_Vmax"
    n_cv = int(cv_mask.sum())

    if n_cv > 0:
        I_cv = I[cv_mask]
        V_cv = V[cv_mask]
        I_cv_abs_initial = float(abs(I_cv[0]))
        I_cv_abs_final = float(abs(I_cv[-1]))
        I_cv_abs_range = float(np.nanmax(np.abs(I_cv)) - np.nanmin(np.abs(I_cv)))
        V_cv_max = float(np.nanmax(V_cv))
        V_cv_min = float(np.nanmin(V_cv))
    else:
        I_cv_abs_initial = np.nan
        I_cv_abs_final = np.nan
        I_cv_abs_range = np.nan
        V_cv_max = np.nan
        V_cv_min = np.nan

    traj = pd.DataFrame({
        "param_set": param_set,
        "arm": arm,
        "t_s": t,
        "V_V": V,
        "I_A": I,
        "Q_Ah": Q_Ah,
        "stage": stage,
        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "termination_method": "CustomTermination",
        "initial_soc": INITIAL_SOC,
    })

    # Q monotonicity diagnostic
    dQ = np.diff(Q_Ah)
    n_negative_dQ = int(np.sum(dQ < -1e-9)) if len(dQ) else 0

    summary = {
        "param_set": param_set,
        "arm": arm,
        "termination_method": "CustomTermination",
        "initial_soc": INITIAL_SOC,
        "n_samples": len(t),

        "t_end_s": float(t[-1]),
        "Q_end_Ah": float(Q_Ah[-1]),
        "V_end_V": float(V[-1]),
        "I_end_A": float(I[-1]),
        "I_end_abs_A": float(abs(I[-1])),

        "V_min_V": float(np.nanmin(V)),
        "V_max_V": float(np.nanmax(V)),
        "I_min_A": float(np.nanmin(I)),
        "I_max_A": float(np.nanmax(I)),

        "Vmax_cutoff_V": Vmax,
        "I_cutoff_A": I_cutoff_A,
        "I_cutoff_mA": I_cutoff_A * 1000.0,

        "idx_vmax_sample": idx_vmax,
        "t_to_Vmax_sample_s": t_vmax,
        "Q_to_Vmax_sample_Ah": Q_vmax,

        "n_cv_samples": n_cv,
        "has_cv_tail": n_cv > 5,
        "I_cv_abs_initial_A": I_cv_abs_initial,
        "I_cv_abs_final_A": I_cv_abs_final,
        "I_cv_abs_range_A": I_cv_abs_range,
        "V_cv_min_V": V_cv_min,
        "V_cv_max_V": V_cv_max,

        "cutoff_reached_by_final_current": bool(abs(I[-1]) <= I_cutoff_A * 1.05),
        "Vmax_violation_max_over_V": float(np.nanmax(V) - Vmax),

        "Q_start_Ah": float(Q_Ah[0]),
        "Q_min_Ah": float(np.nanmin(Q_Ah)),
        "Q_max_Ah": float(np.nanmax(Q_Ah)),
        "n_negative_dQ_steps": n_negative_dQ,
        "has_positive_discharge_current": bool(np.nanmax(I) > 0),
    }

    return traj, summary


# ---------------------------------------------------------------------
# Run batch
# ---------------------------------------------------------------------

all_traj = []
all_summary = []
errors = []

for param_set in PARAM_SETS_TO_RUN:
    df_ps = design[design["param_set"].eq(param_set)].copy()

    if len(df_ps) != 2:
        errors.append({
            "param_set": param_set,
            "arm": "both",
            "error": f"Expected two design rows, got {len(df_ps)}",
        })
        continue

    for arm in ["DC_reference", "DCAC_full_protocol"]:
        try:
            row = df_ps[df_ps["arm"].eq(arm)].iloc[0].to_dict()
            sol = solve_arm(param_set, arm, row)
            traj, summary = extract_solution(sol, param_set, arm, row)

            all_traj.append(traj)
            all_summary.append(summary)

        except Exception as e:
            print(f"[ERROR] {param_set} | {arm}: {e}")
            errors.append({
                "param_set": param_set,
                "arm": arm,
                "error": str(e)[:1000],
            })

traj_df = pd.concat(all_traj, ignore_index=True) if all_traj else pd.DataFrame()
summary_df = pd.DataFrame(all_summary)
errors_df = pd.DataFrame(errors)


# ---------------------------------------------------------------------
# Pair-level summary
# ---------------------------------------------------------------------

pair_rows = []

for param_set in PARAM_SETS_TO_RUN:
    sub = summary_df[summary_df["param_set"].eq(param_set)]

    if len(sub) != 2:
        pair_rows.append({
            "param_set": param_set,
            "pair_status": "missing_arm",
        })
        continue

    dc = sub[sub["arm"].eq("DC_reference")].iloc[0]
    dcac = sub[sub["arm"].eq("DCAC_full_protocol")].iloc[0]

    pair_rows.append({
        "param_set": param_set,
        "pair_status": "ok",

        "initial_soc": INITIAL_SOC,

        "t_end_DC_s": dc["t_end_s"],
        "t_end_DCAC_s": dcac["t_end_s"],
        "dt_total_s": dc["t_end_s"] - dcac["t_end_s"],
        "dt_total_min": (dc["t_end_s"] - dcac["t_end_s"]) / 60.0,

        "Q_end_DC_Ah": dc["Q_end_Ah"],
        "Q_end_DCAC_Ah": dcac["Q_end_Ah"],
        "Q_end_diff_Ah": dc["Q_end_Ah"] - dcac["Q_end_Ah"],

        "t_to_Vmax_DC_s": dc["t_to_Vmax_sample_s"],
        "t_to_Vmax_DCAC_s": dcac["t_to_Vmax_sample_s"],
        "dt_to_Vmax_s": dc["t_to_Vmax_sample_s"] - dcac["t_to_Vmax_sample_s"],

        "Q_to_Vmax_DC_Ah": dc["Q_to_Vmax_sample_Ah"],
        "Q_to_Vmax_DCAC_Ah": dcac["Q_to_Vmax_sample_Ah"],
        "Q_to_Vmax_shift_Ah": dc["Q_to_Vmax_sample_Ah"] - dcac["Q_to_Vmax_sample_Ah"],

        "DC_cutoff_reached": dc["cutoff_reached_by_final_current"],
        "DCAC_cutoff_reached": dcac["cutoff_reached_by_final_current"],

        "DC_has_cv_tail": dc["has_cv_tail"],
        "DCAC_has_cv_tail": dcac["has_cv_tail"],

        "DC_Vmax_violation_max_over_V": dc["Vmax_violation_max_over_V"],
        "DCAC_Vmax_violation_max_over_V": dcac["Vmax_violation_max_over_V"],

        "DCAC_n_negative_dQ_steps": dcac["n_negative_dQ_steps"],
        "DCAC_has_positive_discharge_current": dcac["has_positive_discharge_current"],
    })

pair_df = pd.DataFrame(pair_rows)


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_traj = DATA / "day20B_step4_full_protocol_batch_trajectories.csv"
out_summary = DATA / "day20B_step4_full_protocol_batch_summary.csv"
out_pair = DATA / "day20B_step4_full_protocol_batch_pair_summary.csv"
out_errors = DATA / "day20B_step4_full_protocol_batch_errors.csv"

traj_df.to_csv(out_traj, index=False)
summary_df.to_csv(out_summary, index=False)
pair_df.to_csv(out_pair, index=False)
errors_df.to_csv(out_errors, index=False)

print(f"\nWrote: {out_traj} ({len(traj_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
print(f"Wrote: {out_pair} ({len(pair_df)} rows)")
print(f"Wrote: {out_errors} ({len(errors_df)} rows)")

print("\nBatch summary:")
display(summary_df)

print("\nPair summary:")
display(pair_df)

if len(errors_df):
    print("\nErrors:")
    display(errors_df)


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert len(errors_df) == 0, (
    "Batch contains errors:\n"
    f"{errors_df}"
)

assert len(summary_df) == 2 * len(PARAM_SETS_TO_RUN), (
    f"Expected {2 * len(PARAM_SETS_TO_RUN)} arm summaries, got {len(summary_df)}"
)

assert (summary_df["has_cv_tail"]).all(), (
    "Expected all arms to contain CV tail."
)

assert (summary_df["cutoff_reached_by_final_current"]).all(), (
    "Expected all arms to reach normalized cutoff current."
)

assert (summary_df["Vmax_violation_max_over_V"] < 0.02).all(), (
    "At least one arm exceeds Vmax by more than 20 mV."
)

assert (pair_df["pair_status"] == "ok").all(), (
    "At least one parameter set has missing pair arm."
)

assert (pair_df["dt_total_s"] > 0).all(), (
    "Expected positive full-protocol gain for all three smoke pairs."
)

print("\n" + "=" * 72)
print("Cell 12 PASSED — Day20B full-protocol batch runner complete")
print("=" * 72)

Cell 12 — Day20B full CC+CV batch runner
Parameter sets: ['Chen2020', 'OKane2022', 'ORegan2022']
Initial SOC   : 0.05

[Chen2020 | DC_reference] building experiment
[Chen2020 | DC_reference] solving...
[Chen2020 | DC_reference] solve succeeded
[Chen2020 | DCAC_full_protocol] building experiment
[Chen2020 DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[Chen2020 | DCAC_full_protocol] solving...
[Chen2020 | DCAC_full_protocol] solve succeeded
[OKane2022 | DC_reference] building experiment
[OKane2022 | DC_reference] solving...
[OKane2022 | DC_reference] solve succeeded
[OKane2022 | DCAC_full_protocol] building experiment
[OKane2022 DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[OKane2022 | DCAC_full_protocol] solving...
[OKane2022 | DCAC_full_protocol] solve succeeded
[ORegan2022 | DC_reference] building experiment
[ORegan2022 | DC_reference] solving...


At t = 76.734, , mxstep steps taken before reaching tout.
At t = 26.7352, , mxstep steps taken before reaching tout.


[ORegan2022 | DC_reference] solve succeeded
[ORegan2022 | DCAC_full_protocol] building experiment
[ORegan2022 DCAC drive-cycle] dt=10.000s, n=6001, I_min=-3.500000 A, I_max=1.500000 A
[ORegan2022 | DCAC_full_protocol] solving...


At t = 26.5773, , mxstep steps taken before reaching tout.
At t = 26.5773, , mxstep steps taken before reaching tout.
At t = 6.57732, , mxstep steps taken before reaching tout.
At t = 6.57732, , mxstep steps taken before reaching tout.
At t = 6.57732, , mxstep steps taken before reaching tout.
At t = 2.82732, , mxstep steps taken before reaching tout.
At t = 0.952322, , mxstep steps taken before reaching tout.


[ORegan2022 | DCAC_full_protocol] solve succeeded

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step4_full_protocol_batch_trajectories.csv (10382 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step4_full_protocol_batch_summary.csv (6 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step4_full_protocol_batch_pair_summary.csv (3 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step4_full_protocol_batch_errors.csv (0 rows)

Batch summary:


,param_set,arm,termination_method,initial_soc,n_samples,t_end_s,Q_end_Ah,V_end_V,I_end_A,I_end_abs_A,...,I_cv_abs_range_A,V_cv_min_V,V_cv_max_V,cutoff_reached_by_final_current,Vmax_violation_max_over_V,Q_start_Ah,Q_min_Ah,Q_max_Ah,n_negative_dQ_steps,has_positive_discharge_current
0,Chen2020,DC_reference,CustomTermination,0.05,1938,19352.284482,4.872815,4.2,-0.073539,0.073539,...,0.906023,4.2,4.2,True,1.412204e-13,-0.0,-0.0,4.872815,0,False
1,Chen2020,DCAC_full_protocol,CustomTermination,0.05,1713,17105.917724,4.872588,4.2,-0.073539,0.073539,...,3.303071,4.2,4.2,True,1.145750e-13,-0.0,-0.0,4.872588,453,True
2,OKane2022,DC_reference,CustomTermination,0.05,1917,19141.366422,4.835569,4.2,-0.073539,0.073539,...,0.906380,4.2,4.2,True,2.664535e-15,-0.0,-0.0,4.835569,0,False
3,OKane2022,DCAC_full_protocol,CustomTermination,0.05,1726,17232.149132,4.836702,4.2,-0.073539,0.073539,...,3.145615,4.2,4.2,True,2.664535e-15,-0.0,-0.0,4.836702,464,True
4,ORegan2022,DC_reference,CustomTermination,0.05,1790,17868.068927,4.792207,4.4,-0.073539,0.073539,...,0.639672,4.4,4.4,True,4.884981e-14,-0.0,-0.0,4.792207,0,False
5,ORegan2022,DCAC_full_protocol,CustomTermination,0.05,1298,12931.901269,4.796522,4.4,-0.073539,0.073539,...,6.863368,4.4,4.4,True,3.951790e-10,-0.0,-0.0,4.796522,332,True



Pair summary:


,param_set,pair_status,initial_soc,t_end_DC_s,t_end_DCAC_s,dt_total_s,dt_total_min,Q_end_DC_Ah,Q_end_DCAC_Ah,Q_end_diff_Ah,...,Q_to_Vmax_DCAC_Ah,Q_to_Vmax_shift_Ah,DC_cutoff_reached,DCAC_cutoff_reached,DC_has_cv_tail,DCAC_has_cv_tail,DC_Vmax_violation_max_over_V,DCAC_Vmax_violation_max_over_V,DCAC_n_negative_dQ_steps,DCAC_has_positive_discharge_current
0,Chen2020,ok,0.05,19352.284482,17105.917724,2246.366759,37.439446,4.872815,4.872588,0.000226,...,3.770222,0.857446,True,True,True,True,1.412204e-13,1.145750e-13,453,True
1,OKane2022,ok,0.05,19141.366422,17232.149132,1909.217290,31.820288,4.835569,4.836702,-0.001133,...,3.790496,0.806319,True,True,True,True,2.664535e-15,2.664535e-15,464,True
2,ORegan2022,ok,0.05,17868.068927,12931.901269,4936.167658,82.269461,4.792207,4.796522,-0.004315,...,3.210362,1.513654,True,True,True,True,4.884981e-14,3.951790e-10,332,True



Cell 12 PASSED — Day20B full-protocol batch runner complete


In [23]:
# Phase / current sanity check before Cell 12

design_check = pd.read_csv(DATA / "day20B_step0_full_protocol_design.csv")

dcac = design_check[design_check["arm"].eq("DCAC_full_protocol")].copy()

rows = []

for _, r in dcac.iterrows():
    I_DC = float(r["I_DC_A"])
    I_AC = float(r["I_AC_A"])
    f = float(r["f_Hz"])
    T = float(r["T_period_s"])

    t_probe = np.array([0.0, 0.01 * T, 0.25 * T, 0.50 * T, 0.75 * T])
    I_probe = -I_DC - I_AC * np.sin(2 * np.pi * f * t_probe)

    rows.append({
        "param_set": r["param_set"],
        "I_t0_A": I_probe[0],
        "I_0p01T_A": I_probe[1],
        "I_0p25T_A": I_probe[2],
        "I_0p50T_A": I_probe[3],
        "I_0p75T_A": I_probe[4],
        "initial_direction": "charge_first" if I_probe[1] < I_probe[0] else "not_charge_first",
        "I_min_expected_A": -I_DC - I_AC,
        "I_max_expected_A": -I_DC + I_AC,
        "peak_charge_C": float(r["peak_charge_C"]),
        "peak_discharge_C": float(r["peak_discharge_C"]),
    })

phase_check = pd.DataFrame(rows)
display(phase_check)

assert (phase_check["initial_direction"] == "charge_first").all()
assert (phase_check["peak_charge_C"] <= 1.0 + 1e-12).all()

,param_set,I_t0_A,I_0p01T_A,I_0p25T_A,I_0p50T_A,I_0p75T_A,initial_direction,I_min_expected_A,I_max_expected_A,peak_charge_C,peak_discharge_C
0,Chen2020,-1.0,-1.156976,-3.5,-1.0,1.5,charge_first,-3.5,1.5,0.7,0.3
1,OKane2022,-1.0,-1.156976,-3.5,-1.0,1.5,charge_first,-3.5,1.5,0.7,0.3
2,ORegan2022,-1.0,-1.156976,-3.5,-1.0,1.5,charge_first,-3.5,1.5,0.7,0.3


In [25]:
# Cell 13 — Day20B full-protocol batch Δt(Q) segment decomposition
#
# Purpose:
#   Extend Cell 10 from Chen2020 to the full Day20B batch:
#       Chen2020
#       OKane2022
#       ORegan2022
#
# Definition:
#   Δt_full(Q) = t_DC(Q) − t_DCAC(Q)
#
# Segment classification:
#   A: Q <= Q_DCAC,Vmax
#      shared pre-DCAC-Vmax region
#
#   B: Q_DCAC,Vmax < Q <= Q_DC,Vmax
#      voltage-boundary / control-state split
#      DCAC already voltage-limited, DC still approaching Vmax
#
#   D: Q > Q_DC,Vmax
#      late post-DC-Vmax / CV feedback region
#
# Outputs:
#   data/day20B_step5_full_protocol_batch_dtQ_segments.csv
#   data/day20B_step5_full_protocol_batch_segment_summary.csv
#   data/day20B_step5_full_protocol_batch_anchor_points.csv
#   data/day20B_step5_full_protocol_batch_Q_monotonicity_diagnostic.csv
#
# Important:
#   Uses raw strict-net first-passage. No cummax. No monotonic forcing.

import numpy as np
import pandas as pd

TRAJ = DATA / "day20B_step4_full_protocol_batch_trajectories.csv"
PAIR = DATA / "day20B_step4_full_protocol_batch_pair_summary.csv"
SUMMARY = DATA / "day20B_step4_full_protocol_batch_summary.csv"

assert TRAJ.exists(), f"Missing: {TRAJ}"
assert PAIR.exists(), f"Missing: {PAIR}"
assert SUMMARY.exists(), f"Missing: {SUMMARY}"

traj = pd.read_csv(TRAJ)
pair_df = pd.read_csv(PAIR)
arm_summary = pd.read_csv(SUMMARY)

PARAM_SETS = ["Chen2020", "OKane2022", "ORegan2022"]
N_Q = 120

print("=" * 72)
print("Cell 13 — full-protocol batch Δt(Q) segment decomposition")
print("=" * 72)
print(f"Trajectory rows: {len(traj)}")
print(f"Pair rows      : {len(pair_df)}")
print(f"Arm summaries  : {len(arm_summary)}")
print()


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def first_passage_time_by_Q(df, Q_target_Ah, value_cols=None):
    """
    Raw strict-net first-passage by Q_Ah with linear interpolation.

    Works for non-monotone Q(t) by scanning the first crossing:
        t(Q*) = min{t : Q(t) >= Q*}

    Does not use searchsorted.
    """
    if value_cols is None:
        value_cols = []

    q = df["Q_Ah"].to_numpy(dtype=float)
    t = df["t_s"].to_numpy(dtype=float)

    crossed = q >= Q_target_Ah

    if not crossed.any():
        out = {
            "found": False,
            "t_s": np.nan,
            "Q_Ah": Q_target_Ah,
        }
        for c in value_cols:
            out[c] = np.nan
        return out

    idx = int(np.argmax(crossed))

    if idx == 0:
        out = {
            "found": True,
            "t_s": float(t[0]),
            "Q_Ah": Q_target_Ah,
        }
        for c in value_cols:
            out[c] = df[c].iloc[0]
        return out

    q0, q1 = q[idx - 1], q[idx]
    t0, t1 = t[idx - 1], t[idx]

    if q1 == q0:
        frac = 1.0
    else:
        frac = (Q_target_Ah - q0) / (q1 - q0)
        frac = float(np.clip(frac, 0.0, 1.0))

    out = {
        "found": True,
        "t_s": float(t0 + frac * (t1 - t0)),
        "Q_Ah": Q_target_Ah,
    }

    for c in value_cols:
        y = df[c].to_numpy()
        if not np.issubdtype(np.asarray(y).dtype, np.number):
            out[c] = str(y[idx])
        else:
            y0, y1 = y[idx - 1], y[idx]
            out[c] = float(y0 + frac * (y1 - y0))

    return out


def classify_segment(Q_Ah, Q_DCAC_vmax_Ah, Q_DC_vmax_Ah):
    if Q_Ah <= Q_DCAC_vmax_Ah + 1e-12:
        return "A_pre_DCAC_Vmax"
    if Q_Ah <= Q_DC_vmax_Ah + 1e-12:
        return "B_between_DCAC_Vmax_and_DC_Vmax"
    return "D_late_CV_feedback_region"


def segment_interpretation(segment):
    if segment == "A_pre_DCAC_Vmax":
        return "shared pre-Vmax region; Segment A geometry-dominated"
    if segment == "B_between_DCAC_Vmax_and_DC_Vmax":
        return "DCAC already voltage-limited while DC still approaches Vmax"
    if segment == "D_late_CV_feedback_region":
        return "late post-DC-Vmax region; both branches have reached voltage boundary"
    return "unknown"


def q_monotonicity_diag(param_set, arm, df):
    q = df["Q_Ah"].to_numpy(dtype=float)
    i = df["I_A"].to_numpy(dtype=float)
    dQ = np.diff(q)

    return {
        "param_set": param_set,
        "arm": arm,
        "n_samples": len(df),
        "Q_start_Ah": float(q[0]),
        "Q_end_Ah": float(q[-1]),
        "Q_min_Ah": float(np.nanmin(q)),
        "Q_max_Ah": float(np.nanmax(q)),
        "n_negative_dQ_steps": int(np.sum(dQ < -1e-9)),
        "min_dQ_Ah": float(np.nanmin(dQ)) if len(dQ) else np.nan,
        "I_min_A": float(np.nanmin(i)),
        "I_max_A": float(np.nanmax(i)),
        "has_positive_discharge_current": bool(np.nanmax(i) > 0),
        "Q_monotonic_required": False if arm == "DCAC_full_protocol" else True,
    }


# ---------------------------------------------------------------------
# Main decomposition
# ---------------------------------------------------------------------

dtq_rows = []
seg_summary_rows = []
anchor_rows = []
qdiag_rows = []

for param_set in PARAM_SETS:
    print(f"\n[{param_set}] processing")

    pair_sub = pair_df[pair_df["param_set"].eq(param_set)]
    assert len(pair_sub) == 1, f"{param_set}: expected one pair summary row, got {len(pair_sub)}"
    pair = pair_sub.iloc[0]

    dc = (
        traj[(traj["param_set"].eq(param_set)) & (traj["arm"].eq("DC_reference"))]
        .copy()
        .sort_values("t_s")
        .reset_index(drop=True)
    )
    dcac = (
        traj[(traj["param_set"].eq(param_set)) & (traj["arm"].eq("DCAC_full_protocol"))]
        .copy()
        .sort_values("t_s")
        .reset_index(drop=True)
    )

    assert len(dc) > 0, f"{param_set}: no DC_reference rows"
    assert len(dcac) > 0, f"{param_set}: no DCAC_full_protocol rows"

    for name, df in [("DC_reference", dc), ("DCAC_full_protocol", dcac)]:
        t = df["t_s"].to_numpy(dtype=float)
        assert np.all(np.diff(t) >= 0), f"{param_set}/{name}: time not monotone"
        qdiag_rows.append(q_monotonicity_diag(param_set, name, df))

    Q_end_DC = float(dc["Q_Ah"].iloc[-1])
    Q_end_DCAC = float(dcac["Q_Ah"].iloc[-1])
    Q_common_max = min(Q_end_DC, Q_end_DCAC)

    Q_DC_vmax = float(pair["Q_to_Vmax_DC_Ah"])
    Q_DCAC_vmax = float(pair["Q_to_Vmax_DCAC_Ah"])

    Q_lo = max(0.05 * Q_common_max, 0.05)
    Q_hi = Q_common_max

    Q_grid = np.linspace(Q_lo, Q_hi, N_Q)

    print(f"  Q_common_max = {Q_common_max:.6f} Ah")
    print(f"  Q_DCAC,Vmax  = {Q_DCAC_vmax:.6f} Ah")
    print(f"  Q_DC,Vmax    = {Q_DC_vmax:.6f} Ah")
    print(f"  Q grid       = [{Q_lo:.6f}, {Q_hi:.6f}] Ah, n={N_Q}")

    # -----------------------------------------------------------------
    # Δt(Q) curves
    # -----------------------------------------------------------------

    for Q in Q_grid:
        dc_fp = first_passage_time_by_Q(
            dc,
            Q,
            value_cols=["V_V", "I_A", "stage"],
        )
        dcac_fp = first_passage_time_by_Q(
            dcac,
            Q,
            value_cols=["V_V", "I_A", "stage"],
        )

        found = bool(dc_fp["found"] and dcac_fp["found"])

        if found:
            t_dc = float(dc_fp["t_s"])
            t_dcac = float(dcac_fp["t_s"])
            dt = t_dc - t_dcac
        else:
            t_dc = np.nan
            t_dcac = np.nan
            dt = np.nan

        seg_label = classify_segment(Q, Q_DCAC_vmax, Q_DC_vmax)

        dtq_rows.append({
            "param_set": param_set,
            "Q_Ah": float(Q),
            "Q_frac_common": float(Q / Q_common_max),
            "Q_frac_nominal_5Ah": float(Q / 5.0),

            "segment": seg_label,
            "segment_interpretation": segment_interpretation(seg_label),

            "t_DC_s": t_dc,
            "t_DCAC_s": t_dcac,
            "dt_full_s": float(dt),
            "dt_full_min": float(dt / 60.0) if np.isfinite(dt) else np.nan,

            "V_DC_V": dc_fp["V_V"],
            "I_DC_A": dc_fp["I_A"],
            "stage_DC": dc_fp["stage"],

            "V_DCAC_V": dcac_fp["V_V"],
            "I_DCAC_A": dcac_fp["I_A"],
            "stage_DCAC": dcac_fp["stage"],

            "Q_to_Vmax_DC_Ah": Q_DC_vmax,
            "Q_to_Vmax_DCAC_Ah": Q_DCAC_vmax,
            "Q_common_max_Ah": Q_common_max,

            "DCAC_has_reached_Vmax": bool(Q >= Q_DCAC_vmax),
            "DC_has_reached_Vmax": bool(Q >= Q_DC_vmax),

            "found_both": found,
        })

    # -----------------------------------------------------------------
    # Anchor points
    # -----------------------------------------------------------------

    for label, Q in [
        ("Q80_common", 0.80 * Q_common_max),
        ("Q90_common", 0.90 * Q_common_max),
        ("Q80_nominal_5Ah", 0.80 * 5.0),
        ("Q90_nominal_5Ah", 0.90 * 5.0),
    ]:
        if Q <= Q_common_max + 1e-12:
            dc_fp = first_passage_time_by_Q(dc, Q, value_cols=["V_V", "I_A", "stage"])
            dcac_fp = first_passage_time_by_Q(dcac, Q, value_cols=["V_V", "I_A", "stage"])

            dt = (
                float(dc_fp["t_s"] - dcac_fp["t_s"])
                if dc_fp["found"] and dcac_fp["found"]
                else np.nan
            )

            seg_label = classify_segment(Q, Q_DCAC_vmax, Q_DC_vmax)

            anchor_rows.append({
                "param_set": param_set,
                "anchor_label": label,
                "Q_Ah": float(Q),
                "Q_frac_common": float(Q / Q_common_max),
                "Q_frac_nominal_5Ah": float(Q / 5.0),
                "segment": seg_label,
                "t_DC_s": dc_fp["t_s"],
                "t_DCAC_s": dcac_fp["t_s"],
                "dt_full_s": dt,
                "dt_full_min": dt / 60.0 if np.isfinite(dt) else np.nan,
                "stage_DC": dc_fp["stage"],
                "stage_DCAC": dcac_fp["stage"],
                "V_DC_V": dc_fp["V_V"],
                "V_DCAC_V": dcac_fp["V_V"],
                "I_DC_A": dc_fp["I_A"],
                "I_DCAC_A": dcac_fp["I_A"],
            })

# Convert curve rows to DataFrame before segment summaries
dtq_df = pd.DataFrame(dtq_rows)

# ---------------------------------------------------------------------
# Segment summaries
# ---------------------------------------------------------------------

for (param_set, seg_label), g in dtq_df.groupby(["param_set", "segment"], sort=False):
    dt = g["dt_full_s"].to_numpy(dtype=float)
    finite = np.isfinite(dt)

    if not finite.any():
        continue

    seg_summary_rows.append({
        "param_set": param_set,
        "segment": seg_label,
        "segment_interpretation": segment_interpretation(seg_label),
        "n_points": int(len(g)),
        "n_finite": int(finite.sum()),

        "Q_lo_Ah": float(g["Q_Ah"].min()),
        "Q_hi_Ah": float(g["Q_Ah"].max()),

        "dt_full_mean_s": float(np.nanmean(dt)),
        "dt_full_median_s": float(np.nanmedian(dt)),
        "dt_full_min_s": float(np.nanmin(dt)),
        "dt_full_max_s": float(np.nanmax(dt)),
        "dt_full_mean_min": float(np.nanmean(dt) / 60.0),
        "dt_full_median_min": float(np.nanmedian(dt) / 60.0),
        "dt_full_min_min": float(np.nanmin(dt) / 60.0),
        "dt_full_max_min": float(np.nanmax(dt) / 60.0),

        "stage_DC_modes": "; ".join(g["stage_DC"].value_counts().index.astype(str).tolist()),
        "stage_DCAC_modes": "; ".join(g["stage_DCAC"].value_counts().index.astype(str).tolist()),
        "V_DC_median_V": float(np.nanmedian(g["V_DC_V"])),
        "V_DCAC_median_V": float(np.nanmedian(g["V_DCAC_V"])),
        "I_DC_median_A": float(np.nanmedian(g["I_DC_A"])),
        "I_DCAC_median_A": float(np.nanmedian(g["I_DCAC_A"])),
    })

seg_summary_df = pd.DataFrame(seg_summary_rows)
anchors_df = pd.DataFrame(anchor_rows)
qdiag_df = pd.DataFrame(qdiag_rows)


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_curves = DATA / "day20B_step5_full_protocol_batch_dtQ_segments.csv"
out_summary = DATA / "day20B_step5_full_protocol_batch_segment_summary.csv"
out_anchors = DATA / "day20B_step5_full_protocol_batch_anchor_points.csv"
out_qdiag = DATA / "day20B_step5_full_protocol_batch_Q_monotonicity_diagnostic.csv"

dtq_df.to_csv(out_curves, index=False)
seg_summary_df.to_csv(out_summary, index=False)
anchors_df.to_csv(out_anchors, index=False)
qdiag_df.to_csv(out_qdiag, index=False)

print(f"\nWrote: {out_curves} ({len(dtq_df)} rows)")
print(f"Wrote: {out_summary} ({len(seg_summary_df)} rows)")
print(f"Wrote: {out_anchors} ({len(anchors_df)} rows)")
print(f"Wrote: {out_qdiag} ({len(qdiag_df)} rows)")

print("\nSegment summary:")
display(seg_summary_df)

print("\nAnchor points:")
display(anchors_df)

print("\nQ monotonicity diagnostics:")
display(qdiag_df)


# ---------------------------------------------------------------------
# Compact diagnostic tables
# ---------------------------------------------------------------------

print("\nMedian Δt by segment [min]:")
display(
    seg_summary_df.pivot(
        index="param_set",
        columns="segment",
        values="dt_full_median_min",
    )
)

print("\nAnchor Δt [min]:")
display(
    anchors_df.pivot(
        index="param_set",
        columns="anchor_label",
        values="dt_full_min",
    )
)


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert dtq_df["found_both"].all(), "Some Q targets not reached by both arms."

for param_set in PARAM_SETS:
    sub = dtq_df[dtq_df["param_set"].eq(param_set)]
    segs = set(sub["segment"])
    assert "A_pre_DCAC_Vmax" in segs, f"{param_set}: missing Segment A"
    assert "B_between_DCAC_Vmax_and_DC_Vmax" in segs, f"{param_set}: missing Segment B"
    assert "D_late_CV_feedback_region" in segs, f"{param_set}: missing Segment D"

assert (seg_summary_df["dt_full_median_min"] > 0).all(), (
    "Expected positive median Δt in all segments and parameter sets."
)

dcac_diag = qdiag_df[qdiag_df["arm"].eq("DCAC_full_protocol")]
assert (dcac_diag["n_negative_dQ_steps"] > 0).all(), (
    "Expected non-monotone strict-net Q in all DCAC arms due to AC reversal."
)

print("\n" + "=" * 72)
print("Cell 13 PASSED — full-protocol batch Δt(Q) segment decomposition complete")
print("=" * 72)

Cell 13 — full-protocol batch Δt(Q) segment decomposition
Trajectory rows: 10382
Pair rows      : 3
Arm summaries  : 6


[Chen2020] processing
  Q_common_max = 4.872588 Ah
  Q_DCAC,Vmax  = 3.770222 Ah
  Q_DC,Vmax    = 4.627667 Ah
  Q grid       = [0.243629, 4.872588] Ah, n=120

[OKane2022] processing
  Q_common_max = 4.835569 Ah
  Q_DCAC,Vmax  = 3.790496 Ah
  Q_DC,Vmax    = 4.596815 Ah
  Q grid       = [0.241778, 4.835569] Ah, n=120

[ORegan2022] processing
  Q_common_max = 4.792207 Ah
  Q_DCAC,Vmax  = 3.210362 Ah
  Q_DC,Vmax    = 4.724016 Ah
  Q grid       = [0.239610, 4.792207] Ah, n=120

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step5_full_protocol_batch_dtQ_segments.csv (360 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step5_full_protocol_batch_segment_summary.csv (9 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step5_full_protocol_batch_anchor_points.csv (12 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_ste

,param_set,segment,segment_interpretation,n_points,n_finite,Q_lo_Ah,Q_hi_Ah,dt_full_mean_s,dt_full_median_s,dt_full_min_s,...,dt_full_mean_min,dt_full_median_min,dt_full_min_min,dt_full_max_min,stage_DC_modes,stage_DCAC_modes,V_DC_median_V,V_DCAC_median_V,I_DC_median_A,I_DCAC_median_A
0,Chen2020,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,91,91,0.243629,3.744523,1262.728299,1266.180112,466.605845,...,21.045472,21.103002,7.776764,32.537896,CC_until_Vmax,CC_until_Vmax,3.751356,3.853670,-1.000000,-3.243297
1,Chen2020,B_between_DCAC_Vmax_and_DC_Vmax,DCAC already voltage-limited while DC still ap...,22,22,3.783422,4.600297,1687.112922,1752.128618,807.030333,...,28.118549,29.202144,13.450506,38.331119,CC_until_Vmax,CV_after_Vmax,4.136063,4.200000,-1.000000,-2.377669
2,Chen2020,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,7,7,4.639196,4.872588,2260.960726,2259.923602,2235.360151,...,37.682679,37.665393,37.256003,38.130891,CV_after_Vmax,CV_after_Vmax,4.200000,4.200000,-0.462265,-0.453075
3,OKane2022,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,92,92,0.241778,3.754677,1278.953706,1312.930143,436.486822,...,21.315895,21.882169,7.274780,33.338440,CC_until_Vmax,CC_until_Vmax,3.756931,3.870245,-1.000000,-3.236450
4,OKane2022,B_between_DCAC_Vmax_and_DC_Vmax,DCAC already voltage-limited while DC still ap...,21,21,3.793280,4.565346,1422.714705,1483.810642,606.687042,...,23.711912,24.730177,10.111451,33.221490,CC_until_Vmax,CV_after_Vmax,4.137540,4.200000,-1.000000,-2.312447
5,OKane2022,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,7,7,4.603949,4.835569,1961.644309,1961.445579,1946.558122,...,32.694072,32.690760,32.442635,33.046965,CV_after_Vmax,CV_after_Vmax,4.200000,4.200000,-0.472976,-0.466959
6,ORegan2022,A_pre_DCAC_Vmax,shared pre-Vmax region; Segment A geometry-dom...,78,78,0.239610,3.185408,1465.645296,1438.136870,439.310746,...,24.427422,23.968948,7.321846,39.768637,CC_until_Vmax,CC_until_Vmax,3.651030,3.736384,-1.000000,-3.281273
7,ORegan2022,B_between_DCAC_Vmax_and_DC_Vmax,DCAC already voltage-limited while DC still ap...,40,40,3.223666,4.715693,3425.046416,3520.355054,1663.072303,...,57.084107,58.672584,27.717872,79.653035,CC_until_Vmax,CV_after_Vmax,4.099161,4.400000,-1.000000,-2.658104
8,ORegan2022,D_late_CV_feedback_region,late post-DC-Vmax region; both branches have r...,2,2,4.753950,4.792207,4944.376036,4944.376036,4827.295275,...,82.406267,82.406267,80.454921,84.357613,CV_after_Vmax,CV_after_Vmax,4.400000,4.400000,-0.291348,-0.382139



Anchor points:


,param_set,anchor_label,Q_Ah,Q_frac_common,Q_frac_nominal_5Ah,segment,t_DC_s,t_DCAC_s,dt_full_s,dt_full_min,stage_DC,stage_DCAC,V_DC_V,V_DCAC_V,I_DC_A,I_DCAC_A
0,Chen2020,Q80_common,3.898071,0.800000,0.779614,B_between_DCAC_Vmax_and_DC_Vmax,14033.054680,12944.723636,1088.331044,18.138851,CC_until_Vmax,CV_after_Vmax,4.108356,4.2,-1.0,-2.986573
1,Chen2020,Q90_common,4.385330,0.900000,0.877066,B_between_DCAC_Vmax_and_DC_Vmax,15787.186515,13676.462135,2110.724381,35.178740,CC_until_Vmax,CV_after_Vmax,4.149006,4.2,-1.0,-1.745372
2,Chen2020,Q80_nominal_5Ah,4.000000,0.820919,0.800000,B_between_DCAC_Vmax_and_DC_Vmax,14400.000000,13071.972574,1328.027426,22.133790,CC_until_Vmax,CV_after_Vmax,4.121730,4.2,-1.0,-2.789010
3,Chen2020,Q90_nominal_5Ah,4.500000,0.923534,0.900000,B_between_DCAC_Vmax_and_DC_Vmax,16200.000000,13946.289763,2253.710237,37.561837,CC_until_Vmax,CV_after_Vmax,4.166696,4.2,-1.0,-1.333211
4,OKane2022,Q80_common,3.868455,0.800000,0.773691,B_between_DCAC_Vmax_and_DC_Vmax,13926.438334,13136.788514,789.649820,13.160830,CC_until_Vmax,CV_after_Vmax,4.107505,4.2,-1.0,-2.958882
5,OKane2022,Q90_common,4.352012,0.900000,0.870402,B_between_DCAC_Vmax_and_DC_Vmax,15667.243126,13867.831425,1799.411700,29.990195,CC_until_Vmax,CV_after_Vmax,4.148909,4.2,-1.0,-1.749168
6,OKane2022,Q80_nominal_5Ah,4.000000,0.827204,0.800000,B_between_DCAC_Vmax_and_DC_Vmax,14400.000000,13305.020590,1094.979410,18.249657,CC_until_Vmax,CV_after_Vmax,4.124606,4.2,-1.0,-2.686148
7,OKane2022,Q90_nominal_5Ah,4.500000,0.930604,0.900000,B_between_DCAC_Vmax_and_DC_Vmax,16200.000000,14229.237423,1970.762577,32.846043,CC_until_Vmax,CV_after_Vmax,4.173334,4.2,-1.0,-1.225423
8,ORegan2022,Q80_common,3.833766,0.800000,0.766753,B_between_DCAC_Vmax_and_DC_Vmax,13801.557411,10608.141370,3193.416041,53.223601,CC_until_Vmax,CV_after_Vmax,4.068258,4.4,-1.0,-4.905090
9,ORegan2022,Q90_common,4.312987,0.900000,0.862597,B_between_DCAC_Vmax_and_DC_Vmax,15526.752087,11231.565382,4295.186706,71.586445,CC_until_Vmax,CV_after_Vmax,4.144442,4.4,-1.0,-1.989672



Q monotonicity diagnostics:


...


Median Δt by segment [min]:


segment,A_pre_DCAC_Vmax,B_between_DCAC_Vmax_and_DC_Vmax,D_late_CV_feedback_region
param_set,,,
Chen2020,21.103002,29.202144,37.665393
OKane2022,21.882169,24.730177,32.690760
ORegan2022,23.968948,58.672584,82.406267



Anchor Δt [min]:


anchor_label,Q80_common,Q80_nominal_5Ah,Q90_common,Q90_nominal_5Ah
param_set,,,,
Chen2020,18.138851,22.133790,35.178740,37.561837
OKane2022,13.160830,18.249657,29.990195,32.846043
ORegan2022,53.223601,60.022519,71.586445,77.179788



Cell 13 PASSED — full-protocol batch Δt(Q) segment decomposition complete


In [26]:
# Cell 14 — Day20B batch verdict + current C-rate audit
#
# Purpose:
#   Close the Day20B full-protocol batch at verdict level.
#
# Inputs:
#   data/day20B_step0_full_protocol_design.csv
#   data/day20B_step4_full_protocol_batch_trajectories.csv
#   data/day20B_step4_full_protocol_batch_summary.csv
#   data/day20B_step4_full_protocol_batch_pair_summary.csv
#   data/day20B_step5_full_protocol_batch_segment_summary.csv
#   data/day20B_step5_full_protocol_batch_anchor_points.csv
#   data/day20B_step5_full_protocol_batch_Q_monotonicity_diagnostic.csv
#
# Outputs:
#   data/day20B_step6_full_protocol_current_audit.csv
#   data/day20B_step6_full_protocol_batch_verdict.csv
#   docs/day20B_full_protocol_batch_verdict.md
#
# Key audit:
#   - Convert I_A to I_C = I_A / Q_nom_Ah.
#   - Verify CC drive-cycle current stays within design peak C-rate.
#   - Flag CV current transients separately.
#   - Preserve mechanism boundary:
#       full-protocol raw gain supported;
#       non-geometric Segment-A acceleration not supported;
#       dominant interpretation = voltage-boundary / control-state split.

import numpy as np
import pandas as pd
from pathlib import Path

DESIGN = DATA / "day20B_step0_full_protocol_design.csv"
TRAJ = DATA / "day20B_step4_full_protocol_batch_trajectories.csv"
ARM_SUMMARY = DATA / "day20B_step4_full_protocol_batch_summary.csv"
PAIR = DATA / "day20B_step4_full_protocol_batch_pair_summary.csv"
SEG_SUMMARY = DATA / "day20B_step5_full_protocol_batch_segment_summary.csv"
ANCHORS = DATA / "day20B_step5_full_protocol_batch_anchor_points.csv"
QDIAG = DATA / "day20B_step5_full_protocol_batch_Q_monotonicity_diagnostic.csv"

for p in [DESIGN, TRAJ, ARM_SUMMARY, PAIR, SEG_SUMMARY, ANCHORS, QDIAG]:
    assert p.exists(), f"Missing required file: {p}"

design = pd.read_csv(DESIGN)
traj = pd.read_csv(TRAJ)
arm_summary = pd.read_csv(ARM_SUMMARY)
pair = pd.read_csv(PAIR)
seg = pd.read_csv(SEG_SUMMARY)
anchors = pd.read_csv(ANCHORS)
qdiag = pd.read_csv(QDIAG)

PARAM_SETS = ["Chen2020", "OKane2022", "ORegan2022"]

print("=" * 72)
print("Cell 14 — Day20B batch verdict + current C-rate audit")
print("=" * 72)
print(f"Trajectory rows : {len(traj)}")
print(f"Pair rows       : {len(pair)}")
print(f"Segment rows    : {len(seg)}")
print(f"Anchor rows     : {len(anchors)}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def ffloat(x):
    if pd.isna(x):
        return np.nan
    return float(x)


def fmt(x, nd=3):
    if pd.isna(x):
        return "nan"
    return f"{float(x):.{nd}f}"


def to_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return bool(x)


def classify_evidence(cv_peak_charge_C, full_protocol_ok):
    """
    Evidence-level classification.

    A CV charge-current peak above 1C is not a protocol phase error.
    It is a CV feedback transient and should be treated as a warning.
    """
    if not full_protocol_ok:
        return "invalid_or_incomplete"

    if np.isfinite(cv_peak_charge_C) and cv_peak_charge_C > 1.0:
        return "valid_with_CV_current_transient_warning"

    return "valid"


def get_segment_row(param_set, segment):
    out = seg[(seg["param_set"].eq(param_set)) & (seg["segment"].eq(segment))]
    assert len(out) == 1, f"Expected one segment row for {param_set}/{segment}, got {len(out)}"
    return out.iloc[0]


# ---------------------------------------------------------------------
# Current C-rate audit
# ---------------------------------------------------------------------

design_meta = design[
    [
        "param_set", "arm", "Q_nom_Ah",
        "DC_C", "AC_C", "peak_charge_C", "peak_discharge_C",
        "I_DC_A", "I_AC_A", "I_cutoff_A", "I_cutoff_mA",
        "Vmax_cutoff_V", "f_Hz", "T_period_s",
    ]
].drop_duplicates(["param_set", "arm"])

traj_meta = traj.merge(
    design_meta,
    on=["param_set", "arm"],
    how="left",
    validate="many_to_one",
)

assert traj_meta["Q_nom_Ah"].notna().all(), "Failed to join Q_nom_Ah onto trajectories."

traj_meta["I_C"] = traj_meta["I_A"] / traj_meta["Q_nom_Ah"]
traj_meta["charge_current_C"] = np.maximum(-traj_meta["I_C"], 0.0)
traj_meta["discharge_current_C"] = np.maximum(traj_meta["I_C"], 0.0)

current_rows = []

for (param_set, arm, stage), g in traj_meta.groupby(["param_set", "arm", "stage"], sort=False):
    row0 = g.iloc[0]

    I_C = g["I_C"].to_numpy(dtype=float)
    charge_C = g["charge_current_C"].to_numpy(dtype=float)
    discharge_C = g["discharge_current_C"].to_numpy(dtype=float)

    expected_peak_charge_C = ffloat(row0["peak_charge_C"])
    expected_peak_discharge_C = ffloat(row0["peak_discharge_C"])

    charge_peak_C = float(np.nanmax(charge_C))
    discharge_peak_C = float(np.nanmax(discharge_C))

    # CC stage should match prescribed design.
    if stage == "CC_until_Vmax":
        cc_peak_charge_ok = charge_peak_C <= expected_peak_charge_C + 0.02
        cc_peak_discharge_ok = discharge_peak_C <= expected_peak_discharge_C + 0.02
    else:
        cc_peak_charge_ok = np.nan
        cc_peak_discharge_ok = np.nan

    # CV stage is a feedback stage; current can transiently overshoot.
    cv_charge_transient_warning = (
        stage == "CV_after_Vmax"
        and arm == "DCAC_full_protocol"
        and charge_peak_C > 1.0
    )

    cv_charge_above_cc_peak = (
        stage == "CV_after_Vmax"
        and charge_peak_C > expected_peak_charge_C + 0.05
    )

    current_rows.append({
        "param_set": param_set,
        "arm": arm,
        "stage": stage,
        "n_samples": len(g),

        "Q_nom_Ah": ffloat(row0["Q_nom_Ah"]),
        "DC_C": ffloat(row0["DC_C"]),
        "AC_C": ffloat(row0["AC_C"]),
        "expected_peak_charge_C": expected_peak_charge_C,
        "expected_peak_discharge_C": expected_peak_discharge_C,

        "I_A_min": float(np.nanmin(g["I_A"])),
        "I_A_max": float(np.nanmax(g["I_A"])),
        "I_C_min": float(np.nanmin(I_C)),
        "I_C_max": float(np.nanmax(I_C)),
        "charge_peak_C": charge_peak_C,
        "discharge_peak_C": discharge_peak_C,

        "cc_peak_charge_ok": cc_peak_charge_ok,
        "cc_peak_discharge_ok": cc_peak_discharge_ok,
        "cv_charge_transient_warning": bool(cv_charge_transient_warning),
        "cv_charge_above_cc_peak": bool(cv_charge_above_cc_peak),

        "interpretation": (
            "CC current matches design"
            if stage == "CC_until_Vmax" and bool(cc_peak_charge_ok) and bool(cc_peak_discharge_ok)
            else "CV feedback current transient"
            if stage == "CV_after_Vmax" and bool(cv_charge_transient_warning)
            else "CV feedback current within expected range"
            if stage == "CV_after_Vmax"
            else "requires_review"
        ),
    })

current_audit = pd.DataFrame(current_rows)

out_current = DATA / "day20B_step6_full_protocol_current_audit.csv"
current_audit.to_csv(out_current, index=False)

print(f"\nWrote: {out_current} ({len(current_audit)} rows)")
display(current_audit)


# ---------------------------------------------------------------------
# Batch verdict table
# ---------------------------------------------------------------------

verdict_rows = []

for param_set in PARAM_SETS:
    p = pair[pair["param_set"].eq(param_set)]
    assert len(p) == 1, f"Expected one pair row for {param_set}, got {len(p)}"
    p = p.iloc[0]

    seg_A = get_segment_row(param_set, "A_pre_DCAC_Vmax")
    seg_B = get_segment_row(param_set, "B_between_DCAC_Vmax_and_DC_Vmax")
    seg_D = get_segment_row(param_set, "D_late_CV_feedback_region")

    a = anchors[anchors["param_set"].eq(param_set)]

    q80_common = a[a["anchor_label"].eq("Q80_common")].iloc[0]
    q90_common = a[a["anchor_label"].eq("Q90_common")].iloc[0]
    q80_nom = a[a["anchor_label"].eq("Q80_nominal_5Ah")].iloc[0]
    q90_nom = a[a["anchor_label"].eq("Q90_nominal_5Ah")].iloc[0]

    qdiag_d = qdiag[(qdiag["param_set"].eq(param_set)) & (qdiag["arm"].eq("DC_reference"))].iloc[0]
    qdiag_a = qdiag[(qdiag["param_set"].eq(param_set)) & (qdiag["arm"].eq("DCAC_full_protocol"))].iloc[0]

    dcac_cv_current = current_audit[
        (current_audit["param_set"].eq(param_set))
        & (current_audit["arm"].eq("DCAC_full_protocol"))
        & (current_audit["stage"].eq("CV_after_Vmax"))
    ]
    assert len(dcac_cv_current) == 1, f"Missing DCAC CV current audit for {param_set}"
    dcac_cv_current = dcac_cv_current.iloc[0]

    dcac_cc_current = current_audit[
        (current_audit["param_set"].eq(param_set))
        & (current_audit["arm"].eq("DCAC_full_protocol"))
        & (current_audit["stage"].eq("CC_until_Vmax"))
    ]
    assert len(dcac_cc_current) == 1, f"Missing DCAC CC current audit for {param_set}"
    dcac_cc_current = dcac_cc_current.iloc[0]

    full_protocol_ok = (
        str(p["pair_status"]) == "ok"
        and to_bool(p["DC_cutoff_reached"])
        and to_bool(p["DCAC_cutoff_reached"])
        and to_bool(p["DC_has_cv_tail"])
        and to_bool(p["DCAC_has_cv_tail"])
        and ffloat(p["DC_Vmax_violation_max_over_V"]) < 0.02
        and ffloat(p["DCAC_Vmax_violation_max_over_V"]) < 0.02
    )

    anchors_common_in_B = (
        q80_common["segment"] == "B_between_DCAC_Vmax_and_DC_Vmax"
        and q90_common["segment"] == "B_between_DCAC_Vmax_and_DC_Vmax"
    )

    anchors_nominal_in_B = (
        q80_nom["segment"] == "B_between_DCAC_Vmax_and_DC_Vmax"
        and q90_nom["segment"] == "B_between_DCAC_Vmax_and_DC_Vmax"
    )

    cv_peak_charge_C = ffloat(dcac_cv_current["charge_peak_C"])
    evidence_status = classify_evidence(cv_peak_charge_C, full_protocol_ok)

    verdict_rows.append({
        "param_set": param_set,
        "full_protocol_ok": bool(full_protocol_ok),
        "evidence_status": evidence_status,

        "dt_total_min": ffloat(p["dt_total_min"]),
        "dt_total_s": ffloat(p["dt_total_s"]),

        "Q_end_DC_Ah": ffloat(p["Q_end_DC_Ah"]),
        "Q_end_DCAC_Ah": ffloat(p["Q_end_DCAC_Ah"]),
        "Q_end_diff_Ah": ffloat(p["Q_end_diff_Ah"]),

        "Q_to_Vmax_DC_Ah": ffloat(p["Q_to_Vmax_DC_Ah"]),
        "Q_to_Vmax_DCAC_Ah": ffloat(p["Q_to_Vmax_DCAC_Ah"]),
        "Q_to_Vmax_shift_Ah": ffloat(p["Q_to_Vmax_shift_Ah"]),
        "dt_to_Vmax_s": ffloat(p["dt_to_Vmax_s"]),

        "segment_A_median_dt_min": ffloat(seg_A["dt_full_median_min"]),
        "segment_B_median_dt_min": ffloat(seg_B["dt_full_median_min"]),
        "segment_D_median_dt_min": ffloat(seg_D["dt_full_median_min"]),

        "Q80_common_dt_min": ffloat(q80_common["dt_full_min"]),
        "Q90_common_dt_min": ffloat(q90_common["dt_full_min"]),
        "Q80_nominal_5Ah_dt_min": ffloat(q80_nom["dt_full_min"]),
        "Q90_nominal_5Ah_dt_min": ffloat(q90_nom["dt_full_min"]),

        "anchors_Q80_Q90_common_in_segment_B": bool(anchors_common_in_B),
        "anchors_Q80_Q90_nominal_in_segment_B": bool(anchors_nominal_in_B),

        "DCAC_CC_charge_peak_C": ffloat(dcac_cc_current["charge_peak_C"]),
        "DCAC_CC_discharge_peak_C": ffloat(dcac_cc_current["discharge_peak_C"]),
        "DCAC_CC_peak_charge_ok": to_bool(dcac_cc_current["cc_peak_charge_ok"]),
        "DCAC_CC_peak_discharge_ok": to_bool(dcac_cc_current["cc_peak_discharge_ok"]),

        "DCAC_CV_charge_peak_C": cv_peak_charge_C,
        "DCAC_CV_discharge_peak_C": ffloat(dcac_cv_current["discharge_peak_C"]),
        "DCAC_CV_current_transient_warning": to_bool(dcac_cv_current["cv_charge_transient_warning"]),
        "DCAC_CV_charge_above_CC_peak": to_bool(dcac_cv_current["cv_charge_above_cc_peak"]),

        "DC_Q_negative_steps": int(qdiag_d["n_negative_dQ_steps"]),
        "DCAC_Q_negative_steps": int(qdiag_a["n_negative_dQ_steps"]),
        "DCAC_has_positive_discharge_current": to_bool(qdiag_a["has_positive_discharge_current"]),

        "mechanism_verdict": (
            "positive_full_protocol_raw_gain_boundary_control_state_split_not_Segment_A_residual"
        ),
        "interpretation": (
            "Full-protocol raw gain is positive and persists into late CV. "
            "Q80/Q90 anchors lie in Segment B, where DCAC is already voltage-limited "
            "while DC remains in CC. This supports voltage-boundary/control-state split, "
            "not non-geometric Segment-A acceleration."
        ),
    })

verdict = pd.DataFrame(verdict_rows)

out_verdict = DATA / "day20B_step6_full_protocol_batch_verdict.csv"
verdict.to_csv(out_verdict, index=False)

print(f"\nWrote: {out_verdict} ({len(verdict)} rows)")
display(verdict)


# ---------------------------------------------------------------------
# Markdown documentation
# ---------------------------------------------------------------------

DOCS.mkdir(exist_ok=True)
out_doc = DOCS / "day20B_full_protocol_batch_verdict.md"

lines = []
lines.append("# Day20B — Full-protocol PyBaMM batch verdict")
lines.append("")
lines.append("Status: batch audit closed")
lines.append("")
lines.append("## Scope")
lines.append("")
lines.append("This document summarizes the full CC+CV PyBaMM simulations for:")
lines.append("")
lines.append("```text")
for ps in PARAM_SETS:
    lines.append(ps)
lines.append("```")
lines.append("")
lines.append("Protocol logic:")
lines.append("")
lines.append("```text")
lines.append("DC reference:")
lines.append("  CC at -0.2C until Vmax")
lines.append("  CV at Vmax until |I| <= normalized cutoff")
lines.append("")
lines.append("DCAC:")
lines.append("  charge-first DC+AC during CC")
lines.append("  AC off at Vmax")
lines.append("  pure DC-CV at Vmax until |I| <= normalized cutoff")
lines.append("```")
lines.append("")
lines.append("The cutoff current is normalized:")
lines.append("")
lines.append("```text")
lines.append("I_cutoff = (0.05 / 3.4) · Q_nom")
lines.append("```")
lines.append("")
lines.append("## Full-protocol result")
lines.append("")
lines.append("| param_set | evidence status | total Δt [min] | Q_to_Vmax shift [Ah] | Segment A median [min] | Segment B median [min] | Segment D median [min] |")
lines.append("|---|---|---:|---:|---:|---:|---:|")

for _, r in verdict.iterrows():
    lines.append(
        f"| {r['param_set']} | {r['evidence_status']} | "
        f"{fmt(r['dt_total_min'], 3)} | "
        f"{fmt(r['Q_to_Vmax_shift_Ah'], 6)} | "
        f"{fmt(r['segment_A_median_dt_min'], 3)} | "
        f"{fmt(r['segment_B_median_dt_min'], 3)} | "
        f"{fmt(r['segment_D_median_dt_min'], 3)} |"
    )

lines.append("")
lines.append("## Anchor points")
lines.append("")
lines.append("| param_set | Q80 common [min] | Q90 common [min] | Q80 nominal [min] | Q90 nominal [min] | Q80/Q90 in Segment B |")
lines.append("|---|---:|---:|---:|---:|---|")

for _, r in verdict.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{fmt(r['Q80_common_dt_min'], 3)} | "
        f"{fmt(r['Q90_common_dt_min'], 3)} | "
        f"{fmt(r['Q80_nominal_5Ah_dt_min'], 3)} | "
        f"{fmt(r['Q90_nominal_5Ah_dt_min'], 3)} | "
        f"{r['anchors_Q80_Q90_common_in_segment_B']} |"
    )

lines.append("")
lines.append("## Current C-rate audit")
lines.append("")
lines.append("| param_set | CC charge peak [C] | CC discharge peak [C] | CC ok | CV charge peak [C] | CV transient warning |")
lines.append("|---|---:|---:|---|---:|---|")

for _, r in verdict.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{fmt(r['DCAC_CC_charge_peak_C'], 3)} | "
        f"{fmt(r['DCAC_CC_discharge_peak_C'], 3)} | "
        f"{r['DCAC_CC_peak_charge_ok'] and r['DCAC_CC_peak_discharge_ok']} | "
        f"{fmt(r['DCAC_CV_charge_peak_C'], 3)} | "
        f"{r['DCAC_CV_current_transient_warning']} |"
    )

lines.append("")
lines.append("## Interpretation")
lines.append("")
lines.append("All three full-protocol simulations show positive raw Δt(Q) and positive total full-protocol time gain.")
lines.append("")
lines.append("For all three parameter sets, Q80/Q90 anchors lie in Segment B:")
lines.append("")
lines.append("```text")
lines.append("DCAC is already voltage-limited / CV-controlled")
lines.append("DC remains in CC")
lines.append("```")
lines.append("")
lines.append("Therefore, the batch supports a voltage-boundary / control-state split explanation. It does not support a claim of non-geometric Segment-A acceleration.")
lines.append("")
lines.append("ORegan2022 shows a CV current transient above 1C. This is not a CC waveform phase error; the CC drive-cycle audit remains correct. ORegan2022 should be treated as valid with a CV-current-transient warning.")
lines.append("")
lines.append("## Final Day20B verdict")
lines.append("")
lines.append("```text")
lines.append("Full-protocol raw gain: supported across Chen2020, OKane2022, ORegan2022.")
lines.append("Segment-A non-geometric acceleration: not supported.")
lines.append("Dominant mechanism class: voltage-boundary / control-state split, with late-CV preservation of earlier advantage.")
lines.append("```")
lines.append("")

out_doc.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote: {out_doc}")


# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert (verdict["full_protocol_ok"]).all(), "Not all parameter sets passed full-protocol checks."

assert (verdict["dt_total_min"] > 0).all(), "Expected positive total full-protocol gain."

assert (verdict["anchors_Q80_Q90_common_in_segment_B"]).all(), (
    "Expected Q80/Q90 common anchors to lie in Segment B for all parameter sets."
)

assert (verdict["DCAC_CC_peak_charge_ok"]).all(), (
    "DCAC CC charge peak exceeds design bound."
)

assert (verdict["DCAC_CC_peak_discharge_ok"]).all(), (
    "DCAC CC discharge peak exceeds design bound."
)

assert (verdict["DCAC_Q_negative_steps"] > 0).all(), (
    "Expected DCAC strict-net Q to be non-monotone due to AC reversal."
)

print("\n" + "=" * 72)
print("Cell 14 PASSED — Day20B batch verdict + current audit complete")
print("=" * 72)

Cell 14 — Day20B batch verdict + current C-rate audit
Trajectory rows : 10382
Pair rows       : 3
Segment rows    : 9
Anchor rows     : 12

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step6_full_protocol_current_audit.csv (12 rows)


,param_set,arm,stage,n_samples,Q_nom_Ah,DC_C,AC_C,expected_peak_charge_C,expected_peak_discharge_C,I_A_min,I_A_max,I_C_min,I_C_max,charge_peak_C,discharge_peak_C,cc_peak_charge_ok,cc_peak_discharge_ok,cv_charge_transient_warning,cv_charge_above_cc_peak,interpretation
0,Chen2020,DC_reference,CC_until_Vmax,1668,5.0,0.2,0.0,0.2,0.0,-1.000270,-1.000000,-0.200054,-0.200000,0.200054,0.000000,True,True,False,False,CC current matches design
1,Chen2020,DC_reference,CV_after_Vmax,270,5.0,0.2,0.0,0.2,0.0,-0.979562,-0.073539,-0.195912,-0.014708,0.195912,0.000000,NaN,NaN,False,False,CV feedback current within expected range
2,Chen2020,DCAC_full_protocol,CC_until_Vmax,1282,5.0,0.2,0.5,0.7,0.3,-3.499999,1.500000,-0.700000,0.300000,0.700000,0.300000,True,True,False,False,CC current matches design
3,Chen2020,DCAC_full_protocol,CV_after_Vmax,431,5.0,0.2,0.5,0.7,0.3,-3.376611,-0.073539,-0.675322,-0.014708,0.675322,0.000000,NaN,NaN,False,False,CV feedback current within expected range
4,OKane2022,DC_reference,CC_until_Vmax,1657,5.0,0.2,0.0,0.2,0.0,-1.000265,-1.000000,-0.200053,-0.200000,0.200053,0.000000,True,True,False,False,CC current matches design
5,OKane2022,DC_reference,CV_after_Vmax,260,5.0,0.2,0.0,0.2,0.0,-0.979920,-0.073539,-0.195984,-0.014708,0.195984,0.000000,NaN,NaN,False,False,CV feedback current within expected range
6,OKane2022,DCAC_full_protocol,CC_until_Vmax,1307,5.0,0.2,0.5,0.7,0.3,-3.499994,1.500000,-0.699999,0.300000,0.699999,0.300000,True,True,False,False,CC current matches design
7,OKane2022,DCAC_full_protocol,CV_after_Vmax,419,5.0,0.2,0.5,0.7,0.3,-3.219155,-0.073539,-0.643831,-0.014708,0.643831,0.000000,NaN,NaN,False,False,CV feedback current within expected range
8,ORegan2022,DC_reference,CC_until_Vmax,1703,5.0,0.2,0.0,0.2,0.0,-1.000305,-1.000000,-0.200061,-0.200000,0.200061,0.000000,True,True,False,False,CC current matches design
9,ORegan2022,DC_reference,CV_after_Vmax,87,5.0,0.2,0.0,0.2,0.0,-0.709172,-0.069500,-0.141834,-0.013900,0.141834,0.000000,NaN,NaN,False,False,CV feedback current within expected range



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step6_full_protocol_batch_verdict.csv (3 rows)


,param_set,full_protocol_ok,evidence_status,dt_total_min,dt_total_s,Q_end_DC_Ah,Q_end_DCAC_Ah,Q_end_diff_Ah,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,...,DCAC_CC_peak_discharge_ok,DCAC_CV_charge_peak_C,DCAC_CV_discharge_peak_C,DCAC_CV_current_transient_warning,DCAC_CV_charge_above_CC_peak,DC_Q_negative_steps,DCAC_Q_negative_steps,DCAC_has_positive_discharge_current,mechanism_verdict,interpretation
0,Chen2020,True,valid,37.439446,2246.366759,4.872815,4.872588,0.000226,4.627667,3.770222,...,True,0.675322,0.0,False,False,0,453,True,positive_full_protocol_raw_gain_boundary_contr...,Full-protocol raw gain is positive and persist...
1,OKane2022,True,valid,31.820288,1909.217290,4.835569,4.836702,-0.001133,4.596815,3.790496,...,True,0.643831,0.0,False,False,0,464,True,positive_full_protocol_raw_gain_boundary_contr...,Full-protocol raw gain is positive and persist...
2,ORegan2022,True,valid_with_CV_current_transient_warning,82.269461,4936.167658,4.792207,4.796522,-0.004315,4.724016,3.210362,...,True,1.387381,0.0,True,True,0,332,True,positive_full_protocol_raw_gain_boundary_contr...,Full-protocol raw gain is positive and persist...


Wrote: /Users/louislu/pybamm-dcac-superimposed/docs/day20B_full_protocol_batch_verdict.md

Cell 14 PASSED — Day20B batch verdict + current audit complete


## Day20B closure — full-protocol batch verdict

Day20B implemented and audited a PyBaMM full CC+CV protocol that matches the MJ1 experimental control logic at the protocol level:

1. Charge-first DC+AC during CC  
2. AC off at the first voltage-limit event  
3. Pure DC-CV after the voltage boundary  
4. Termination at normalized cutoff current  

The cutoff current is not transferred as an absolute 50 mA value. It is normalized as

$$
I_{\mathrm{cutoff}}
=
\frac{0.05}{3.4}
\cdot
Q_{\mathrm{nom}}
$$

For the three audited 5 Ah parameter sets, this gives

$$
I_{\mathrm{cutoff}}
=
73.529 \ \mathrm{mA}
$$

## Batch result

The full-protocol batch was successfully completed for:

- Chen2020
- OKane2022
- ORegan2022

All three cases reached the normalized cutoff current and contained a valid CV tail.

| Parameter set | Full-protocol raw gain | Evidence status |
|---|---:|---|
| Chen2020 | 37.44 min | valid |
| OKane2022 | 31.82 min | valid |
| ORegan2022 | 82.27 min | valid with CV current transient warning |

## Current convention audit

The CC current convention is correct for all three DCAC cases:

$$
I_{\mathrm{py}}(t)
=
-I_{\mathrm{DC}}
-
I_{\mathrm{AC}}\sin(2\pi f t)
$$

This is the charge-first convention.

The CC-stage current bounds are:

- peak charge current: 0.7C
- peak discharge current: 0.3C

Thus, the Day20B batch does **not** repeat the historical discharge-first implementation mismatch from nb04–nb20.

ORegan2022 shows a CV-stage current transient above 1C. This is not a CC waveform or phase error; it is a voltage-feedback transient during CV. ORegan2022 is therefore retained as:

```text
valid_with_CV_current_transient_warning
```

## Segment interpretation

The batch confirms the Day20A/Day20B mechanism boundary:

- Segment A remains a prescribed-current / geometry-dominated region.
- Q80 and Q90 anchors occur in Segment B for all three parameter sets.
- Segment B corresponds to a control-state split: DCAC is already voltage-limited while DC remains in CC.
- Late CV preserves the earlier advantage rather than erasing it.

The correct mechanism-level interpretation is therefore:

```text
Full-protocol raw gain is supported.
Non-geometric Segment-A acceleration is not supported.
The dominant explanation is voltage-boundary / control-state split,
with late-CV preservation of the earlier advantage.
```

## Day20B status

Day20B is closed as a full-protocol batch audit.

The next step should not be another phase audit. The next step is to decide whether to:

1. extend the full-protocol batch to additional parameter sets, or  
2. translate the Day20B segment logic into a JES2 methodological discussion.